# Banking Churn Reduction - V7

Notebook analisis dan simulasi dengan penjelasan Indonesia. V7 memakai **modul Python yang sama** untuk Colab, CLI, dan uji otomatis. V6 tetap tersedia sebagai versi terdahulu.

**Cara memakai:** jalankan dari atas ke bawah melalui **Runtime > Run all**. Form konfigurasi memilih CSV referensi atau demonstrasi sintetis. Setelah training selesai, ubah salah satu dari tujuh form simulasi lalu jalankan hanya sel tersebut. Laporan HTML/PDF dan ZIP diperbarui oleh form advisor, ketidakpastian, dan batch; empat form dasar menyimpan snapshot tersendiri.

**Yang dipelajari:** memverifikasi data, memisahkan train/validation/test, memilih model tanpa memakai test, membaca metrik pada data tidak seimbang, dan menghubungkan prioritas churn dengan kapasitas serta asumsi ekonomi. Skor historis belum mengukur pelanggan yang terselamatkan.

Seluruh output training dan scoring disimpan dalam folder run tersendiri. File public repo hanya berisi sumber dan ringkasan agregat. Hindari membagikan ZIP/model/hasil scoring bila memakai data pelanggan sebenarnya.

[Repository dan dokumentasi](https://github.com/alandehars/banking-churn-reduction) | [Notebook V6](https://colab.research.google.com/github/alandehars/banking-churn-reduction/blob/main/notebooks/banking_churn_v6.ipynb)

## 1. Lingkungan yang dapat direproduksi

Python **3.12 dan 3.13** didukung; Colab terbaru memakai 3.13. Dependensi berikut berasal dari `uv.lock`; instalasi notebook memakai ekspor dengan hash, sedangkan CI memakai `uv sync --locked`. Windows dan Linux menjalankan versi akar yang sama, dengan dependensi sistem yang sesuai masing-masing OS/Python. Hasil numerik masih dapat sedikit berbeda menurut hardware.

Sel ini memeriksa paket terpasang sebelum mengimpor modul analisis. Jika paket yang sudah dimuat perlu diganti, instalasi akan meminta **Runtime > Restart session**, lalu **Run all** lagi. Ini mencegah proses memakai campuran biner lama dan baru. Internet diperlukan hanya saat instalasi atau unduh CSV referensi; mode sintetis tidak mengunduh data.

In [ ]:
#@title 1.1 - Verifikasi paket terkunci
LOCKED_REQUIREMENTS = "appnope==1.0.0 ; sys_platform == 'darwin' \\\n    --hash=sha256:685db59cb6043c3c2e528adc0b3bce3a5f8d09bcf7492c6ea650d1b7421f3c49 \\\n    --hash=sha256:6fe0c04218aab65c54c4ff81638cdbf848d89f5653b74d68638a137f200dd16e\n    # via ipykernel\nasttokens==3.0.2 \\\n    --hash=sha256:3ecdbd8f2cc195f53ccada3a613538bb5f9ef6f6869129f13e03c30a677b8fe2 \\\n    --hash=sha256:9da13157f5b28becde0bd374fc677dcd3c290614264eff096f167c469cd9f933\n    # via stack-data\nattrs==26.1.0 \\\n    --hash=sha256:c647aa4a12dfbad9333ca4e71fe62ddc36f4e63b2d260a37a8b83d2f043ac309 \\\n    --hash=sha256:d03ceb89cb322a8fd706d4fb91940737b6642aa36998fe130a9bc96c985eff32\n    # via\n    #   jsonschema\n    #   referencing\nbeautifulsoup4==4.15.0 \\\n    --hash=sha256:288e3ca7d54b06f2ac191970bc275c1939cb46d450b255bf6718b04aa37ab4f7 \\\n    --hash=sha256:d6f88de62e1d4e38ecb1077eb9724cd0eff29d2a08ca16a401e9b9e93f117cf9\n    # via gdown\ncertifi==2026.10.23 \\\n    --hash=sha256:a9895e6e2758b371e0545e321a3a8fd1fcfd9829ada8824a936dfb16aaaf36cd \\\n    --hash=sha256:e9b54de76e530bb6cf0ccd036787103acb58fd42ba930ea8bf948ddbaa2f61d1\n    # via requests\ncffi==2.1.1 ; implementation_name == 'pypy' \\\n    --hash=sha256:046bfc24911b37851ee1b51aab8bffe713d89c68c6a057b09484ce9fd5f69b4e \\\n    --hash=sha256:19ee6127ee34de7d83ce3d371ebc5ed91addbdcc39f9ab15ce4eb35a4e534971 \\\n    --hash=sha256:1aa5645c30469b09530c4ebca77ebf8f17618293c58f8549cb1a543a50236e7d \\\n    --hash=sha256:208f941bb9d18e768138677f0a6d2ce01f590df56043dda1df1535ac57c88517 \\\n    --hash=sha256:210019b6c7cf07f081b4c54635c8cf744377001350e29cc0f81c4377b4797735 \\\n    --hash=sha256:31348097ff5bbe827ccc41795d4dd099d9f0625e7def00ee653c137a490c2a6c \\\n    --hash=sha256:3143d81e29e1e20a9ce10901ec369012947876596f75a222235965f2b7ae832e \\\n    --hash=sha256:334644fbac4eff73d985a17a91226df55d0f394160c4cfb880e084c8f7161cac \\\n    --hash=sha256:4a7c934f7360e8cd64fe9efadcbd10c7c6364f531e432b9a4bf5ccbc9e0e8b50 \\\n    --hash=sha256:63bbfd5ded17c4840ac07cd8f1c21ba9d9708141f840b324f422f41b207e3973 \\\n    --hash=sha256:68e62fe11f30d5ca8289242866f0a5291402d8529ca2178ab8afc5c9694ae890 \\\n    --hash=sha256:6a8dddef476fab96d066d578fc88526767b836ab5ab21754e1d5bf3879c31c7c \\\n    --hash=sha256:7bde5e4cc5c10140859842b9d383af292b22639a4dffb725314baf45968cef80 \\\n    --hash=sha256:811bd1e21d32de12efca32393a0ab3f5133b54fce9bd44b8bd77ab07da14bf6a \\\n    --hash=sha256:9d2055050ea716bd38b7f7f1579c275386646b4894c155a3e2f3cd62ed41b7c6 \\\n    --hash=sha256:a2d7755bef5a12ed488f4ef1f1b69ee9191d7396083b755a5d2295f6edb4768b \\\n    --hash=sha256:a931079504ecc49efed7744c476a5c343a92fabf66dec2db95edb1b2fdc770e2 \\\n    --hash=sha256:aa9511c62d14da7aacc9b4bf51f3f697a621e83b2d6919008243c3aad168eea3 \\\n    --hash=sha256:b5bdfd1c873d4e093aabc0ca84c4ca6dbc4f752afb5c86f146d9742580c9da2e \\\n    --hash=sha256:c1453022f490d2459a11819d83ad1d586e9ff65a12ac3e705ffebd46d3685dcf \\\n    --hash=sha256:c8c69575568085ba0b1b10c0249d779a214aea6f6522e949a0fc9fb0fcb449d0 \\\n    --hash=sha256:dd31f52ea1086513bb9df30f8fcee9b8918323ae067a3d5b78bc826a000712be \\\n    --hash=sha256:e0bcb7e0f677f543555d2adff3bf19c05f66cdb4796e5ff602442ab2fe3c4ef7 \\\n    --hash=sha256:f16c709686a78c727bbbf059f92b0bf41c6fc60deec706d2dc19f529175a6125 \\\n    --hash=sha256:f53e442b08449d42821fa4a4fba000095af9f62742a500f978a9f557ec44339a \\\n    --hash=sha256:f81b3b8f3d4e343550fa4baa0e479bba9f2d29ce9c2e9b51d1ce1718d7442fcf \\\n    --hash=sha256:fcd22650c908d7b7da162bbfaab594a1227a15d1643a98c68b122ac642fa2264\n    # via pyzmq\ncharset-normalizer==3.5.2 \\\n    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \\\n    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \\\n    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \\\n    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \\\n    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \\\n    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \\\n    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \\\n    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \\\n    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \\\n    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \\\n    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \\\n    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \\\n    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \\\n    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \\\n    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \\\n    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \\\n    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \\\n    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \\\n    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \\\n    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \\\n    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \\\n    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \\\n    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \\\n    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \\\n    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \\\n    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \\\n    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \\\n    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \\\n    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \\\n    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \\\n    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \\\n    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \\\n    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \\\n    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \\\n    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \\\n    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \\\n    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \\\n    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \\\n    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \\\n    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \\\n    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \\\n    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \\\n    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \\\n    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \\\n    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \\\n    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \\\n    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \\\n    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \\\n    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \\\n    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \\\n    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \\\n    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \\\n    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \\\n    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \\\n    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1\n    # via\n    #   reportlab\n    #   requests\ncloudpickle==3.1.2 \\\n    --hash=sha256:7fda9eb655c9c230dab534f1983763de5835249750e85fbcef43aaa30a9a2414 \\\n    --hash=sha256:9acb47f6afd73f60dc1df93bb801b472f05ff42fa6c84167d25cb206be1fbf4a\n    # via joblib\ncolorama==0.4.6 ; sys_platform == 'win32' \\\n    --hash=sha256:08695f5cb7ed6e0531a20572697297273c47b8cae5a63ffc6d6ed5c201be6e44 \\\n    --hash=sha256:4f1d9991f5acc0ca119f9d443620b77f9d6b33703e51011c16baf57afb285fc6\n    # via\n    #   ipython\n    #   tqdm\ncomm==0.2.3 \\\n    --hash=sha256:2dc8048c10962d55d7ad693be1e7045d891b7ce8d999c97963a5e3e99c055971 \\\n    --hash=sha256:c615d91d75f7f04f095b30d1c1711babd43bdc6419c1be9886a85f2f4e489417\n    # via\n    #   ipykernel\n    #   ipywidgets\ncontourpy==1.4.0 \\\n    --hash=sha256:0c7a4c2716a4e98342221954416a836cca77c996c14ddbf22b5f01d5d93ca09c \\\n    --hash=sha256:186ba929df36d61b6127da2e89cd1357e4cb79aca647381ab1a6feb0b152877b \\\n    --hash=sha256:196759a3e4db60e1546167e361090ebb6d199aa3aafc9e34ea17a5ac3b23e814 \\\n    --hash=sha256:20156f5a1ac4f8ce02656e39a61e82164a3d359796dc8026f75b062783d500e1 \\\n    --hash=sha256:20da2f86bfaed60dbba729fe738c8241b1dd22b6cf7e8cbbf30df290bd04ba28 \\\n    --hash=sha256:2deb580178ca19437a84bd77e4bc2cd91a8ccad212413a83c273900868b5978c \\\n    --hash=sha256:3863ef2e2b13fe93f8c0ebb08ee400cb07153b8b0e91c5acb26e4537f283634f \\\n    --hash=sha256:423bd5f4b3f11d54a8c597234e513382e3454bb106b8ec7ae32ba4ed63f8d99b \\\n    --hash=sha256:43c3ccbb32c6294b183dcc8e8c46dacf5ecef809497e3d48a5be298eef185ad0 \\\n    --hash=sha256:510f7d93d94cf6ebf4e2cd0640bea16ab8affac7547230175b6745b596ce0599 \\\n    --hash=sha256:5450f091ac1be0be3ad3a2a3b3f23b5e443e78c670ced4fd347d626f92a28fd2 \\\n    --hash=sha256:6507a016976a75a15aee47809a34605de6244ccc056e6164bf9fb14b27eecad4 \\\n    --hash=sha256:6ced1670fafee703b8277ab1645072a226e74f167166eae146fb743916119b67 \\\n    --hash=sha256:6e697d94e69f499ff6bebb899cae97a58d5d14f0e1fe9568b43a0248d2f9af8c \\\n    --hash=sha256:7f861af508fca2384eef392bc1d8377cfa349b2b854cb11b45d89c75780e2561 \\\n    --hash=sha256:86d05cec773c9507a3950122e0e40ce77c23c75ceeb2fc189514e71893cbb34b \\\n    --hash=sha256:875f42444c9cf48d56f724f2637e60d0f73b3b12c9041e1484580a233edf9591 \\\n    --hash=sha256:9c0e07c691f3b3321913ed9b8161c50ea5f77fadd006f52f5e755359b0dcbfc5 \\\n    --hash=sha256:a03b32c2e7eda8b17757c022162c13c86f5c3d6f937dddf9ba3c3ff7b513953b \\\n    --hash=sha256:abd0f7e51ecc52bf8c95713bd80b1c17c516a71fa391c54475308a4377903c2b \\\n    --hash=sha256:c76f3a5318164db7d9401132fc1b5364b784c613c93fa506e3b5e6d1bf353ec8 \\\n    --hash=sha256:e21d1d4a9db5b3a793649c7ab0ddb4697f2818929eec871a7b9c45fce7b2a1ea \\\n    --hash=sha256:f1219a8898523cba821085f2da8a1b962cc696325763f09d7f93538ba43b2d70 \\\n    --hash=sha256:f863c6100bf926cf47d13f3cd75f9bb8ebd98aaab230eeb4468b91f98f39a6b3\n    # via matplotlib\ncycler==0.12.1 \\\n    --hash=sha256:85cef7cff222d8644161529808465972e51340599459b8ac3ccbac5a854e0d30 \\\n    --hash=sha256:88bb128f02ba341da8ef447245a9e138fae777f6a23943da4540077d3601eb1c\n    # via matplotlib\ndebugpy==1.8.22 \\\n    --hash=sha256:12bc7f368182b517cf26c76a2393fff65354e365fa1552b6241e66edff17997b \\\n    --hash=sha256:371a4ba4a5975eb958393903f3254cf983da7b1c7c178b3f987ee427f42515e3 \\\n    --hash=sha256:4ad076f4f66cb8acb79e4384d48b1ea60a0b1957aa0b1112d4887ea3a4df60e0 \\\n    --hash=sha256:8a697acec45dbc70d17fb5d9f4f61989fc294d273c69de487fd10cb35fdd75eb \\\n    --hash=sha256:a9e9d3550e15ca479c59333e90845029190531f0cacfedab3b815a57bd913947 \\\n    --hash=sha256:ba810a66b437e3c43ca0ce0892404f010338286263ca8e5108442d76a9485337 \\\n    --hash=sha256:c1ffb9953708b648ecf6acd2e5ad2c002bf68785ed618197dd4463a2a7226f39 \\\n    --hash=sha256:c21dd7e1ec22556bb41dc3bf6b86c603e440ec889c4acb27d7206354b2106c54 \\\n    --hash=sha256:e489c7268e1c7b41e13b438d9c533d2a7af73fb59bf8cd30fead8286c1c39c4e \\\n    --hash=sha256:f49b1d6cecf326b63c06d7f517e4a6642777f758c58cb799e7bc5a6dd74721c2\n    # via ipykernel\nexecuting==2.3.0 \\\n    --hash=sha256:15919cb5d667e5cb4e099511971d00d659573fff2dd5c4e6cd8b71636c7858d2 \\\n    --hash=sha256:736e859c9f8701f11fcf516856f26f562e04776387824b43a35a1dfe21c84122\n    # via stack-data\nfastjsonschema==2.22.2 \\\n    --hash=sha256:0fb3915616adac85ccfdd737d26be1089845d2019819505b42d39888458f74d4 \\\n    --hash=sha256:72064e12356a7d6ef02165be2946b9abadbdf238536e07eb587e3dbaa33099cf\n    # via nbformat\nfilelock==4.2.4 \\\n    --hash=sha256:301241438c57936ef93f4243a94f2559022268bfcf9b375a9a72eb0f8db8358e \\\n    --hash=sha256:c6d8663535e1650ee430507b4ffbac7b464dd64b7647ea0112a9f406499d163d\n    # via gdown\nfonttools==4.67.0 \\\n    --hash=sha256:0781fe22583529e1e98bb8a3a33040632e202a4c427ed7e65412c41a21b8ebcb \\\n    --hash=sha256:1be99c1f07fca59510d657ef3eae584b5273fa4e203aff2383b3520744e19536 \\\n    --hash=sha256:2a09d33a9264a6b29efca9dc633b53969aaedb250a9c8521d60f51280cef65ca \\\n    --hash=sha256:2bfab2f5d1d255dec82f4bd082a1c10e77df808e42210890f50a9c30bf91570e \\\n    --hash=sha256:36f0fee56227b909c9d1392f17b23803616f1f04efbe020c176d9945cabc0be5 \\\n    --hash=sha256:3cb57e6600ca77c0b1729cf8adc23bc0652633a37f18cfa934d9c7bc3de25519 \\\n    --hash=sha256:4304f03ed7f4ba000a8dcc941ad854bfa52e2f3b6112b8f099b6f431cf98e701 \\\n    --hash=sha256:48696b630069e29b8aa5ea8b034e4f651a2e112073938ec16bd536dadde1debf \\\n    --hash=sha256:50c41e30aa2e0130b80d1a58ac0f3ea7c02a854a70dbea1ff8d88e0ce524806f \\\n    --hash=sha256:7343cd0ef70edf8be7f4913cb9b55b992fb4e04055b47dcfecddcc2eb045a9d2 \\\n    --hash=sha256:768a33bbe6ec5ba8f19979f938752f06d4e614cb554fd47abd7830f2007660e3 \\\n    --hash=sha256:8239e2ca24878715a19f061d065b5721e87da81d145e48b3418f771a469b5a24 \\\n    --hash=sha256:ad813967410ba6d24a52850df59b164ee17883f17b96a91b4b0ac6e9d7b5a118 \\\n    --hash=sha256:ad8b4f7c754a627e91908fa1a1ccc90b489cd2810c0ba16acd26ea2ff5273db7 \\\n    --hash=sha256:e0ca4c8438dd6320f5850c9bbee3b3980455ee3bac602a9a0299caf9e799a0e8 \\\n    --hash=sha256:e8a8545cbd58bd29494ffe81e3cb35f8a29332a8e495c42bec334145ce8cd65b \\\n    --hash=sha256:eb3c98cac93aac4b9f6e3ce2008325340b234cc9b0338ca6b513f31962a1e278 \\\n    --hash=sha256:fcb9743140419410161acfe7ec205fb0a8a703acfccb85b586becb5a97c047c9\n    # via matplotlib\ngdown==6.4.2 \\\n    --hash=sha256:a5454bb4a2c2770fb4cd0105d11fd75d3484a0f664565686a08a948caad8c188 \\\n    --hash=sha256:feca9f54800b90f639729975a003958393f3003313c3e9cdb6c9838fe42ae5c6\n    # via banking-churn-reduction\nidna==3.20 \\\n    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \\\n    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c\n    # via requests\nipykernel==7.4.0 \\\n    --hash=sha256:4330114d22b9b33575b2c7c68753fc8faabbbcfbc60b0cdff9f14dd4ce63742f \\\n    --hash=sha256:a6757f790ddc5a6006b813d82da1a9dbb65b3a00f2b6607df776418c6566ec1c\n    # via banking-churn-reduction\nipython==9.17.1 \\\n    --hash=sha256:6d1645743cfd1a07eb695d85aa2b5fa66721f8cbae9431d4049f7084bbf06509 \\\n    --hash=sha256:8919be8c27f20a6f4423145028063f6637b42a03ce57665bb12015ee1f073529\n    # via\n    #   ipykernel\n    #   ipywidgets\nipython-pygments-lexers==1.1.1 \\\n    --hash=sha256:09c0138009e56b6854f9535736f4171d855c8c08a563a0dcd8022f78355c7e81 \\\n    --hash=sha256:a9462224a505ade19a605f71f8fa63c2048833ce50abc86768a0d81d876dc81c\n    # via ipython\nipywidgets==8.1.9 \\\n    --hash=sha256:bcccba38a6ec3253f7a39c943cea5b9ad01999ce071396171adbc51c6a6a8613 \\\n    --hash=sha256:f2b8cbcaae10252b809fbe4d7470db75c09b769a32cbf816d20e5ca6d3c5a79d\n    # via banking-churn-reduction\njedi==0.20.1 \\\n    --hash=sha256:0fb16d86c4a4c73c37ba518c77419975e30fcc620658a8d14fbb5720cdd34142 \\\n    --hash=sha256:2f71208c3f9c1bca057c0e90d3f272aba44ace88fc4067d7587e9e069331b7e5\n    # via ipython\njoblib==1.6.0 \\\n    --hash=sha256:2ccc96785b12046c08fd6d55839c12857831b54a3c1673ffadd2f04bfc4eda03 \\\n    --hash=sha256:3dbbf9f6e4b592a2357b854608e980fe6390d131d7a82f011a377ef2ebef7aba\n    # via\n    #   banking-churn-reduction\n    #   scikit-learn\njsonschema==4.26.0 \\\n    --hash=sha256:0c26707e2efad8aa1bfc5b7ce170f3fccc2e4918ff85989ba9ffa9facb2be326 \\\n    --hash=sha256:d489f15263b8d200f8387e64b4c3a75f06629559fb73deb8fdfb525f2dab50ce\n    # via nbformat\njsonschema-specifications==2025.9.1 \\\n    --hash=sha256:98802fee3a11ee76ecaca44429fda8a41bff98b00a0f2838151b113f210cc6fe \\\n    --hash=sha256:b540987f239e745613c7a9176f3edb72b832a4ac465cf02712288397832b5e8d\n    # via jsonschema\njupyter-client==8.10.0 \\\n    --hash=sha256:5f73f24f22fa25192cfff6b23c051932a2473a797b05734aff495b392103e14e \\\n    --hash=sha256:9f7116294dca55f1785be880057d44544db9b1567718d92cb33c58886afb9497\n    # via\n    #   ipykernel\n    #   nbclient\njupyter-core==5.9.1 \\\n    --hash=sha256:4d09aaff303b9566c3ce657f580bd089ff5c91f5f89cf7d8846c3cdf465b5508 \\\n    --hash=sha256:ebf87fdc6073d142e114c72c9e29a9d7ca03fad818c5d300ce2adc1fb0743407\n    # via\n    #   ipykernel\n    #   jupyter-client\n    #   nbclient\n    #   nbformat\njupyterlab-widgets==3.0.17 \\\n    --hash=sha256:40ac1e9955acf116c4d995d9bfa082d86ad9ec6d91c4f134827cf5e0a5eb75e0 \\\n    --hash=sha256:6e61fe21ca8a66039180a5cc52a433e07279d2fee79c8be963e00d55193f17a8\n    # via ipywidgets\nkiwisolver==1.5.1 \\\n    --hash=sha256:06a6917674de9e0fe3f66f5430787f59a9f2ddb64af9b714eaec547e29ef5c19 \\\n    --hash=sha256:072bdb15a3c19a5b5dbc8f8fb1f4e1884bf4f3507eeb4cc6334401274d37a5c0 \\\n    --hash=sha256:0ebdef3eae5336568147c39a55be6a2036ffde53faa9ca2d978989ae7c2da12c \\\n    --hash=sha256:1209042a623ddfda5497e4066c7b77651dde8e1d3a9dd97599dc7e97f3b9b78c \\\n    --hash=sha256:1798e83840c3f627246104c4d8a9639c60fa068adf9ce92b61791781fa8a68c1 \\\n    --hash=sha256:186884a58486651e3c217b6acea0a53eaa9498fdd472057c46f2f0fb5c25aad5 \\\n    --hash=sha256:1ed0f5e49d0ceff8b72190824d9e59c062fbbc02c231b853112c78474b3f5ec2 \\\n    --hash=sha256:21e46b23a2da695c364124817bc01d970effd5483147f8d66a6a7167e3f6b851 \\\n    --hash=sha256:2551cf9917af48ee7c4b29cc82320489508cf96fd26a51f6fc124de661cd44c7 \\\n    --hash=sha256:26e8268480be5061d509e29669d59103c067a26377a56491630ece11762e3858 \\\n    --hash=sha256:34633ecf50d16187ab8e5528b7a2530f2feb4e23f300db4672538b51cfc5cd38 \\\n    --hash=sha256:38f6e0deb4d0a4615efe0c4efc5990b06ae450ab50a0b321c0b078b6d238c083 \\\n    --hash=sha256:482676e5bd48d70ac99d9fc78863469845421e01184fa83f1f9366dc49f7e974 \\\n    --hash=sha256:63fb7294b768f444eb4b068965f2662f28c2fd4161e23bd60fcf3ff27b74c046 \\\n    --hash=sha256:6a797a1cefc8b9c93170db580337e1fe3d011ad18b1299943231279406342048 \\\n    --hash=sha256:74ad5c3dad54a4641b4c28cd15ded70899d04459c6c7aeacafea716be97cce6d \\\n    --hash=sha256:75d9b1cf8258462dbdc1eeda718c96ea7f079324c09067f6daabfcf37712b7fe \\\n    --hash=sha256:77a4c8187a5948d7f8795adb765a3c7b553d07d86d88e43038fc32fc1fb9a3f3 \\\n    --hash=sha256:876bbfd276473d3daffe30e8c975df4ed9429967b41a6cb362dbb5155b6f13ad \\\n    --hash=sha256:886fc26012f0e8b5f69d1cfe6d711f6b11f194621539bf8e6bb1c25c5dc82724 \\\n    --hash=sha256:8a34616dc2521cc8dc1d7d081734da63539f021ac0450ce950908340c6e7aa2f \\\n    --hash=sha256:8bf4df63592c2a66b4f8edc5df2544998c288aa02f96ce0acd880cd1de8c8127 \\\n    --hash=sha256:8fca690b00c4c48f6c2a547b0160ed511357093a4e4c9b47e0fadf3128066d89 \\\n    --hash=sha256:a5a00665d1a0e26763a7338d7e911d4598fbc1d50dd0d6b7919b7dc6c5d6569f \\\n    --hash=sha256:a5ca5aebae78a0bc13c1943af4af615d4966c5b650b05d5aa83b50e427196fee \\\n    --hash=sha256:ad8b9671348d7c8716715652ae11f85ed0eb99e265a2df2ca490577d69860b2c \\\n    --hash=sha256:aefe930d113798330e9462f7874542977869c0613cba3262e2de3a8d5dee8f3a \\\n    --hash=sha256:b03af77d77e50edba2030fd5f7c352ff209314b09030a3cba7c14edf9a09a444 \\\n    --hash=sha256:b6ae6a0328f0bc035741820fdeecdcd67bf4694eee03972e843663107122f450 \\\n    --hash=sha256:bfd1de989b3330420e29de39352f5c049905c9e3ee67233a50d550e3d652c148 \\\n    --hash=sha256:c90d3022d8a94778939cda8638c6c8da8fa757b8958dad7ec868ce29c87681b8 \\\n    --hash=sha256:d09037ca068d784ebc4aec290ef952ca27ac15dd9c0b5801a88c6e1096b83e6b \\\n    --hash=sha256:d27c2123977cb9269c30a49ba45f03a4323017ef693e19db4ec9dbe1299a3002 \\\n    --hash=sha256:d79308fa689fac89cbcfbd4dbfc80b5f95c54c5a7fd4d194be221f9d33d026e6 \\\n    --hash=sha256:dc23390afe9f4ef9ac3bcc72a03a56eebbde03f4c571a32cb38f859cff9a6524 \\\n    --hash=sha256:f1303ef2eec81262a4b708c3e858afe58d7c75ad91c1c05266eda7673369859a \\\n    --hash=sha256:f942903fde7363d1d879057ec5de01310efda2597161784d752fa9953a01a71a\n    # via matplotlib\nmatplotlib==3.11.2 \\\n    --hash=sha256:1944895967f87c84c9b4bad29a707b31f5b36df4a3a2339ea1f8ea3ce5105539 \\\n    --hash=sha256:1b9a7ad579856284135e401ecc918c5f8a017ee30539298862a109f51b971710 \\\n    --hash=sha256:2a8285cea8ef4d92aa041d1c33788bcae82248503300f93f1ca2136b9049452f \\\n    --hash=sha256:3aa4b8516fd26659e4363abbf317c703d9116496c5db2e9d0609a2866dd39dd2 \\\n    --hash=sha256:3e8576f7c47e02fd4f21f44171302d2d1d58d4471d46da3d71fe8899d19539d9 \\\n    --hash=sha256:643ff850d8e0f5b8319337f87ed3cb59506afb3df3cc48de777d85871233be7b \\\n    --hash=sha256:7d43ff8cebb50840648cb6429b2228621dbd709010f3117b3740abb20abf21c0 \\\n    --hash=sha256:7e5a90f8a707ebb6004a713b1cff091a40cc5df4c7c1cb165e5a505ebc11c292 \\\n    --hash=sha256:af2661f6ac6bbd1d081996f54fdd9385715c625ace8cec0869055c0cfbf38981 \\\n    --hash=sha256:bc067c462a86f0e57bf52fc6e058d90171a5420007dac00c46e90153050c69a7 \\\n    --hash=sha256:c5c1c68ee401fc98271263410f0e5ce88285abacf7627132914e8adf3d70ff43 \\\n    --hash=sha256:cec596316640f2b394b8f0daa0ea61a8eae82d017b620b9f202befb972a59ea4 \\\n    --hash=sha256:df4f7784aca81a94f254c0a2767d592ee25f407e488f5fa7203e51093fb6ca27 \\\n    --hash=sha256:ef31985c4dedb5f1424e1aec6849a47dd37689cb7fa3c20b1b82187f26806261 \\\n    --hash=sha256:ef752769cd962f39ea0b6ffc82d1ea43a0012c5a6157c7a075212fa509cfcff2\n    # via\n    #   banking-churn-reduction\n    #   seaborn\nmatplotlib-inline==0.2.2 \\\n    --hash=sha256:3c821cf1c209f59fb2d2d64abbf5b23b67bcb2210d663f9918dd851c6da1fcf6 \\\n    --hash=sha256:72f3fe8fce36b70d4a5b612f899090cd0401deddc4ea90e1572b9f4bfb058c79\n    # via\n    #   ipykernel\n    #   ipython\nnarwhals==2.27.1 \\\n    --hash=sha256:aed93076a3ea42d9c32c88e4eb5ea422a21937011cbe1f480f9572a523c82094 \\\n    --hash=sha256:d057df13f5852b8e157596e82eb5e955fad267425df5e420e0ee9863da483b31\n    # via scikit-learn\nnbclient==0.11.0 \\\n    --hash=sha256:04a134a5b087f2c5887f228aca155db50169b8cd9334dee6942c8e927e56081a \\\n    --hash=sha256:ef7fa0d59d6e1d41103933d8a445a18d5de860ca6b613b87b8574accdb3c2895\n    # via banking-churn-reduction\nnbformat==5.11.1 \\\n    --hash=sha256:32d4521c68c6e7d5b29c76defaeed9f42ea733142b9b19f88277ce10390b9c4d \\\n    --hash=sha256:cc6698fa75f4fab8755ead786317815f13a6fee3b53311c0abb1a8b51d52f7ec\n    # via\n    #   banking-churn-reduction\n    #   nbclient\nnest-asyncio2==1.7.4 \\\n    --hash=sha256:86e77b52c3d1496b294af3c944b44213b9b82a12a5ee590c0acc4ca4692ef6c7 \\\n    --hash=sha256:995b1409d926ec55e0876ea39a69bffdd8852cf1db3652c177e8c8ffdbcaf3cc\n    # via ipykernel\nnumpy==2.5.4 \\\n    --hash=sha256:1c2e71b04c6cad90026e544501bbe0ab9290fa8a4d845e7e8c0d124fb429c988 \\\n    --hash=sha256:2377da2dd3ba2c1200956acbab2a358c83b8e1f8531191672d1cd6ad83250d53 \\\n    --hash=sha256:2fa3328f784fc8277fc48026f6cad516f5c561c5d8e2e39b3c9e0c8f23223b34 \\\n    --hash=sha256:381a7a3d2e65e64c0ec302795ab9dc12bb1e73f150904699c153716177eebdaf \\\n    --hash=sha256:5258bc06526964be5face2fc6f756857a3f24f21ec3e72ca131337a75b165d6c \\\n    --hash=sha256:6d6a71b9d9a97c03633aa12565ef2825ffa036cc1d99cfd50dacf0f128af4fe2 \\\n    --hash=sha256:6ffa07666f8da0eef81d149934a626d0d95fbd6838432a33e66245423a9062c0 \\\n    --hash=sha256:7415db95818b39ec475a5eea54d9e3b6bc83e3912158e46da3438cdce399804d \\\n    --hash=sha256:8b4d2fd2d34e5f8c9235ee787de5631a37a28402b15cb80814df973d2be54129 \\\n    --hash=sha256:9968ab7e49b93ac6e1c3b2239732183152c9150f16308d30b66a372cffe3483c \\\n    --hash=sha256:9a94cf751c9ad8ebaa835bcd3d40dacf8534ad086b88c38029b65123c7999d2a \\\n    --hash=sha256:9cb18a327b49c5c337f972b03682f6a49855525faaf3c0d3e9c96cd0fd8880a8 \\\n    --hash=sha256:a7b1b6353e36a7e50de2973a38d705c88ee93adcf120673cee7f45a4a3fa223a \\\n    --hash=sha256:aa1cce2ff3f8d953de38b76bf44602caeb69f101430208f64a10067f7cb4b1d3 \\\n    --hash=sha256:aec3fc4b32ff82421274f5d205c559c51c840c8df66a78efd7f3612dd005a26a \\\n    --hash=sha256:b11e8fda06a7d69f15ebf542660b74466c2e51094800c1fb794f47ad4faeef17 \\\n    --hash=sha256:b86966fbe4ad7de710422175572bcdc75fdedadfb54bc6fab7deabccddd7780b \\\n    --hash=sha256:b89d0aaae2fe498c648f4c4795c084db535af5bd98ef942b2a3681fb74ce8645 \\\n    --hash=sha256:bc39ac66a7a9a3fbd6134fda43136b60ffde99c8f4501e64e0d2b24da137babf \\\n    --hash=sha256:c6342f54c67093cae5c0227eb0eb772fdb79f2a2c37a6eb278b9909ee06aa356 \\\n    --hash=sha256:d8200f16437b289a5bb927c6e184eccc3e8389bc0070fea4cd5b9e13c1757959 \\\n    --hash=sha256:fbde6962867ee75b48b0ee29b2b9372ec5d617799dbaf38e82dc0596f2f7738a \\\n    --hash=sha256:fe4d21ab149f15e4e6043dfb0de87e6e5f34ac176cde83060e9802981fca2ac2\n    # via\n    #   banking-churn-reduction\n    #   contourpy\n    #   matplotlib\n    #   pandas\n    #   scikit-learn\n    #   scipy\n    #   seaborn\n    #   xgboost\nnvidia-nccl-cu13==2.32.3 ; sys_platform == 'linux' \\\n    --hash=sha256:1459723080ac889d73a26edfa3e04383a7928ab31ac8f0ec43b3ea9548b04ff3 \\\n    --hash=sha256:a5bee92b2f4af218c109f8d221c3c9adcc752b94ae0ffcb0ed5abf9341a7724c\n    # via xgboost\npackaging==26.3 \\\n    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \\\n    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c\n    # via\n    #   banking-churn-reduction\n    #   ipykernel\n    #   matplotlib\n    #   wheel\npandas==3.0.6 \\\n    --hash=sha256:0704044b676496b8350e023b09f174a26772456c974a2b11c36bebb558c9490d \\\n    --hash=sha256:097090508a1dd335013d39106fc10b20f4fd4a171638e47b77d55798ed9dab6c \\\n    --hash=sha256:1e92d9fa834c7d877130027cddc0cad8dcff97c1f6cca26bd6310f847228b658 \\\n    --hash=sha256:2a8fc94be2ee5f1d86f97aacd8cc566f81680b6498e76f3007421bb5d98151bf \\\n    --hash=sha256:3ef908d28590b3f42d7070e7ad8f9b34b442b260b7f3c1afb57e0040c58cdb1b \\\n    --hash=sha256:569e114072b24fc4970c12e2b4bab252671668a40b324318903380cab0254c0c \\\n    --hash=sha256:60d81f9e1799b36f3739e7fff44d1fbb2e8fd5a271b3863e03de9715fccda0fa \\\n    --hash=sha256:66b07ef7315a31bfe1089cd3d71a7de781c9dca986762d0b4fe7c0ef17465d10 \\\n    --hash=sha256:7dac2d65e9087e8e7b5a45fe15c4920911a221df061ab629943ce016489145c7 \\\n    --hash=sha256:9ae8073aed8e21d1a7fe263dcdc6840743549722a6738198a0a46000fa9476f2 \\\n    --hash=sha256:9dab635a549e58a053c7b0fa054dc0bd7be22f0ed9a720f4a85d5fb993276172 \\\n    --hash=sha256:b27c8d890e4aa2171437ae2a39de1d215e674158e4865c4023a8b31c932513b2 \\\n    --hash=sha256:cc39303913e2ea129915670de5d1c9fbd647f543bb72e5543bac8baa94e9e42f \\\n    --hash=sha256:e3dccb584123b399c07562ac4d62543e90ede49ddf8ce3c13ffc64cbe828c281 \\\n    --hash=sha256:e7c1905ef02c3d6d43d9dbd5b6ccb4da4870a0b0c821bbc103fbdb6f3ad2707b \\\n    --hash=sha256:f3ce8a6968045481e91a3990e797e348ce13db45ee164a7095bbc824e26c09dd \\\n    --hash=sha256:f4e7c52eb108d752e7592268108fd3e98efd76d83a3125cdd06c621c2e44359b \\\n    --hash=sha256:f8029ec0f1f89e4f985929ce1f6626dabf3140d61a4e9c1215afdab34eaf9a5d\n    # via\n    #   banking-churn-reduction\n    #   seaborn\nparso==0.8.7 \\\n    --hash=sha256:a8926eb2a1b915486941fdbd31e86a4baf88fe8c210f25f2f35ecec5b574ca1c \\\n    --hash=sha256:eaaac4c9fdd5e9e8852dc778d2d7405897ec510f2a298071453e5e3a07914bb1\n    # via jedi\npexpect==4.9.0 ; sys_platform != 'emscripten' and sys_platform != 'win32' \\\n    --hash=sha256:7236d1e080e4936be2dc3e326cec0af72acf9212a7e1d060210e70a47e253523 \\\n    --hash=sha256:ee7d41123f3c9911050ea2c2dac107568dc43b2d3b0c7557a33212c398ead30f\n    # via ipython\npillow==13.0.0 \\\n    --hash=sha256:02464da86f561f410e13b061ec37c3068a9cb557a095494157742d07b3c24a27 \\\n    --hash=sha256:08b826a9bf1265c76de04a42061d61ea6b109418b7826f28ff1ecdfaa26c5502 \\\n    --hash=sha256:12b0fa15879fe5a9d9eebe66e4e2d74128a773b1a4448010c9c714f45e04b9d8 \\\n    --hash=sha256:19b66ca0a4d25cc0200b381ce824c7ea090db61427315e73cfb05dbf1b35c369 \\\n    --hash=sha256:23edf4ec4b6f4f240e8df6e4985e9ee2ced86474632433e55fb1e2548744bf40 \\\n    --hash=sha256:2d50500a4a3bc6469adcdb319cf05d688b08ca3bf0f22a05339f3b25b433a599 \\\n    --hash=sha256:34fb3db1149c879343861831728c243c432d4a5afc8f1878ca7612ef8de2a033 \\\n    --hash=sha256:40a266371ece581e5de13b6635ef78d7487824af80561b6ca27694a88c19bc0f \\\n    --hash=sha256:42d63be3af4f9b951b199b0a9d23365d37ab14687e6d77b74940432c8515c4ca \\\n    --hash=sha256:4bd024d567eca931b668978d672dba914d4b8e2628e2c3073a2ad178de8a2620 \\\n    --hash=sha256:567f20f1c75897d7663830da6ea8af719e03d712b1d9ee7f813319f9ece42e5a \\\n    --hash=sha256:5f56499c1a74d2b862beb0c33b8016f99d095858455df9d21583bbbc1c15b582 \\\n    --hash=sha256:67c8d44ad89fa7eb88de5d6ef6199b1e03dd24974f85a67639fc8badb801fe79 \\\n    --hash=sha256:7e91e97a929232cdb828889ee6ad0e48d7f40f4ec38818f4e4a8968b2b1af22e \\\n    --hash=sha256:8619685cad8efc4c38c66d3b36aad854523297c43d748b3b6f0cc03c90ba3463 \\\n    --hash=sha256:8bf295fe459343bc6093c8e5e0df60bfa189e0fcb775a22e2c2488fdeccc6c0d \\\n    --hash=sha256:93ff9c9c3663ce23aaa5d0b6b6abac55359e2a0c06c7e30f3771829af816be63 \\\n    --hash=sha256:9dda0788a9932776a75d8bcc020076edc308ce2508b51f6739a45e3158b3fc42 \\\n    --hash=sha256:a2a14cd73d5e6c927f644b0110ee81f59c4ef506259ec9183904747f2caacc81 \\\n    --hash=sha256:c3bc8ee0271387e0b1e3ad0ff893645990a656e79bf9c42f4f734a3642b11d51 \\\n    --hash=sha256:cdcd1ba258178f7356d74170c9a81ba4eab6a5d55748bc402a5a2a6fe6be3510 \\\n    --hash=sha256:e65dd6e75de67b2362a4fa3e16aa627595f65054dc4ec73d64680c84d592fa61 \\\n    --hash=sha256:e98b4f91f99b24b1da72e10621ba181f7281922384a7403094debb4199de6c00 \\\n    --hash=sha256:ef5fe6b7db3f501c56000ba8fe7c7e1a35ca1d45824fbeb95a67daa85cff4dbe \\\n    --hash=sha256:f11a3c1358bb3c68012a7b1f41b223c3db9f7bfde90b481df028cfe1c3363313 \\\n    --hash=sha256:fc481f742ccda20e9ed760b8f978dff50f2a4e9bc253ac167fb0a05b0783ef52 \\\n    --hash=sha256:ff3fe9eb621fbb9db4c4be33ab946f6bac5f98ab50ca8d3260503653656c961b\n    # via\n    #   matplotlib\n    #   reportlab\nplatformdirs==4.13.5 \\\n    --hash=sha256:05831adf87d392760f4b5c3a0a95579ab5a10b18b9b9bd43680dcb47544788a0 \\\n    --hash=sha256:53d3ec4280b815c1e0b02edad7f9ee8fda1ba9c0865dd9c2e6e63849ce82fc1e\n    # via jupyter-core\nprompt-toolkit==3.0.53 \\\n    --hash=sha256:01c0891d7f9237d5e339f7d3e42cdae80b7534abb1c7c0e3352efba6231492f2 \\\n    --hash=sha256:9ec8a0ad96d5c56148b3f914aa79c1564c3fde5d2e6b876e7bc327e353cf8fa6\n    # via ipython\npsutil==7.2.2 ; sys_platform != 'cygwin' and sys_platform != 'emscripten' \\\n    --hash=sha256:0746f5f8d406af344fd547f1c8daa5f5c33dbc293bb8d6a16d80b4bb88f59372 \\\n    --hash=sha256:076a2d2f923fd4821644f5ba89f059523da90dc9014e85f8e45a5774ca5bc6f9 \\\n    --hash=sha256:1a571f2330c966c62aeda00dd24620425d4b0cc86881c89861fbc04549e5dc63 \\\n    --hash=sha256:1a7b04c10f32cc88ab39cbf606e117fd74721c831c98a27dc04578deb0c16979 \\\n    --hash=sha256:2edccc433cbfa046b980b0df0171cd25bcaeb3a68fe9022db0979e7aa74a826b \\\n    --hash=sha256:8c233660f575a5a89e6d4cb65d9f938126312bca76d8fe087b947b3a1aaac9ee \\\n    --hash=sha256:917e891983ca3c1887b4ef36447b1e0873e70c933afc831c6b6da078ba474312 \\\n    --hash=sha256:ab486563df44c17f5173621c7b198955bd6b613fb87c71c161f827d3fb149a9b \\\n    --hash=sha256:ae0aefdd8796a7737eccea863f80f81e468a1e4cf14d926bd9b6f5f2d5f90ca9 \\\n    --hash=sha256:b0726cecd84f9474419d67252add4ac0cd9811b04d61123054b9fb6f57df6e9e \\\n    --hash=sha256:b58fabe35e80b264a4e3bb23e6b96f9e45a3df7fb7eed419ac0e5947c61e47cc \\\n    --hash=sha256:e78c8603dcd9a04c7364f1a3e670cea95d51ee865e4efb3556a3a63adef958ea \\\n    --hash=sha256:eb7e81434c8d223ec4a219b5fc1c47d0417b12be7ea866e24fb5ad6e84b3d988 \\\n    --hash=sha256:ed0cace939114f62738d808fdcecd4c869222507e266e574799e9c0faa17d486 \\\n    --hash=sha256:fd04ef36b4a6d599bbdb225dd1d3f51e00105f6d48a28f006da7f9822f2606d8\n    # via ipython\nptyprocess==0.7.0 ; sys_platform != 'emscripten' and sys_platform != 'win32' \\\n    --hash=sha256:4b41f3967fce3af57cc7e94b888626c18bf37a083e3651ca8feeb66d492fef35 \\\n    --hash=sha256:5c5d0a3b48ceee0b48485e0c26037c0acd7d29765ca3fbb5cb3831d347423220\n    # via pexpect\npure-eval==0.2.4 \\\n    --hash=sha256:260c2774686e651b79f8b8e7fc9d80b3599ea6a66334b47d5f4abb69fc2c0ea1 \\\n    --hash=sha256:96cae060a313cfaad51bb761278bfb0e62dc0248d9315a81173752dc546cd37a\n    # via stack-data\npycparser==3.12 ; implementation_name == 'pypy' \\\n    --hash=sha256:a076f3f489b39b43ed13563a1dafe42447445bec2e294154ec5c57fcdda4c85e \\\n    --hash=sha256:f03387877eab82ce01bbdfd0ba7252ee55a90b1c708d5129e2ce915475cf1db6\n    # via cffi\npygments==2.21.0 \\\n    --hash=sha256:2363c69b61c4a97c838da3b130dcd6468f4848992b21a82f2a63ec34377137d9 \\\n    --hash=sha256:610ca751c9bc2492b38eb9a38a7fbc93edbbb2d7182edaf34e66ae493dee5c8c\n    # via\n    #   ipython\n    #   ipython-pygments-lexers\npyparsing==3.3.3 \\\n    --hash=sha256:928ae7e20211f3b6f3915a72f06a0cfd29ab9d24279dd6346b6b1a7146397d36 \\\n    --hash=sha256:ece8c00a69cf01b45d0b1dedabb469c90d8caf996d4fda40f147627a122849a4\n    # via matplotlib\npypdf==6.21.2 \\\n    --hash=sha256:0a3e1c5c886a1be03646c16a04827cc166c2a0575c971b26a4bf89686551c2d4 \\\n    --hash=sha256:1d44d1a3a200f25062adff0085b02040e4646431ac2ccac22e42679e91466084\npysocks==1.7.1 \\\n    --hash=sha256:2725bd0a9925919b9b51739eea5f9e2bae91e83288108a9ad338b2e3a4435ee5 \\\n    --hash=sha256:3f8804571ebe159c380ac6de37643bb4685970655d3bba243530d6558b799aa0\n    # via requests\npython-dateutil==3.0.0 \\\n    --hash=sha256:3c79630202e4152c6add678446984236efe4f57e84d8e892aab0637bab82e9fd \\\n    --hash=sha256:7ebcb3c29a92f3ceafe79a7a48a5c45677e9e004bbeec5742ac0bbb00560b7a2\n    # via\n    #   jupyter-client\n    #   matplotlib\n    #   pandas\npyzmq==27.2.0 \\\n    --hash=sha256:00e73942ef12cecbc7951c4a9104bb8ffaed742abb13af2da6833d90dd368cef \\\n    --hash=sha256:1f8079d0521fe94bbb401fe9407578b28f3701627c8be2c9f7e0c5b77dcb0109 \\\n    --hash=sha256:288cc790da0e3064a14a38ddc56ba169dada8c8af4cb86518db2bcbd380eedbb \\\n    --hash=sha256:2c218c6ab8bc447ba62054b581fd30209689d199c6ecb253f79615ca74a38e12 \\\n    --hash=sha256:348d6fd3e4b81ae4580622ea8c2ea60224e84b2ac1b3be4482e6edc7de06e7a3 \\\n    --hash=sha256:54d4259d1bfae24ecdb5ca79f7acc2eac6c286a02d6a0ae617797cb45f0726d3 \\\n    --hash=sha256:591c8de5851c5ea372194469fe97587b97c3b641e9a70f31bb3474acbfde0241 \\\n    --hash=sha256:b8d5f66e4a8246cf77f7b8f7902af64f00553368fa0373c89d99b78f0ad79394 \\\n    --hash=sha256:c551b9e2f86dc625fcb1a032c0d68042678caf96a8dd7c28796766b673bd5b52 \\\n    --hash=sha256:d1526b42a2e725b84ed226f37becedc250c6347594e5ed304e4e9aff68c9aec3 \\\n    --hash=sha256:dcc99ca132b667a4ed750afd42db4ea73288f18425a9b2e3c0af095665c491f5 \\\n    --hash=sha256:dea74fd65f1fc5f7fe167916a473ebe6ed6174e5e5d9de11ea6583661be6cf43 \\\n    --hash=sha256:f707bcf2c1d007d14d70531d4dd7b41060881c73efa845580bf6faaf9ea24d42 \\\n    --hash=sha256:fdaaa4ea3242f6ad298eb5177eb042aea5c73c30e76d20caee7b15af20d24ec2\n    # via\n    #   ipykernel\n    #   jupyter-client\nreferencing==0.37.0 \\\n    --hash=sha256:381329a9f99628c9069361716891d34ad94af76e461dcb0335825aecc7692231 \\\n    --hash=sha256:44aefc3142c5b842538163acb373e24cce6632bd54bdb01b21ad5863489f50d8\n    # via\n    #   jsonschema\n    #   jsonschema-specifications\nreportlab==5.1.0 \\\n    --hash=sha256:59fdbaec5efe437095ad1d7292ec9a271c8ca2d3ec75e889c6e61400929b90df \\\n    --hash=sha256:6878200464527e87672d19e9f7cfc6bb890e3a6931e74ac6a1a8d1754aa55671\n    # via banking-churn-reduction\nrequests==2.34.2 \\\n    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \\\n    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed\n    # via gdown\nrpds-py==2026.9.1 \\\n    --hash=sha256:01445c8d194aa032a08e944f16567672da1c62dbdbefd8b6d0693032e290cf68 \\\n    --hash=sha256:10e208f2425d973938afcd56e28a7c4be32e27b6a60b5d381f49fb9d8acf9759 \\\n    --hash=sha256:172e47169583f46ce118cbec68e6795d0da0f4606b488b6434f8276bca0a058c \\\n    --hash=sha256:2693b2728bbcc48d09a981a356954b0c47c53ff25b545856f28a889ea619f69a \\\n    --hash=sha256:3890a6aa36e6baa53d5258a2a25d3ef8b37ad165a6ab27a892d7c3e3a432cd69 \\\n    --hash=sha256:3b5a6f40f0a1486b4b36c888123afc67acdbd9f33235927acf5ff295429a0ba3 \\\n    --hash=sha256:3cd182d7291d29b92c521a0069d9c01ba6193628a9a105531d11b40a6d731a33 \\\n    --hash=sha256:3e93b2cd69a9830be33e03945cd7cda940a0a8bfcfbff41d6144f0cb0d3d8bd9 \\\n    --hash=sha256:457866b85daf5034296666168b84a69e0b2e89dc4f1af102b46f6448a60b9063 \\\n    --hash=sha256:46d80bc76b51a6c24f9944368c28d38b8bcbcea1da4f2f8d3ebc31a67e8c6ec6 \\\n    --hash=sha256:4793ef7f78268b124b73fa933440f01d258bbae01de9fa53e9080c9ab0425a12 \\\n    --hash=sha256:492e5e428cbe126221611f47e068f01660352feec4ad18bc0f5ea9b2ae88fb14 \\\n    --hash=sha256:501909f2e4a1e2dee528ef766fe3c469060ebc17e54a8383d404ba07a81a6f02 \\\n    --hash=sha256:50906f5aea24b5a865cbd0a589698288631d9f3a54c3a937c83aefa95a0d14af \\\n    --hash=sha256:56c6952a9b15047466d0c2347c446a761d4527f89976156341e68f0ce5cc08b0 \\\n    --hash=sha256:5943980471829f6de242a20b109de3111ba6b77e3af0ffc587028ac854b05e6c \\\n    --hash=sha256:5ce8943f79c2210f7abcc28e86367b03b28d95027fd01c46d2472373ae70c86f \\\n    --hash=sha256:684fd492fff4fead00587544e059be2bbcb6f93454f21fa2a91b66fc7508be82 \\\n    --hash=sha256:6b5b393eda5ea42cca1c1a6665f2a4882b4fd5d1777e41ce0545a107fb008c9d \\\n    --hash=sha256:6b9bf3135b4ad5981df9a73d71a35272d650a2985ae9c2746357b24d59de2448 \\\n    --hash=sha256:6c0dbbcc19735fe5f8b0a54c07659d154a9e69f47e15d0a6ab7299215daf62cb \\\n    --hash=sha256:6cdc537c8633d7fd92a82e2e0d2ab74320a3f63d5e59fb9cf08711e08fe151c4 \\\n    --hash=sha256:76d3af9732d2dab69f28179b40ba2d87e2f1d5824b4a694780aa787d685e8f36 \\\n    --hash=sha256:78326f4cb4427a56ba4996c0762b63be45f06b85f086526420d2b3a66e40f84d \\\n    --hash=sha256:7868b85224291c6cb6759f9b5adb9745f486d226f62b16a614dd5a2a5ab2b35b \\\n    --hash=sha256:815d26356930846a40c7bc1366e7b1b0320ab8a063e66c11298a208bed0fd237 \\\n    --hash=sha256:8601470267d938bcb7f3ab1a336100af51a4fd5b6ed030ef52461bb3ef5e7e07 \\\n    --hash=sha256:88b5268892fde430d5531f95bc560b6efbbd67c929662c586afd729a96e7461c \\\n    --hash=sha256:926bdd3e3b5998ddf70cc64bc8cf57209571f9044542913afb673799fec77dd0 \\\n    --hash=sha256:a36b70596407634ca82d4b989a3729074a008537a0522e4c8046a67c729103e9 \\\n    --hash=sha256:a3a52a3ba86436ab3aef510fbe21512abc2ddd1993005dfe50514bd2284ef025 \\\n    --hash=sha256:addeda51556dac7c1a2f14cda62db8b621cd12afba3091d03a96c72932387eab \\\n    --hash=sha256:b242c27c8f836305a4a72df9cdd564386ac57b807bd252a063223331c9316b37 \\\n    --hash=sha256:b5b8b0753718d258fd454283fbd57e14545d3b40583fa672e27cb4f987626bcc \\\n    --hash=sha256:befc2d6a953e563f8a7bfd87a42c22ebf8a3e980dcb7b6a4d17b70b0e914e8a3 \\\n    --hash=sha256:d1028417bb44037eb3069c1009bd7b7277212876cda22fbe565b0bca9fab6d2c \\\n    --hash=sha256:d151e148117294133bf8af7eeace085e7e87432db15ab6adf640330298a47f6f \\\n    --hash=sha256:d7841166b7fa64c9c56404617ae4341448847482d45933b13135d26c130519e5 \\\n    --hash=sha256:d9edf30457d74eebfd76b045535e36f1cd89062566a128a0db2145ca042d787e \\\n    --hash=sha256:e21c1429e205828ea886a2293a4a2c8e01f4c25d9893ca330e97a6cf73f52e7b \\\n    --hash=sha256:e6ea1cda8d8c688278430e4268a42f5e5da3bdd74578dfadc0820c3f1766ce83 \\\n    --hash=sha256:eba5d173f7d5708b22a93815017a4611873ed54db9f268077c0dd1ed99cfc858 \\\n    --hash=sha256:eef6a03b0b6d08d0835ccfa8ec8d1bc70525e3801387567137b50c557695e6da\n    # via\n    #   jsonschema\n    #   referencing\nscikit-learn==1.9.1 \\\n    --hash=sha256:0c0f8b5d09b44101cea2767f300680bada1ea27f976fe4b48b83950a4f55a49a \\\n    --hash=sha256:2070f271e5375dc42c6bb93b461ab1c0aa5841d4009267e0cfd95a39dca94a43 \\\n    --hash=sha256:2c2b312fd8c02951a364fa120ea08c1cec10d863466bf1701b013152d7537835 \\\n    --hash=sha256:4b59abb30618121cc46b45972d6bf53a7128b4df4cd346c6ca6f4d5f9031e49c \\\n    --hash=sha256:55e79d6e9b0923f1a978179822bd43d7f5543f45e970a00fe861f43486380aba \\\n    --hash=sha256:5990f9c69e431bfaddcde1a6d7c5355243e026bc9b9e560c13893b90dab53fb4 \\\n    --hash=sha256:613f0a783ca05aa844a4e1ac42d48425058f2c52be73f40f8cd98b7cd111acd6 \\\n    --hash=sha256:61cd968ab831a76d0ecbaf0347ab2270268716da28f94fd022497e3d6f205f13 \\\n    --hash=sha256:629cada3e33e2b9bf376cdc7614a47a4140b8aedc1d836579e359736fbd82977 \\\n    --hash=sha256:8c14ce41d561f7749f990b41d6703fe02c4669fbc485e598e069e0a1967b488e \\\n    --hash=sha256:d5945a2908be62350e2978344e62b56c1552c2ca4f844ebf6277c94944d647dd \\\n    --hash=sha256:e4c20a6c017d820faa7ac8c783e3d0c6a9a2e297bf9f55332ca17cdf7fd4d04d \\\n    --hash=sha256:e5d7b18a5b9dca241a74695f3275fa4c895a9dadc72b3d8df5fa9d1083c9b83e\n    # via banking-churn-reduction\nscipy==1.18.1 \\\n    --hash=sha256:3ab3523da44749156e1f68b464dc56af11ae4cbc5c739a49d05f32b982eca9f3 \\\n    --hash=sha256:3c085faa2cfa879c5141df483f836f4d691045a078224a670fa570fa01612d89 \\\n    --hash=sha256:457fd7a2a8edeb044ab6ffbc0aa03ff6cd18491356e5e0c834d76ce621b916d1 \\\n    --hash=sha256:52c4b7422442aba924d03ad4019852b08a92e64ea187b933135687bfe2747307 \\\n    --hash=sha256:559ed65f60c1af5a03f3912605a1b5114f522c7c32fb23c3376ae8f03219fe28 \\\n    --hash=sha256:5e4d44984abc0020154ea81b247adeddcc3ac5527b975ff798bd1ba0adc513c2 \\\n    --hash=sha256:75b00eb8fb802090aa903f4ea1c7f5a584779f967361e68b7e98e531cc2d7174 \\\n    --hash=sha256:78c0665edead396b1abb4897c41a5c1d9bf090c8a637a4c20a61678e0a264e66 \\\n    --hash=sha256:7bbf207c4453ce1ad2e00b17313852b33310b83090c2311bdaf97f93c0380d12 \\\n    --hash=sha256:c35d74ce0e193ff740c2f2be2ac913ddc232fe6c1ff40b26cfecb9c670c63314 \\\n    --hash=sha256:c825cef2f49e46753726a7181a8e199804a912b29519ada542c6ebc654951899 \\\n    --hash=sha256:cd479fc04dd9401e3b4f49e76518768ef99c4f517a98c284eb091fd725719adf \\\n    --hash=sha256:d2924a03db38dc2e848bca2fe9f077dafb891480b91a00a0963a8cf86dfc31c1 \\\n    --hash=sha256:d416b16cccfd70fbf62400e84d0bb2f4e6af519a45557f1692c749b37f14b315 \\\n    --hash=sha256:d65d448389b8436493abcf629cc94ad0cf32aecaf06e1acca1de53cc795f2f12 \\\n    --hash=sha256:e3b417bf8c2c7c16e8f58ad91db17783ec911ac16e7b50eb6eab6e809b4f5b07 \\\n    --hash=sha256:e6fb6a55cc0ba97b59a1f288fb86dc6fce8bdfc0fffcbfd015e3a954bf2a2d93 \\\n    --hash=sha256:e708533e8b2ae2497d65346538a7dcc92814410b25b81432eac66de0f2af8265 \\\n    --hash=sha256:ea324d9dd34c38bfb9bec8ca4d1b407db97dbb74029f566b8e322b1b6fe56fe6 \\\n    --hash=sha256:f55fa87b6c612ecd6b058f167c53231b1d14e412efe361d3d6e38b3631c73218 \\\n    --hash=sha256:fdaf5ea890a6183d0565f51a61799d67081bd5b1cf03c5f4b3fd3732108625c9\n    # via\n    #   banking-churn-reduction\n    #   scikit-learn\n    #   xgboost\nseaborn==0.13.2 \\\n    --hash=sha256:636f8336facf092165e27924f223d3c62ca560b1f2bb5dff7ab7fad265361987 \\\n    --hash=sha256:93e60a40988f4d65e9f4885df477e2fdaff6b73a9ded434c1ab356dd57eefff7\n    # via banking-churn-reduction\nsetuptools==84.2.1 \\\n    --hash=sha256:3ff8b3b5492a0aa66960f4419365a86b84e013070d4313e4d4a6894f5d64d6bc \\\n    --hash=sha256:ca140b45d5325a9478f880e351484f62fef15f336462e449ca0c03c873d589c2\nsix==1.17.0 \\\n    --hash=sha256:4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274 \\\n    --hash=sha256:ff70335d468e7eb6ec65b95b99d3a2836546063f63acc5171de367e834932a81\n    # via python-dateutil\nsoupsieve==3.0.4 \\\n    --hash=sha256:8412826614f9c632227067d0985aeb9f584bd15efc852955430e097a9fe7afbf \\\n    --hash=sha256:924fab84ae21604c19e175e1f39815eb6fa83ba23836e09e9de180925e5bc4c0\n    # via beautifulsoup4\nstack-data==0.6.3 \\\n    --hash=sha256:836a778de4fec4dcd1dcd89ed8abff8a221f58308462e1c4aa2a3cf30148f0b9 \\\n    --hash=sha256:d5558e0c25a4cb0853cddad3d77da9891a08cb85dd9f9f91b9f8cd66e511e695\n    # via ipython\nthreadpoolctl==3.7.0 \\\n    --hash=sha256:61348cfb77d53b9242e0017029244b559b810c142ced65b4e21eeca1843959a7 \\\n    --hash=sha256:cd8b60b5641b45c67bbf73c64c843235fc2d8a480c87389f52f5dbee893b86be\n    # via\n    #   banking-churn-reduction\n    #   scikit-learn\ntornado==6.6 \\\n    --hash=sha256:01abf15429d64f04ccfafc238b191cc2c177108b7a331f4461707ad77198c8c2 \\\n    --hash=sha256:11bb97c00981eb8254e55bd6a5ad960c983ae343fef8ae3ae5584cd1fb70236a \\\n    --hash=sha256:29114ffd4e4de869d390393df9e9a0f191065aae1a96f38a58d35e464c04a867 \\\n    --hash=sha256:31e1a0d5cd704d2fe8a008c1cc27a0e2b67269ca263630a27985dddc160b5537 \\\n    --hash=sha256:33aabe8a03d426cef4e0fe5669f0643cadd983b54957f1a7ac2789ec326e9644 \\\n    --hash=sha256:34c4aa916d7f2b2d5cadee2c6850f5a4a7f3d688cbf58399ea9bd3057af4abd5 \\\n    --hash=sha256:48f26ed9d7beb64e5f06adc864524e581ab6d62ba8ff46eaaa1ac2d0e6d753fd \\\n    --hash=sha256:4e152a3b2d3eae113b836d979823f7bb2320f7e334a84449729964921f374a4c \\\n    --hash=sha256:61fc5513cea97720ab9bf54b04827ac0f4500c63f0b072b963d6145c798b4661 \\\n    --hash=sha256:74416e41e9bdf70fbc2fb242f6fbbc7a58ec11e59d39f15fc5a421ecb6845118 \\\n    --hash=sha256:8cdd69104ae9d705b3f04d2563aa96e357043e267a455802daae9a1e633a37c8 \\\n    --hash=sha256:9aaf0e1c2fb9f046f30e2036b8d2060666399e8464d0a72769e43c754089567b \\\n    --hash=sha256:b54ca69ff93d0374efb083a12a1e9ec2dd2036e8496d197e0e8ef77b8d9135c3 \\\n    --hash=sha256:e912d4c4babd04db6f34b25b748174a4a176debe86c8610beedf1815f9af7fdd \\\n    --hash=sha256:fc7c8a3263501fbe97518cd699101bc3c5f499a7dd4f75fc6cbb3bd9ee1db1f1\n    # via\n    #   ipykernel\n    #   jupyter-client\ntqdm==4.70.1 \\\n    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \\\n    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4\n    # via gdown\ntraitlets==5.16.1 \\\n    --hash=sha256:ed900c2b631aa3a112811139fa97b8d2c3bad5e989656bba4b7e52c7852c18c1 \\\n    --hash=sha256:f775618166caa0396c8e337099240f2bd3e5e917d203b2e6fbe21a58d3cb1f6b\n    # via\n    #   ipykernel\n    #   ipython\n    #   ipywidgets\n    #   jupyter-client\n    #   jupyter-core\n    #   matplotlib-inline\n    #   nbclient\n    #   nbformat\ntyping-extensions==4.16.0 \\\n    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \\\n    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5\n    # via\n    #   beautifulsoup4\n    #   jupyter-client\n    #   referencing\ntzdata==2026.5 \\\n    --hash=sha256:8cc73c0a0bfca7dbfa59235d60b2eff82231dee33f53d206db1acd9173cfc0a7 \\\n    --hash=sha256:b683bd1b6659ddcd810ff02ad09ba821d4bf1065072805063eb35c49617905ac\n    # via\n    #   pandas\n    #   python-dateutil\nurllib3==2.8.0 \\\n    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \\\n    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63\n    # via\n    #   gdown\n    #   requests\nwcwidth==0.9.3 \\\n    --hash=sha256:2f875a4c9e62df2ecfa70d99b24c551f2245609858e179c3a40e74631b2d08a4 \\\n    --hash=sha256:3b119ef60177a92b4f298b80e98464c2922a1fba3ed77fe9a2bcb69edf1b9af4 \\\n    --hash=sha256:537ea00b6792bb243cd09582322c54dbc848879e5d34ea4c5fee0147529685ab \\\n    --hash=sha256:569a0cdb826d3097b6dbacfaa45aa0002d72a2ba6fa8a63251ea43ac4dd7b6a4 \\\n    --hash=sha256:70a2ca239b85a219324968993c76c117fe423d4e1a7c9d860c999c66192f2570 \\\n    --hash=sha256:80557db1f77148b78e1b3c2a9cc4ebc4feabef42aaf69a555565057031c32367 \\\n    --hash=sha256:9ab8bfee107e08de09e15f9f6afb9fe0f64c8d64fd296d0d8557bafe71ea3f93 \\\n    --hash=sha256:9e1cbe18c18d152e1e68e4e2561dab6c2bc988f8fd337a1fb2e1e0809c3fa568 \\\n    --hash=sha256:aab90ab489850f87633922c3064f6d0442d8f6ae5407a62f82bdb1f424179c05 \\\n    --hash=sha256:c12abd75ffcbad417e7770a6feb82413c102ccce5c435068c39a82e2218b00b3 \\\n    --hash=sha256:dfe7cfbd4cb59db9fe0931bd6091ab4a95b3acc5d56bfb50fa9fef0ccba5dc5c \\\n    --hash=sha256:e3c9069c54a2da0bf4f6c7e65a735cb1da0f47601a15cb5fef7e4b6ffca4cf60 \\\n    --hash=sha256:f2f943c4ef14e9baba730740265f4e5bf5fc8473550e36cd92828cd5663561a6\n    # via prompt-toolkit\nwheel==0.48.0 \\\n    --hash=sha256:3217dcc807155e45db462d7ef2431f5ddda0d7273b700d05a67b271ceb1287ab \\\n    --hash=sha256:94800765601e9171bf5d58d066e640662842bcedcbab982b2c90787a2c987322\nwidgetsnbextension==4.0.16 \\\n    --hash=sha256:a31a8774885b96fe825462f5d6496166f0c7cae111195b6465c801d230eb5a4e \\\n    --hash=sha256:adeea0ae78f0856ee4945f413299801b82a0a01416303301f39a704282a37b73\n    # via ipywidgets\nxgboost==3.4.1 \\\n    --hash=sha256:1ea15f15f661825b6a67d87674fb9604a1abb38dd0d4c5cf0486fc85f5203e83 \\\n    --hash=sha256:2d30fa513673101f542fdcbd18f30c8f96c064046f798635ac08663e9969f81b \\\n    --hash=sha256:6968a4c71efdfa859df0dfcad0d99211c95c28c4ffd6aecff46efff77d18026a \\\n    --hash=sha256:6adf2afa396da2ae8ed30295b50b99d4712eed9a6e0ce6cfe069290e4335e51f \\\n    --hash=sha256:7faaf99de26719c22bfae883a02bd56b5a3c2203122616e563cc72b7191b5c96 \\\n    --hash=sha256:a7afd7dbace0951c93aa85ffe046e54bc40893f5b51cd3e7991eb157bf9c7c7c \\\n    --hash=sha256:e9312b30e5679d27c1d8b9ee97e092b964d960a672d5d406d9fb3cd0845c9797\n    # via banking-churn-reduction\n"
import importlib.metadata as package_metadata
import os,subprocess,sys,tempfile
from pathlib import Path
from packaging.requirements import Requirement
if sys.version_info[:2] not in {(3,12),(3,13)}:
    raise RuntimeError("V7 memerlukan Python 3.12 atau 3.13. Versi aktif: "+sys.version.split()[0])
requirements=[]
for line in LOCKED_REQUIREMENTS.splitlines():
    line=line.strip()
    if not line or line.startswith(("#","--")):continue
    entry=line.rstrip(chr(92)).strip()
    req=Requirement(entry)
    if req.marker is None or req.marker.evaluate():requirements.append(req)
missing=[]
for req in requirements:
    try:installed=package_metadata.version(req.name)
    except package_metadata.PackageNotFoundError:installed=None
    if installed is None or not req.specifier.contains(installed,prereleases=True):missing.append(req.name)
if missing:
    if os.getenv("BANK_CHURN_SKIP_INSTALL")=="1":
        raise RuntimeError("Environment terkunci tidak cocok: "+", ".join(missing))
    aliases={"scikit-learn":"sklearn","pillow":"PIL","python-dateutil":"dateutil","pyyaml":"yaml"}
    loaded=any(aliases.get(name,name.replace("-","_")) in sys.modules for name in missing)
    lock_path=Path(tempfile.mkdtemp(prefix="bank-churn-lock-"))/"requirements-lock.txt"
    lock_path.write_text(LOCKED_REQUIREMENTS,encoding="utf-8")
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet","--require-hashes","-r",str(lock_path)])
    if loaded:raise RuntimeError("Paket yang sudah dimuat telah diganti. Runtime > Restart session, lalu Run all kembali.")
print("Python",sys.version.split()[0],"|",len(requirements),"distribusi terkunci cocok dengan environment ini.")

In [ ]:
#@title 1.2 - Muat modul bersama yang terverifikasi
PACKAGE_SOURCES = {'__init__.py': '"""Shared Banking Churn V7 training, inference, decisions and reports."""\n__version__ = "7.0.0"\n', 'cli.py': '"""Command line entrypoints for the same V7 pipeline used in Colab and CI."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport uuid\n\n\ndef main(argv=None):\n    parser=argparse.ArgumentParser(prog="banking-churn")\n    sub=parser.add_subparsers(dest="command",required=True)\n    train=sub.add_parser("train",help="Train on an explicitly supplied CSV")\n    train.add_argument("--data",required=True,type=Path)\n    train.add_argument("--output-base",default="bank_churn_artifacts_v7")\n    train.add_argument("--allow-reviewed-source",action="store_true",help="Explicitly use a reviewed non-reference CSV")\n    smoke=sub.add_parser("smoke",help="Run every stage on labelled synthetic data")\n    smoke.add_argument("--output-base",default="bank_churn_artifacts_v7")\n    smoke.add_argument("--rows",type=int,default=1000)\n    smoke.add_argument("--seed",type=int,default=42)\n    score=sub.add_parser("score",help="Score a CSV with a trusted local bundle")\n    score.add_argument("--bundle",required=True,type=Path)\n    score.add_argument("--input",required=True,type=Path)\n    score.add_argument("--output",required=True,type=Path)\n    report=sub.add_parser("report",help="Refresh aggregate HTML/PDF for an existing run")\n    report.add_argument("--run-dir",required=True,type=Path)\n    args=parser.parse_args(argv)\n    if args.command in {"train","smoke"}:\n        from .pipeline import run_pipeline\n        from .config import TrainingConfig\n        if args.command=="smoke":\n            from .synthetic import write_synthetic_csv\n            path=Path(args.output_base)/"inputs"/("synthetic-"+uuid.uuid4().hex[:8]+".csv")\n            write_synthetic_csv(path,args.rows,args.seed)\n            run=run_pipeline(path,args.output_base,TrainingConfig.quick(args.seed),"synthetic_smoke",False)\n        else:\n            run=run_pipeline(args.data,args.output_base,TrainingConfig(),\n                "external_reviewed" if args.allow_reviewed_source else "reference_source",not args.allow_reviewed_source)\n        print(json.dumps({"run_id":run.run_id,"data_context":run.metadata["data_context"],\n            "synthetic":run.metadata["synthetic"],"checks_passed":len(run.checks),"run_dir":str(run.artifact_dir.resolve()),\n            "model":run.metadata["model_name"]},indent=2))\n    elif args.command=="score":\n        import joblib\n        from .inference import score_customer_csv\n        bundle=joblib.load(args.bundle)\n        result,path=score_customer_csv(args.input,bundle,args.output)\n        print(json.dumps({"rows":len(result),"output":str(path),"policy":bundle["metadata"]["default_policy"]},indent=2))\n    else:\n        from .live import refresh_reports\n        outputs=refresh_reports(args.run_dir)\n        print(json.dumps({key:str(path) for key,path in outputs.items()},indent=2))\n\n\nif __name__=="__main__":\n    main()\n', 'config.py': '"""Explicit training configuration with bounded runtime and no mutable module settings."""\nfrom dataclasses import asdict,dataclass\nimport math\n\n\n@dataclass(frozen=True)\nclass TrainingConfig:\n    seed: int=42\n    n_jobs: int=2\n    outer_folds: int=3\n    inner_folds: int=3\n    search_trials: int=6\n    calibrate: bool=True\n    default_policy: str="balanced_f1"\n    capacity_fraction: float=.2\n    min_validation_precision: float|None=None\n    max_validation_contact_rate: float|None=None\n    bootstrap_resamples: int=500\n    smoke: bool=False\n\n    def validate(self):\n        for name,low,high in [("seed",0,2**32-1),("n_jobs",1,8),("outer_folds",2,10),\n                              ("inner_folds",2,10),("search_trials",1,100),("bootstrap_resamples",20,2000)]:\n            value=getattr(self,name)\n            if isinstance(value,bool) or not isinstance(value,int) or not low<=value<=high:\n                raise ValueError(f"Invalid {name}")\n        if self.default_policy not in {"threshold_0_5","precision_f05","balanced_f1","recall_f2"}:\n            raise ValueError("Unsupported default policy")\n        if not math.isfinite(self.capacity_fraction) or not 0<self.capacity_fraction<=1:\n            raise ValueError("Invalid capacity fraction")\n        for value in [self.min_validation_precision,self.max_validation_contact_rate]:\n            if value is not None and (not math.isfinite(value) or not 0<=value<=1):\n                raise ValueError("Validation constraints must be None or within [0,1]")\n        return self\n\n    def as_dict(self):\n        return asdict(self)\n\n    @classmethod\n    def quick(cls,seed=42):\n        return cls(seed=seed,outer_folds=2,inner_folds=2,search_trials=1,bootstrap_resamples=30,smoke=True)\n', 'data.py': '"""Source loading with reference hash checks; synthetic data must be labelled explicitly."""\nimport hashlib\nimport tempfile\nimport warnings\nfrom pathlib import Path\nimport gdown\nfrom .inference import read_source_csv,normalize_features\nfrom .schema import DATA_DRIVE_ID,DATA_SOURCE_URL,REFERENCE_SHA256\n\ndef load_source_data(data_file, legacy_file=None, strict_reference=True, downloader=None):\n    primary = Path(data_file)\n    legacy = Path(legacy_file) if legacy_file is not None else None\n    temporary = None\n    mode = "local_primary"\n    chosen = primary\n    if not primary.is_file() and legacy is not None and legacy.is_file():\n        chosen, mode = legacy, "local_legacy"\n    elif not primary.is_file():\n        mode = "downloaded"\n        primary.parent.mkdir(parents=True, exist_ok=True)\n        with tempfile.NamedTemporaryFile(prefix=".churn-download-", suffix=".csv", dir=primary.parent, delete=False) as handle:\n            temporary = Path(handle.name)\n        chosen = temporary\n        try:\n            result = (downloader or gdown.download)(id=DATA_DRIVE_ID, output=str(temporary), quiet=False)\n            if not result or not temporary.is_file() or temporary.stat().st_size == 0:\n                raise ValueError("Unduhan tidak menghasilkan file CSV.")\n        except Exception as exc:\n            temporary.unlink(missing_ok=True)\n            raise RuntimeError(\n                "Unduhan sumber gagal. Unggah churn.csv dari DATA_SOURCE_URL ke panel Files Colab, "\n                "lalu jalankan ulang. Nama lama Churn_Modelling.csv juga dikenali."\n            ) from exc\n    try:\n        digest = hashlib.sha256(chosen.read_bytes()).hexdigest()\n        matches = digest == REFERENCE_SHA256\n        if not matches and strict_reference:\n            raise ValueError(\n                "SHA-256 CSV berbeda dari snapshot sumber yang sudah diperiksa. "\n                "Gunakan file dari DATA_SOURCE_URL atau tinjau sumber baru sebelum menonaktifkan STRICT_REFERENCE_HASH."\n            )\n        if not matches:\n            warnings.warn("CSV berbeda dari snapshot sumber. Penggunaan diizinkan secara eksplisit; hasil dapat berubah.")\n        frame = read_source_csv(chosen)\n        if temporary is not None:\n            temporary.replace(primary)  # Hanya file dengan hash/skema yang lolos yang dipasang.\n            chosen = primary\n        report = {\n            "source_url": DATA_SOURCE_URL, "source_drive_id": DATA_DRIVE_ID,\n            "source_drive_filename": "churn.csv", "local_file": str(chosen), "read_mode": mode,\n            "sha256": digest, "reference_matches": matches,\n            "source_rows": len(frame), "source_columns": len(frame.columns),\n        }\n        return frame, report\n    finally:\n        if temporary is not None:\n            temporary.unlink(missing_ok=True)\n', 'decision_tools.py': '"""V6 decision, uncertainty, batch-quality and live-report helpers, embedded in Colab."""\nimport copy\nimport hashlib\nimport json\nimport math\nimport warnings\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\ndef v6_number(value, name, low=0, high=None, integer=False):\n    if isinstance(value, (bool, np.bool_)):\n        raise ValueError(f"{name} harus berupa angka, bukan boolean.")\n    try:\n        number = float(value)\n    except (TypeError, ValueError) as exc:\n        raise ValueError(f"{name} harus berupa angka.") from exc\n    if not np.isfinite(number) or number < low or (high is not None and number > high):\n        raise ValueError(f"{name} berada di luar rentang yang diizinkan.")\n    if integer and not number.is_integer():\n        raise ValueError(f"{name} harus bilangan bulat.")\n    return int(number) if integer else number\n\n\ndef v6_fingerprint(index):\n    values = [str(x) for x in index]\n    return hashlib.sha256(json.dumps(values, separators=(",", ":")).encode()).hexdigest()\n\n\ndef v6_json(value):\n    if isinstance(value, dict):\n        return {str(k): v6_json(v) for k, v in value.items()}\n    if isinstance(value, pd.DataFrame):\n        return v6_json(value.to_dict(orient="records"))\n    if isinstance(value, (list, tuple, np.ndarray)):\n        return [v6_json(x) for x in value]\n    if isinstance(value, (bool, np.bool_)):\n        return bool(value)\n    if isinstance(value, (int, np.integer)):\n        return int(value)\n    if isinstance(value, (float, np.floating)):\n        return float(value) if np.isfinite(value) else None\n    if value is None or isinstance(value, str):\n        return value\n    raise TypeError(f"Nilai tidak dapat diekspor: {type(value).__name__}")\n\n\ndef v6_order(probabilities, index=None):\n    p = np.asarray(probabilities, dtype=float)\n    if p.ndim != 1 or not p.size or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():\n        raise ValueError("Skor harus vektor finite, tidak kosong, antara 0 dan 1.")\n    keys = np.arange(len(p)) if index is None else np.asarray(index)\n    if keys.ndim != 1 or len(keys) != len(p) or pd.isna(keys).any():\n        raise ValueError("Index tidak sesuai skor.")\n    keys = np.array([str(x) for x in keys])\n    if not pd.Index(keys).is_unique:\n        raise ValueError("Index harus unik setelah konversi teks.")\n    return p, np.lexsort((keys, -p))\n\n\ndef v6_policy_advisor(labels, probabilities, locked_policies, max_contacts, budget,\n                      contact_cost, value_per_saved, success_rate, fixed_cost=0,\n                      min_precision=0, index=None):\n    """Choose a conditional scenario using validation only; never updates the model policy."""\n    p, order = v6_order(probabilities, index)\n    y = np.asarray(labels)\n    if y.ndim != 1 or len(y) != len(p) or not np.isin(y, [0, 1]).all():\n        raise ValueError("Label validation harus lengkap, 0/1, dan sesuai skor.")\n    maximum = v6_number(max_contacts, "Batas kontak", high=len(p), integer=True)\n    budget = v6_number(budget, "Anggaran")\n    cost = v6_number(contact_cost, "Biaya kontak")\n    benefit = v6_number(value_per_saved, "Manfaat")\n    success = v6_number(success_rate, "Keberhasilan", high=1)\n    fixed = v6_number(fixed_cost, "Biaya tetap")\n    minimum = v6_number(min_precision, "Minimum precision", high=1)\n    inputs = {"max_contacts": maximum, "budget_um": budget, "contact_cost_um": cost,\n              "value_per_saved_um": benefit, "success_rate_assumption": success,\n              "fixed_cost_um": fixed, "min_validation_precision": minimum}\n\n    cumulative_risk = np.r_[0., np.cumsum(p[order])]\n    cumulative_tp = np.r_[0, np.cumsum(y[order])]\n    k = np.arange(len(p) + 1)\n    total_cost = k * cost + np.where(k > 0, fixed, 0)\n    net = cumulative_risk * success * benefit - total_cost\n    precision = np.divide(cumulative_tp, k, out=np.zeros(len(k)), where=k > 0)\n    eligible = (k > 0) & (k <= maximum) & (total_cost <= budget + 1e-10) & (precision >= minimum)\n    curve = pd.DataFrame({"contacts": k, "risk_sum": cumulative_risk, "tp": cumulative_tp,\n        "validation_precision": precision, "cost_um": total_cost, "expected_net_um": net,\n        "eligible": eligible})\n    feasible_k = k[eligible]\n    best_k = int(feasible_k[np.argmax(net[feasible_k])]) if len(feasible_k) else 0\n    resource_k = int(k[(k <= maximum) & (total_cost <= budget + 1e-10)].max())\n    decisions = {"no_campaign": np.zeros(len(p), dtype=int)}\n    for name, policy in locked_policies.items():\n        if policy.get("kind") == "threshold":\n            threshold = v6_number(policy.get("threshold"), "Threshold", high=1)\n            decisions[name] = (p >= threshold).astype(int)\n        elif policy.get("kind") == "top_k":\n            share = v6_number(policy.get("fraction"), "Fraksi", high=1)\n            decision = np.zeros(len(p), dtype=int)\n            decision[order[:math.ceil(share * len(p))]] = 1\n            decisions[name] = decision\n        else:\n            raise ValueError("Jenis kebijakan terkunci tidak dikenal.")\n    for name, count in [("top_k_resource_limit", resource_k), ("top_k_conditional_value", best_k)]:\n        decision = np.zeros(len(p), dtype=int)\n        decision[order[:count]] = 1\n        decisions[name] = decision\n    rows = []\n    for name, decision in decisions.items():\n        selected = decision.astype(bool)\n        contacts = int(selected.sum())\n        tp = int(y[selected].sum())\n        precision_value = tp / contacts if contacts else None\n        bill = contacts * cost + (fixed if contacts else 0)\n        risk_sum = float(p[selected].sum())\n        resources_ok = contacts <= maximum and bill <= budget + 1e-10\n        quality_ok = bool(contacts and precision_value >= minimum)\n        rows.append({"policy": name, "contacts": contacts, "tp": tp, "fp": contacts - tp,\n            "fn": int(y.sum()) - tp, "validation_precision": precision_value,\n            "validation_recall": tp / int(y.sum()) if y.sum() else None,\n            "risk_sum": risk_sum, "expected_saved": risk_sum * success, "cost_um": bill,\n            "expected_net_um": risk_sum * success * benefit - bill,\n            "resource_feasible": resources_ok, "quality_feasible": quality_ok,\n            "eligible_campaign": resources_ok and quality_ok})\n    candidates = pd.DataFrame(rows)\n    positive = candidates.loc[candidates.eligible_campaign & (candidates.expected_net_um > 1e-10)]\n    if positive.empty:\n        chosen = candidates.loc[candidates.policy == "no_campaign"].iloc[0]\n        status = "no_positive_feasible_campaign"\n    else:\n        chosen = positive.sort_values(["expected_net_um", "contacts", "policy"],\n                                     ascending=[False, True, True]).iloc[0]\n        status = "conditional_candidate_found"\n    return {"inputs": inputs, "status": status, "recommended": v6_json(chosen.to_dict()),\n            "candidates": candidates, "top_k_curve": curve,\n            "data_context": "validation", "business_inputs_are_assumptions": True,\n            "not_a_causal_effect": True, "changes_final_policy": False}\n\n\ndef v6_range(value, name, high=None):\n    if not isinstance(value, (tuple, list)) or len(value) != 2:\n        raise ValueError(f"{name} memerlukan batas bawah dan atas.")\n    lo, hi = (v6_number(x, name, high=high) for x in value)\n    if lo > hi:\n        raise ValueError(f"Batas bawah {name} melebihi batas atas.")\n    return lo, hi\n\n\ndef v6_uncertain_campaign(probabilities, contacts, success_range, cost_range, benefit_range,\n                          fixed_cost=0, n_draws=2000, seed=42, index=None):\n    """Uniform parameter assumptions plus independent outcomes conditional on each draw."""\n    p, order = v6_order(probabilities, index)\n    count = v6_number(contacts, "Kontak", high=len(p), integer=True)\n    draws = v6_number(n_draws, "Draw", low=100, high=10000, integer=True)\n    random_seed = v6_number(seed, "Seed", high=2**32 - 1, integer=True)\n    if count * draws > 50_000_000:\n        raise ValueError("Kontak × draw melebihi 50 juta; kurangi salah satunya.")\n    u_range = v6_range(success_range, "Keberhasilan", high=1)\n    c_range = v6_range(cost_range, "Biaya")\n    b_range = v6_range(benefit_range, "Manfaat")\n    fixed = v6_number(fixed_cost, "Biaya tetap")\n    rng = np.random.default_rng(random_seed)\n    u = rng.uniform(*u_range, size=draws)\n    cost = rng.uniform(*c_range, size=draws)\n    benefit = rng.uniform(*b_range, size=draws)\n    selected_p = p[order[:count]]\n    risk_sum = float(selected_p.sum())\n    total_cost = count * cost + (fixed if count else 0)\n    expected_net = risk_sum * u * benefit - total_cost\n    saved = np.zeros(draws, dtype=int)\n    conditional_variance = np.zeros(draws)\n    if count:\n        for start in range(0, draws, 128):\n            end = min(draws, start + 128)\n            q = u[start:end, None] * selected_p[None, :]\n            saved[start:end] = (rng.random(q.shape) < q).sum(axis=1)\n            conditional_variance[start:end] = benefit[start:end]**2 * (q * (1 - q)).sum(axis=1)\n    realized_net = saved * benefit - total_cost\n    probability = float((realized_net > 0).mean())\n    assumption_sd = float(expected_net.std(ddof=0))\n    conditional_sd = float(np.sqrt(conditional_variance.mean()))\n    summary = {"contacts": count, "risk_sum": risk_sum, "draws": draws, "seed": random_seed,\n        "mean_assumed_expected_net_um": float(expected_net.mean()),\n        "assumed_expected_net_q025_um": float(np.quantile(expected_net, .025)),\n        "assumed_expected_net_q975_um": float(np.quantile(expected_net, .975)),\n        "outcome_net_q025_um": float(np.quantile(realized_net, .025)),\n        "outcome_net_q975_um": float(np.quantile(realized_net, .975)),\n        "profit_fraction_under_assumptions": probability,\n        "profit_fraction_mc_se": float(np.sqrt(probability * (1 - probability) / draws)),\n        "parameter_expected_net_sd_um": assumption_sd,\n        "conditional_outcome_rms_sd_um": conditional_sd,\n        "combined_sd_from_variance_decomposition_um": float(np.hypot(assumption_sd, conditional_sd))}\n    table = pd.DataFrame({"success_assumption": u, "cost_assumption_um": cost,\n        "benefit_assumption_um": benefit, "scenario_expected_net_um": expected_net,\n        "saved_draw": saved, "outcome_net_um": realized_net})\n    return {"summary": summary, "draws": table, "assumptions": {\n        "success_uniform": list(u_range), "cost_uniform_um": list(c_range),\n        "benefit_uniform_um": list(b_range), "fixed_cost_um": fixed,\n        "parameters_independent": True, "outcomes_independent_given_parameters": True,\n        "parameter_ranges_are_user_assumptions": True,\n        "model_calibration_and_population_shift_uncertainty_included": False,\n        "baseline_risk_approximates_no_campaign": True},\n        "data_context": "validation", "not_a_causal_effect": True}\n\n\ndef v6_total_variation(reference, incoming):\n    a, b = np.asarray(reference, dtype=float), np.asarray(incoming, dtype=float)\n    if a.shape != b.shape or not a.size or not np.isfinite(a).all() or not np.isfinite(b).all():\n        raise ValueError("Distribusi tidak sesuai.")\n    if (a < 0).any() or (b < 0).any() or a.sum() <= 0 or b.sum() <= 0:\n        raise ValueError("Frekuensi distribusi harus nonnegatif dan berjumlah positif.")\n    return float(.5 * np.abs(a / a.sum() - b / b.sum()).sum())\n\n\ndef v6_distribution_counts(series, bins=None):\n    if bins is None:\n        # Tagged tuples keep a real string \'__MISSING__\' distinct from a missing value.\n        keys = series.map(lambda x: ("missing", "") if pd.isna(x) else ("value", str(x)))\n        return keys.value_counts().to_dict()\n    numeric = pd.to_numeric(series, errors="raise")\n    return np.r_[np.histogram(numeric.dropna().to_numpy(), bins=bins)[0], numeric.isna().sum()]\n\n\ndef v6_batch_quality(reference, incoming, model_bundle, predict_function,\n                     reference_probabilities, distribution_alert=.15, data_context="external_csv"):\n    alert = v6_number(distribution_alert, "Batas perubahan distribusi", high=1)\n    meta = model_bundle["metadata"]\n    features = meta["features"]\n    invalid = []\n    if not isinstance(incoming, pd.DataFrame) or incoming.empty:\n        invalid.append("Input harus DataFrame dengan minimal satu baris.")\n    elif incoming.columns.duplicated().any():\n        invalid.append("Header duplikat.")\n    else:\n        missing = sorted(set(features) - set(incoming.columns))\n        extra = sorted(set(incoming.columns) - set(features) - {"CustomerId", "Exited"})\n        if missing:\n            invalid.append(f"Kolom wajib hilang: {missing}")\n        if extra:\n            invalid.append(f"Kolom tambahan: {extra}")\n        if not incoming.index.is_unique:\n            invalid.append("Index baris harus unik.")\n        if "CustomerId" in incoming and (incoming.CustomerId.isna().any() or incoming.CustomerId.duplicated().any()):\n            invalid.append("CustomerId harus lengkap dan unik.")\n        if "Exited" in incoming:\n            label = pd.to_numeric(incoming.Exited, errors="coerce")\n            if label.isna().any() or not label.isin([0, 1]).all():\n                invalid.append("Label opsional Exited harus lengkap dan 0/1.")\n    base = {"data_context": data_context, "distribution_alert_assumption": alert,\n            "reference_features": "training", "reference_scores": "validation",\n            "threshold_is_illustrative_not_statistical_significance": True,\n            "model_version": meta.get("revision"), "model_run_id": meta.get("run_id"),\n            "rows": len(incoming) if isinstance(incoming, pd.DataFrame) else 0}\n    scores = pd.DataFrame(columns=["estimated_churn_probability", "flag_for_review", "rank_within_batch", "policy", "decision_threshold"])\n    if not invalid:\n        try:\n            incoming = incoming.copy()\n            if "CustomerId" in incoming:\n                incoming.index = pd.Index(incoming.CustomerId.astype(str), name="CustomerId")\n            for name in meta["numeric_features"] + meta["binary_features"]:\n                incoming[name] = pd.to_numeric(incoming[name], errors="raise").astype(float)\n            for name in meta["categorical_features"]:\n                incoming[name] = incoming[name].map(lambda x: str(x).strip() if pd.notna(x) else np.nan).replace("", np.nan)\n            with warnings.catch_warnings(record=True) as caught:\n                warnings.simplefilter("always")\n                scores = predict_function(incoming.loc[:, features], model_bundle)\n            base["prediction_warnings"] = [str(w.message) for w in caught]\n        except (TypeError, ValueError) as exc:\n            invalid.append(str(exc))\n    if invalid:\n        return {"summary": {**base, "status": "blocked_invalid_schema", "errors": invalid,\n                "scored_rows": 0}, "features": pd.DataFrame(), "scores": scores.iloc[:0].copy()}\n\n    rows = []\n    for name in features:\n        ref, new = reference[name], incoming[name]\n        is_numeric = name in meta["numeric_features"] or name in meta["binary_features"]\n        if is_numeric:\n            ref_num = pd.to_numeric(ref, errors="raise")\n            new_num = pd.to_numeric(new, errors="raise")\n            unique = np.sort(ref_num.dropna().unique())\n            if len(unique) <= 20:\n                # Quantile bins can merge both binary values into one bucket.\n                # Midpoints preserve every observed value for discrete numeric features.\n                cuts = unique[:-1] + (unique[1:] - unique[:-1]) / 2\n            else:\n                cuts = np.unique(np.quantile(ref_num.dropna(), np.linspace(0, 1, 11)))[1:-1]\n            bins = np.r_[-np.inf, cuts, np.inf]\n            a, b = v6_distribution_counts(ref_num, bins), v6_distribution_counts(new_num, bins)\n            distance = v6_total_variation(a, b)\n            outside = float(((new_num < ref_num.min()) | (new_num > ref_num.max())).mean())\n            unseen = 0.\n        else:\n            a, b = v6_distribution_counts(ref), v6_distribution_counts(new)\n            keys = set(a) | set(b)\n            distance = v6_total_variation([a.get(k, 0) for k in keys], [b.get(k, 0) for k in keys])\n            unseen = float((new.notna() & ~new.isin(ref.dropna().unique())).mean())\n            outside = 0.\n        reasons = []\n        if new.isna().mean() > ref.isna().mean() + 1e-12:\n            reasons.append("missing_bertambah")\n        if unseen:\n            reasons.append("kategori_baru")\n        if outside:\n            reasons.append("di_luar_rentang_train")\n        if distance > alert:\n            reasons.append("distribusi_berubah")\n        rows.append({"feature": name, "reference_missing_rate": float(ref.isna().mean()),\n            "incoming_missing_rate": float(new.isna().mean()), "unseen_category_rate": unseen,\n            "out_of_train_range_rate": outside, "total_variation_distance": distance,\n            "flagged": bool(reasons), "reasons": "; ".join(reasons)})\n    table = pd.DataFrame(rows)\n    ref_p, _ = v6_order(reference_probabilities)\n    new_p = scores.estimated_churn_probability.to_numpy()\n    score_shift = v6_total_variation(np.histogram(ref_p, np.linspace(0, 1, 11))[0],\n                                    np.histogram(new_p, np.linspace(0, 1, 11))[0])\n    summary = {**base, "status": "review_needed" if table.flagged.any() or score_shift > alert else "no_alert_at_selected_threshold",\n        "errors": [], "scored_rows": len(scores), "flagged_features": table.loc[table.flagged, "feature"].tolist(),\n        "score_distribution_tv": score_shift, "mean_estimated_risk": float(new_p.mean()),\n        "flagged_for_review": int(scores.flag_for_review.sum()),\n        "accuracy_not_estimated_without_labels": "Exited" not in incoming,\n        "distribution_alert_proves_performance_drop": False}\n    if "Exited" in incoming:\n        y = pd.to_numeric(incoming.Exited).to_numpy()\n        pred = scores.flag_for_review.to_numpy()\n        tp = int(((pred == 1) & (y == 1)).sum())\n        summary["label_audit"] = {"positive_labels": int(y.sum()), "true_positive": tp,\n            "precision": tp / int(pred.sum()) if pred.sum() else None,\n            "recall": tp / int(y.sum()) if y.sum() else None,\n            "is_external_validation": data_context == "external_csv",\n            "labels_used_to_refit_or_select_policy": False}\n    return {"summary": summary, "features": table, "scores": scores}\n\n\ndef v6_compare_versions(baseline, current):\n    fields = ["data_sha256", "split_fingerprints", "features", "seed", "default_policy", "capacity_fraction"]\n    differences = [k for k in fields if baseline.get(k) != current.get(k)]\n    versions_differ = baseline.get("versions") != current.get("versions")\n    rows = []\n    for name, old_value in baseline["metrics"].items():\n        value = current["metrics"].get(name)\n        comparable = not differences and old_value is not None and value is not None\n        rows.append({"metric": name, "v5_reference": old_value, "v6_current": value,\n                     "delta_v6_minus_v5": value - old_value if comparable else None,\n                     "data_and_protocol_match": not differences})\n    return {"table": pd.DataFrame(rows), "summary": {"baseline_run_id": baseline["run_id"],\n        "current_run_id": current["run_id"], "baseline_environment": baseline.get("environment"),\n        "current_environment": current.get("environment"), "protocol_mismatches": differences,\n        "package_versions_differ": versions_differ,\n        "timing_is_hardware_dependent": True,\n        "comparison_is_descriptive_not_independent_validation": True,\n        "selection_uses_this_comparison": False}}\n\n\ndef v6_test_narrative(context):\n    d, m = context["default_metrics"], context["test_scores"]\n    return (\n        f"# Interpretasi otomatis · run {context[\'run_id\']}\\n\\n"\n        f"Model **{context[\'model\']}**, dipilih pada train, memberi ROC-AUC **{m[\'roc_auc\']:.3f}** "\n        f"dan AP **{m[\'average_precision\']:.3f}** pada test berisi {m[\'test_rows\']:,} baris "\n        f"dengan {m[\'test_churn\']:,} label churn. Prevalensi test {m[\'test_prevalence\']:.2%}.\\n\\n"\n        f"Default **{context[\'default_policy\']}**, threshold **{context[\'threshold\']:.6f}** dari validation: "\n        f"{int(d[\'contacts\']):,} prioritas ({d[\'contact_rate\']:.2%}), {int(d[\'tp\'])} churn terdeteksi, "\n        f"{int(d[\'fp\'])} alarm keliru, dan {int(d[\'fn\'])} kasus terlewat. "\n        f"Precision **{d[\'precision\']:.1%}**, recall **{d[\'recall\']:.1%}**.\\n\\n"\n        "ROC-AUC menjelaskan ranking; AP merangkum precision–recall. Keduanya berbeda dari accuracy "\n        "dan precision pada satu threshold. Flag merupakan prioritas peninjauan; deteksi label historis "\n        "belum mengukur pelanggan yang terselamatkan. Cocokkan jumlah prioritas dengan kapasitas dan "\n        "biaya operasional melalui validation. Test lama ini tetap audit internal yang telah dilihat "\n        "selama pengembangan; validasi pada data baru dan pilot terkontrol masih diperlukan.\\n\\n"\n        f"Sumber SHA-256: `{context[\'data_sha256\']}`. Angka dihitung dari run ini.\\n"\n    )\n\n\ndef v6_refresh_report(directory, context):\n    directory = Path(directory)\n    sections = {}\n    for name in ["advisor", "uncertainty", "comparison", "batch"]:\n        path = directory / f"v6_{name}_latest.json"\n        if path.is_file():\n            item = json.loads(path.read_text(encoding="utf-8"))\n            if item["run_id"] != context["run_id"] or item["source_sha256"] != context["data_sha256"]:\n                raise ValueError("Snapshot berasal dari run/sumber lain.")\n            sections[name] = item\n    report = v6_test_narrative(context)\n    if "advisor" in sections:\n        a = sections["advisor"]["summary"]\n        best = a["recommended"]\n        report += "\\n## Keputusan skenario terbaru · validation\\n\\n"\n        if a["status"] == "conditional_candidate_found":\n            report += (f"Kandidat bersyarat: **{best[\'policy\']}**, {best[\'contacts\']} kontak, "\n                       f"biaya {best[\'cost_um\']:,.2f} UM, expected net {best[\'expected_net_um\']:,.2f} UM. ")\n        else:\n            report += "**Tidak ada kampanye bernilai positif yang memenuhi batas input.** Pilihan skenario: tidak menjalankan kampanye. "\n        report += "Keputusan memakai asumsi biaya, manfaat, dan keberhasilan; tidak mengganti kebijakan audit model.\\n"\n    if "uncertainty" in sections:\n        s = sections["uncertainty"]["summary"]["results"]\n        report += (f"\\n## Ketidakpastian skenario terbaru\\n\\nDengan {s[\'contacts\']} kontak tetap, rentang kuantil "\n                   f"outcome net 2,5%–97,5% adalah {s[\'outcome_net_q025_um\']:,.2f} sampai "\n                   f"{s[\'outcome_net_q975_um\']:,.2f} UM; proporsi draw positif "\n                   f"{s[\'profit_fraction_under_assumptions\']:.1%}. Rentang parameter adalah asumsi pengguna, "\n                   "sehingga proporsi ini bukan peluang keuntungan yang terverifikasi dari kampanye nyata.\\n")\n    if "comparison" in sections:\n        s = sections["comparison"]["summary"]\n        report += (f"\\n## Pembanding V5\\n\\nBaseline {s[\'baseline_run_id\']}; perbedaan protokol: "\n                   f"{s[\'protocol_mismatches\'] or \'tidak ada\'}. Paket berbeda: {s[\'package_versions_differ\']}. "\n                   "Tabel perbandingan bersifat deskriptif; pemilihan model tetap pada train.\\n")\n    if "batch" in sections:\n        s = sections["batch"]["summary"]\n        report += (f"\\n## Pemeriksaan batch terbaru\\n\\nKonteks **{s[\'data_context\']}**, status **{s[\'status\']}**, "\n                   f"{s[\'scored_rows\']} baris berhasil diskor. ")\n        if s.get("flagged_features"):\n            report += "Perlu tinjauan: " + ", ".join(s["flagged_features"]) + ". "\n        if s.get("errors"):\n            report += "Kesalahan input: " + "; ".join(s["errors"]) + ". "\n        report += "Perubahan distribusi tidak langsung membuktikan penurunan akurasi.\\n"\n    report += ("\\n## Tindakan berikutnya\\n\\nKonfirmasi horizon churn dan satuan manfaat; kumpulkan data baru "\n               "yang sesuai; uji penawaran terhadap kelompok kontrol sebelum memperluas program. "\n               "Contoh validation dan data stres sintetis di notebook berfungsi sebagai demonstrasi.\\n")\n    payload = v6_json({"run_id": context["run_id"], "source_sha256": context["data_sha256"],\n        "updated_at_utc": datetime.now(timezone.utc).isoformat(), "model_context": context,\n        "latest_sections": sections})\n    (directory / "current_report.md").write_text(report, encoding="utf-8")\n    (directory / "current_report.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    return report\n\n\ndef v6_save_section(kind, inputs, summary, directory, context, tables=None):\n    if kind not in {"advisor", "uncertainty", "comparison", "batch"}:\n        raise ValueError("Bagian V6 tidak dikenal.")\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    payload = v6_json({"kind": kind, "run_id": context["run_id"],\n        "source_sha256": context["data_sha256"], "created_at_utc": datetime.now(timezone.utc).isoformat(),\n        "inputs": inputs, "summary": summary})\n    target = directory / f"v6_{kind}_latest.json"\n    temporary = target.with_suffix(".json.tmp")\n    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    temporary.replace(target)\n    for name, table in (tables or {}).items():\n        if not name.replace("_", "").isalnum():\n            raise ValueError("Nama tabel ekspor tidak valid.")\n        table.to_csv(directory / f"v6_{kind}_{name}_latest.csv", index=True if name == "scores" else False)\n    report = v6_refresh_report(directory, context)\n    base = directory.parent.parent\n    manifest_path = base / "latest_run.json"\n    if manifest_path.is_file():\n        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n        if manifest.get("run_id") == context["run_id"] and manifest.get("status") == "complete":\n            import shutil\n            shutil.make_archive(str(base / context["run_id"]), "zip", root_dir=directory)\n    return target, report\n', 'inference.py': '"""Shared schema validation and inference used by V7 notebooks, CLI and tests."""\nimport csv\nimport math\nimport tempfile\nimport warnings\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom .schema import TARGET,NUMERIC_COLS,BINARY_COLS,CATEGORY_COLS,FEATURES,NUMERIC_RULES\n\n\ndef read_source_csv(path, required_columns=None, dtype=None):\n    # pandas mengganti header duplikat; periksa header asli terlebih dahulu.\n    with Path(path).open("r", encoding="utf-8-sig", newline="") as handle:\n        reader = csv.reader(handle, strict=True)\n        header = next((row for row in reader if row), None)\n        if not header:\n            raise ValueError("CSV kosong atau tidak memiliki header.")\n        header = [name.strip() for name in header]\n        if any(not name for name in header):\n            raise ValueError("Header CSV mengandung nama kolom kosong.")\n        if len(header) != len(set(header)):\n            raise ValueError("Header CSV duplikat setelah pembersihan.")\n        expected = FEATURES + [TARGET] if required_columns is None else list(required_columns)\n        absent = sorted(set(expected) - set(header))\n        if absent:\n            raise ValueError(f"Kolom wajib tidak tersedia: {absent}")\n        for row in reader:\n            if row and len(row) != len(header):\n                raise ValueError(f"Jumlah kolom tidak konsisten pada baris CSV {reader.line_num}.")\n    frame = pd.read_csv(path, encoding="utf-8-sig", header=0, names=header, dtype=dtype)\n    if frame.empty:\n        raise ValueError("CSV tidak memiliki baris data.")\n    return frame\n\n\ndef normalize_features(frame, strict_columns=False, numeric_rules=None):\n    if not isinstance(frame, pd.DataFrame) or frame.empty:\n        raise ValueError("Input harus DataFrame berisi minimal satu baris.")\n    if frame.columns.duplicated().any():\n        raise ValueError("Kolom input duplikat.")\n    absent = sorted(set(FEATURES) - set(frame.columns))\n    if absent:\n        raise ValueError(f"Fitur wajib tidak tersedia: {absent}")\n    extra = sorted(set(frame.columns) - set(FEATURES))\n    if strict_columns and extra:\n        raise ValueError(f"Kolom prediksi tidak dikenal: {extra}. Pilih hanya fitur model sebelum prediksi.")\n    clean = frame.loc[:, FEATURES].copy()\n    rules = NUMERIC_RULES if numeric_rules is None else numeric_rules\n    for name in NUMERIC_COLS + BINARY_COLS:\n        clean[name] = pd.to_numeric(clean[name], errors="raise").astype(float)\n        if np.isinf(clean[name]).any():\n            raise ValueError(f"{name} mengandung infinity.")\n        nonmissing = clean[name].dropna()\n        rule = rules.get(name, {})\n        if "minimum" in rule and (nonmissing < rule["minimum"]).any():\n            raise ValueError(f"{name} di bawah batas minimum {rule[\'minimum\']} pada skema sumber.")\n        if "maximum" in rule and (nonmissing > rule["maximum"]).any():\n            raise ValueError(f"{name} di atas batas maksimum {rule[\'maximum\']} pada skema sumber.")\n        if rule.get("integer") and (nonmissing != np.floor(nonmissing)).any():\n            raise ValueError(f"{name} harus bilangan bulat sesuai skema sumber.")\n    for name in BINARY_COLS:\n        if not clean[name].dropna().isin([0, 1]).all():\n            raise ValueError(f"{name} harus 0, 1, atau missing.")\n    for name in CATEGORY_COLS:\n        clean[name] = clean[name].map(lambda v: str(v).strip() if pd.notna(v) else np.nan)\n        clean[name] = clean[name].replace("", np.nan).astype(object)\n    return clean\n\n\ndef validate_probabilities(probabilities):\n    p = np.asarray(probabilities, dtype=float)\n    if p.ndim != 1 or p.size == 0 or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():\n        raise ValueError("Probabilitas harus vektor finite, tidak kosong, dan berada di antara 0 dan 1.")\n    return p\n\n\ndef rank_order(probabilities, index=None):\n    p = validate_probabilities(probabilities)\n    keys = np.arange(len(p)) if index is None else np.asarray(index)\n    if keys.ndim != 1 or len(keys) != len(p) or pd.isna(keys).any():\n        raise ValueError("Index ranking tidak sesuai skor atau mengandung nilai kosong.")\n    text_keys = np.asarray([str(key) for key in keys])\n    if not pd.Index(text_keys).is_unique:\n        raise ValueError("Index ranking harus unik, termasuk setelah diubah menjadi teks.")\n    return np.lexsort((text_keys, -p))\n\n\ndef apply_policy(probabilities, policy, index=None):\n    p = validate_probabilities(probabilities)\n    kind = policy.get("kind")\n    if kind == "threshold":\n        threshold = policy.get("threshold")\n        if threshold is None or not np.isfinite(threshold) or not 0 <= threshold <= 1:\n            raise ValueError("Threshold kebijakan tidak valid.")\n        return (p >= threshold).astype(int)\n    if kind == "top_k":\n        fraction = policy.get("fraction")\n        if fraction is None or not np.isfinite(fraction) or not 0 <= fraction <= 1:\n            raise ValueError("Fraksi kapasitas harus antara 0 dan 1.")\n        k = min(len(p), math.ceil(fraction * len(p)))\n        decision = np.zeros(len(p), dtype=int)\n        decision[rank_order(p, index)[:k]] = 1\n        return decision\n    raise ValueError(f"Jenis kebijakan tidak dikenal: {kind}")\n\n\ndef predict_customers(frame, model_bundle, policy_name=None):\n    meta = model_bundle["metadata"]\n    if meta["features"] != FEATURES:\n        raise ValueError("Skema fitur notebook berbeda dari bundle.")\n    clean = normalize_features(frame, strict_columns=True, numeric_rules=meta["numeric_rules"])\n    for column in meta["categorical_features"]:\n        unseen = sorted(set(clean[column].dropna()) - set(meta["training_categories"][column]))\n        if unseen:\n            warnings.warn(f"Kategori baru pada {column}: {unseen}; skor di luar dukungan kategori train.")\n    for column, bounds in meta["training_numeric_ranges"].items():\n        outside = (clean[column] < bounds["minimum"]) | (clean[column] > bounds["maximum"])\n        if outside.any():\n            warnings.warn(f"{column} di luar rentang train pada {int(outside.sum())} baris.")\n    estimator = model_bundle["estimator"]\n    class_index = int(np.flatnonzero(estimator.classes_ == meta["positive_class"])[0])\n    score = validate_probabilities(estimator.predict_proba(clean)[:, class_index])\n    if "policies" in meta:\n        name = meta["default_policy"] if policy_name is None else policy_name\n        if name not in meta["policies"]:\n            raise ValueError(f"Kebijakan tidak tersedia dalam bundle: {name}")\n        policy = meta["policies"][name]\n    else:  # Kompatibilitas untuk bundle review 2.\n        name = "stored_threshold"\n        policy = {"kind": "threshold", "threshold": meta["threshold"]}\n    order = rank_order(score, frame.index)\n    ranks = np.empty(len(score), dtype=int)\n    ranks[order] = np.arange(1, len(score) + 1)\n    return pd.DataFrame({\n        "estimated_churn_probability": score,\n        "flag_for_review": apply_policy(score, policy, frame.index),\n        "rank_within_batch": ranks, "policy": name,\n        "decision_threshold": policy.get("threshold", np.nan),\n    }, index=frame.index)\n\n\ndef score_customer_csv(input_path, model_bundle, output_path=None, policy_name=None):\n    input_path = Path(input_path)\n    frame = read_source_csv(input_path, required_columns=FEATURES, dtype={"CustomerId": "string"})\n    extra = sorted(set(frame.columns) - set(FEATURES) - {"CustomerId"})\n    if extra:\n        raise ValueError(f"Batch prediksi hanya menerima fitur dan CustomerId opsional. Kolom tambahan: {extra}")\n    if "CustomerId" in frame:\n        if frame["CustomerId"].isna().any() or not frame["CustomerId"].is_unique:\n            raise ValueError("CustomerId pada batch harus lengkap dan unik.")\n        frame = frame.set_index("CustomerId")\n    else:\n        frame.index = pd.Index(np.arange(1, len(frame) + 1), name="row_number")\n    result = predict_customers(frame.loc[:, FEATURES], model_bundle, policy_name)\n    target = Path(output_path) if output_path is not None else input_path.with_name(input_path.stem + "_scored.csv")\n    if target.resolve() == input_path.resolve():\n        raise ValueError("Output prediksi tidak boleh menimpa CSV input.")\n    target.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(prefix=".scores-", suffix=".csv", dir=target.parent, delete=False) as handle:\n        temporary = Path(handle.name)\n    try:\n        result.to_csv(temporary, index=True)\n        temporary.replace(target)\n    finally:\n        temporary.unlink(missing_ok=True)\n    return result, target\n', 'live.py': '"""Keep V7 report exports and the current run archive in sync after form changes."""\nimport json\nfrom pathlib import Path\nimport shutil\n\nfrom .decision_tools import v6_save_section\nfrom .reporting import generate_executive_reports\n\n\ndef refresh_reports(directory):\n    directory=Path(directory)\n    outputs=generate_executive_reports(directory)\n    meta=json.loads((directory/"model_metadata.json").read_text(encoding="utf-8"))\n    base=directory.parent.parent\n    latest=base/"latest_run.json"\n    if latest.is_file():\n        manifest=json.loads(latest.read_text(encoding="utf-8"))\n        if manifest.get("run_id")==meta["run_id"] and manifest.get("status")=="complete":\n            shutil.make_archive(str(base/meta["run_id"]),"zip",root_dir=directory)\n    return outputs\n\n\ndef save_live_section(kind,inputs,summary,directory,context,tables=None):\n    path,report=v6_save_section(kind,inputs,summary,directory,context,tables)\n    refresh_reports(directory)\n    return path,report\n', 'pipeline.py': '"""Shared V7 training/evaluation pipeline used by notebooks, CLI and integration tests."""\nfrom dataclasses import dataclass\nfrom datetime import datetime,timezone\nimport ast\nimport hashlib\nimport importlib.metadata as package_metadata\nimport json\nimport platform\nfrom pathlib import Path\nimport shutil\nimport time\nimport uuid\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.calibration import CalibratedClassifierCV\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.ensemble import HistGradientBoostingClassifier,RandomForestClassifier\nfrom sklearn.inspection import permutation_importance\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (average_precision_score,brier_score_loss,confusion_matrix,log_loss,roc_auc_score)\nfrom sklearn.model_selection import StratifiedKFold,train_test_split\nfrom threadpoolctl import threadpool_limits\nfrom xgboost import XGBClassifier\n\nfrom . import __version__\nfrom .config import TrainingConfig\nfrom .data import load_source_data\nfrom .decision_tools import (v6_compare_versions,v6_fingerprint,v6_json,v6_policy_advisor,\n    v6_refresh_report,v6_save_section,v6_uncertain_campaign,v6_batch_quality)\nfrom .inference import normalize_features,predict_customers,apply_policy\nfrom .schema import (TARGET,FEATURES,NUMERIC_COLS,BINARY_COLS,CATEGORY_COLS,NUMERIC_RULES,\n    DATA_SOURCE_URL,DATA_DRIVE_ID,SOURCE_NOTEBOOK_ID,REFERENCE_SHA256)\nfrom .training_core import nested_compare,make_search,choose_threshold,policy_metrics,bootstrap_intervals\n\n\n@dataclass\nclass TrainingRun:\n    model: object\n    bundle: dict\n    metadata: dict\n    artifact_dir: Path\n    frames: dict\n    labels: dict\n    probabilities: dict\n    tables: dict\n    context: dict\n    checks: list\n\n    @property\n    def run_id(self):\n        return self.metadata["run_id"]\n\n\ndef model_specifications(config):\n    specs={\n        "Dummy":{"estimator":DummyClassifier(strategy="prior"),"parameters":{}},\n        "LogisticRegression":{"estimator":LogisticRegression(max_iter=2000,random_state=config.seed),\n            "parameters":{"model__C":[.05,.2,1.,5.,20.],"model__class_weight":[None,"balanced"]}},\n        "RandomForest":{"estimator":RandomForestClassifier(n_jobs=config.n_jobs,random_state=config.seed),\n            "parameters":{"model__n_estimators":[150,300],"model__max_depth":[6,12,None],\n                "model__min_samples_leaf":[2,5,10],"model__max_features":["sqrt",.8],"model__class_weight":[None,"balanced_subsample"]}},\n        "HistGradientBoosting":{"estimator":HistGradientBoostingClassifier(early_stopping=False,random_state=config.seed),\n            "parameters":{"model__max_iter":[150,300],"model__learning_rate":[.04,.08,.12],\n                "model__max_leaf_nodes":[7,15,31],"model__l2_regularization":[0.,1.,5.],"model__min_samples_leaf":[15,30,50]}},\n        "XGBoost":{"estimator":XGBClassifier(objective="binary:logistic",eval_metric="logloss",tree_method="hist",n_jobs=config.n_jobs,random_state=config.seed),\n            "parameters":{"model__n_estimators":[200,400,600],"model__learning_rate":[.03,.06,.1],\n                "model__max_depth":[2,3,4,5],"model__min_child_weight":[1,5,10],"model__subsample":[.8,1.],\n                "model__colsample_bytree":[.8,1.],"model__reg_lambda":[1.,3.,8.]}}\n    }\n    if config.smoke:\n        specs={name:specs[name] for name in ["Dummy","LogisticRegression","HistGradientBoosting"]}\n        specs["HistGradientBoosting"]["parameters"]={"model__max_iter":[40],"model__max_leaf_nodes":[7],"model__min_samples_leaf":[15]}\n        specs["LogisticRegression"]["parameters"]={"model__C":[1.]}\n    return specs\n\n\ndef runtime_versions():\n    packages=["numpy","pandas","scipy","scikit-learn","matplotlib","seaborn","joblib","xgboost", "gdown",\n              "packaging","ipywidgets","threadpoolctl","nbformat","nbclient","ipykernel","reportlab"]\n    return {**{name:package_metadata.version(name) for name in packages},"python":platform.python_version(),"banking-churn-reduction":__version__}\n\n\ndef _segment_audit(frame,labels,prediction):\n    audit=frame[["Geography","Gender","Age"]].copy()\n    audit["AgeGroup"]=pd.cut(audit.Age,[-np.inf,30,45,60,np.inf],labels=["<=30","31-45","46-60",">60"])\n    audit["actual"],audit["predicted"]=labels,prediction\n    rows=[]\n    for attribute in ["Geography","Gender","AgeGroup"]:\n        for group,sub in audit.groupby(attribute,observed=True,dropna=False):\n            positives=int(sub.actual.sum());negative=len(sub)-positives\n            tn,fp,fn,tp=confusion_matrix(sub.actual,sub.predicted,labels=[0,1]).ravel()\n            rows.append({"attribute":attribute,"group":str(group),"n":len(sub),"churn_n":positives,\n                "churn_rate":float(sub.actual.mean()),"recall":tp/positives if positives else np.nan,\n                "false_positive_rate":fp/negative if negative else np.nan,\n                "limited_sample":len(sub)<100 or min(positives,negative)<20})\n    return pd.DataFrame(rows)\n\n\ndef portable_inference_source():\n    """Export exact shared schema/inference code without requiring this project package."""\n    from . import schema,inference\n    schema_source=Path(schema.__file__).read_text(encoding="utf-8")\n    inference_source=Path(inference.__file__).read_text(encoding="utf-8")\n    tree=ast.parse(inference_source)\n    keep=[node for node in tree.body if not (isinstance(node,ast.ImportFrom) and node.level)]\n    return \'"""Portable V7 inference. Load only a trusted bundle with matching package versions."""\\n\'+schema_source+"\\n"+ast.unparse(ast.Module(body=keep,type_ignores=[]))+"\\n"\n\n\ndef _write_json(path,value):\n    Path(path).write_text(json.dumps(v6_json(value),ensure_ascii=False,indent=2,allow_nan=False)+"\\n",encoding="utf-8")\n\n\ndef run_training(data_file,output_base="bank_churn_artifacts_v7",config=None,data_context="reference_source",strict_reference=True):\n    """Train and audit once; no test-based model/policy selection and no directory reuse."""\n    config=(config or TrainingConfig()).validate()\n    if data_context not in {"reference_source","synthetic_smoke","external_reviewed"}:\n        raise ValueError("Data context must be explicit")\n    if data_context=="synthetic_smoke" and (not config.smoke or strict_reference):\n        raise ValueError("Synthetic smoke data requires quick configuration and explicit non-reference mode")\n    if config.smoke and data_context!="synthetic_smoke":\n        raise ValueError("Smoke configuration is reserved for explicitly synthetic data")\n    if not strict_reference and data_context=="reference_source":\n        raise ValueError("Unverified data cannot be labelled as the reference source")\n    with threadpool_limits(limits=config.n_jobs):\n        return _run_training(data_file,Path(output_base),config,data_context,strict_reference)\n\n\ndef _run_training(data_file,base,config,data_context,strict_reference):\n    started=time.perf_counter()\n    run_id=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"-"+uuid.uuid4().hex[:8]\n    directory=base/"runs"/run_id\n    directory.mkdir(parents=True,exist_ok=False)\n    raw,source=load_source_data(Path(data_file),strict_reference=strict_reference)\n    frame=raw.drop_duplicates().copy()\n    if "CustomerId" in frame and (frame.CustomerId.isna().any() or frame.CustomerId.duplicated().any()):\n        raise ValueError("CustomerId must be complete and unique; repeated customers need group/time splitting")\n    labels=pd.to_numeric(frame[TARGET],errors="raise")\n    if labels.isna().any() or not labels.isin([0,1]).all() or labels.nunique()!=2 or labels.value_counts().min()<30:\n        raise ValueError("Target must be complete, binary and have at least 30 records per class")\n    labels=labels.astype(int)\n    features=normalize_features(frame)\n    development,test,yd,yt=train_test_split(features,labels,test_size=.2,stratify=labels,random_state=config.seed)\n    train,valid,ytrain,yvalid=train_test_split(development,yd,test_size=.25,stratify=yd,random_state=config.seed)\n    if train.isna().all().any():\n        raise ValueError("A training feature is entirely missing")\n    frames={"train":train,"validation":valid,"test":test}\n    targets={"train":ytrain,"validation":yvalid,"test":yt}\n    split_fingerprints={key:v6_fingerprint(value.index) for key,value in frames.items()}\n    leaderboard,folds,oof,fold_ids=nested_compare(train,ytrain,model_specifications(config),config.outer_folds,\n        config.inner_folds,config.search_trials,config.seed)\n    name=str(leaderboard.iloc[0].model)\n    search=make_search(model_specifications(config)[name],config.seed+10,config.inner_folds,config.search_trials)\n    if config.calibrate and name!="Dummy":\n        estimator=CalibratedClassifierCV(estimator=search,method="sigmoid",ensemble=False,\n            cv=StratifiedKFold(n_splits=config.outer_folds,shuffle=True,random_state=config.seed+20),n_jobs=1).fit(train,ytrain)\n        fitted_search=estimator.calibrated_classifiers_[0].estimator\n    else:\n        fitted_search=search.fit(train,ytrain);estimator=fitted_search\n    positive=int(np.flatnonzero(estimator.classes_==1)[0])\n    pv=estimator.predict_proba(valid)[:,positive]\n    raw_p=fitted_search.predict_proba(valid)[:,positive]\n    policies={"threshold_0_5":{"kind":"threshold","threshold":.5}}\n    threshold_tables={}\n    for policy,beta in [("precision_f05",.5),("balanced_f1",1.),("recall_f2",2.)]:\n        threshold,table=choose_threshold(yvalid,pv,beta,config.min_validation_precision,config.max_validation_contact_rate)\n        policies[policy]={"kind":"threshold","threshold":threshold,"validation_beta":beta}\n        threshold_tables[policy]=table\n    policies["capacity_top_k"]={"kind":"top_k","fraction":config.capacity_fraction}\n    threshold=float(policies[config.default_policy]["threshold"])\n    # First evaluate test only after family, calibration and policies have been fixed.\n    pt=estimator.predict_proba(test)[:,positive]\n    prediction=apply_policy(pt,policies[config.default_policy],test.index)\n    valid_metrics=pd.DataFrame([policy_metrics(yvalid,pv,p,key,valid.index) for key,p in policies.items()])\n    test_metrics=pd.DataFrame([policy_metrics(yt,pt,p,key,test.index) for key,p in policies.items()])\n    default=test_metrics.loc[test_metrics.policy==config.default_policy].iloc[0]\n    scores={"roc_auc":roc_auc_score(yt,pt),"average_precision":average_precision_score(yt,pt),\n        "brier_score":brier_score_loss(yt,pt),"log_loss":log_loss(yt,pt),"test_rows":len(yt),\n        "test_churn":int(yt.sum()),"test_prevalence":float(yt.mean())}\n    perm=permutation_importance(estimator,valid,yvalid,scoring="average_precision",n_repeats=2 if config.smoke else 5,\n        random_state=config.seed,n_jobs=1)\n    importance=pd.DataFrame({"feature":FEATURES,"ap_decrease_mean":perm.importances_mean,\n        "ap_decrease_sd":perm.importances_std}).sort_values("ap_decrease_mean",ascending=False)\n    capacity=[]\n    for fraction in sorted({.05,.1,.2,.3,config.capacity_fraction}):\n        decision=apply_policy(pt,{"kind":"top_k","fraction":fraction},test.index).astype(bool)\n        k=int(decision.sum());tp=int(yt.to_numpy()[decision].sum())\n        capacity.append({"capacity_fraction":fraction,"contacts":k,"churn_captured":tp,\n            "precision_at_k":tp/k,"recall_at_k":tp/int(yt.sum()),"lift_at_k":(tp/k)/float(yt.mean())})\n    versions=runtime_versions()\n    metadata={"revision":7,"package_version":__version__,"run_id":run_id,"created_at_utc":datetime.now(timezone.utc).isoformat(),\n        "data_context":data_context,"synthetic":data_context=="synthetic_smoke","data_sha256":source["sha256"],\n        "source_reference_matches":bool(source["reference_matches"]),"source_rows":len(frame),"rows_used":len(frame),\n        "source_url":DATA_SOURCE_URL if data_context=="reference_source" else None,\n        "source_notebook_id":SOURCE_NOTEBOOK_ID,"data_drive_id":DATA_DRIVE_ID if data_context=="reference_source" else None,\n        "config":config.as_dict(),"seed":config.seed,"model_name":name,"model":name,\n        "selection_metric":"train_nested_cv_average_precision","outer_folds":config.outer_folds,\n        "inner_folds":config.inner_folds,"search_trials":config.search_trials,"best_parameters":fitted_search.best_params_,\n        "calibrated":config.calibrate and name!="Dummy","calibration_method":"sigmoid" if config.calibrate and name!="Dummy" else None,\n        "threshold":threshold,"default_policy":config.default_policy,"policies":policies,"features":FEATURES,\n        "numeric_features":NUMERIC_COLS,"binary_features":BINARY_COLS,"categorical_features":CATEGORY_COLS,\n        "numeric_rules":NUMERIC_RULES,"training_categories":{c:sorted(train[c].dropna().unique().tolist()) for c in CATEGORY_COLS},\n        "training_numeric_ranges":{c:{"minimum":float(train[c].min()),"maximum":float(train[c].max())} for c in NUMERIC_COLS},\n        "positive_class":1,"versions":versions,"test_scores":scores,"split_fingerprints":split_fingerprints,\n        "test_holdout_status":"synthetic_test_only" if config.smoke else "iterative_internal_holdout",\n        "training_seconds":time.perf_counter()-started,\n        "limits":["Synthetic tests do not validate bank performance" if config.smoke else "No new temporal/external validation",\n            "No causal retention effect estimated","Business parameters remain assumptions"]}\n    bundle={"estimator":estimator,"metadata":v6_json(metadata)}\n    joblib.dump(bundle,directory/"bank_churn_bundle.joblib",compress=3)\n    reloaded=joblib.load(directory/"bank_churn_bundle.joblib")\n    example=pd.DataFrame([{ "CreditScore":650,"Age":45,"Tenure":5,"Balance":50000.,"NumOfProducts":2,\n        "EstimatedSalary":60000.,"HasCrCard":1,"IsActiveMember":1,"Geography":"France","Gender":"Male"}],\n        index=pd.Index(["EXAMPLE-001"],name="example_id"))\n    portable=portable_inference_source()\n    (directory/"predict_customers.py").write_text(portable,encoding="utf-8")\n    standalone={"__name__":"portable_inference_check"}\n    exec(compile(portable,"predict_customers.py","exec"),standalone)\n    tables={"nested_model_comparison":leaderboard,"nested_outer_folds":folds,\n        "validation_policies":valid_metrics,"test_policies":test_metrics,\n        "test_bootstrap_intervals":bootstrap_intervals(yt,pt,threshold,config.bootstrap_resamples,config.seed),\n        "test_capacity":pd.DataFrame(capacity),"validation_importance":importance,\n        "test_segments":_segment_audit(test,yt,prediction),\n        "calibration_diagnostic":pd.DataFrame([{"model":"raw","AP":average_precision_score(yvalid,raw_p),"Brier":brier_score_loss(yvalid,raw_p)},\n            {"model":"calibrated" if metadata["calibrated"] else "final","AP":average_precision_score(yvalid,pv),"Brier":brier_score_loss(yvalid,pv)}]),\n        "split_summary":pd.DataFrame([{"split":k,"rows":len(v),"churn_n":int(v.sum()),"churn_rate":float(v.mean())} for k,v in targets.items()])}\n    context=v6_json({"run_id":run_id,"data_sha256":source["sha256"],"model":name,"test_scores":scores,\n        "default_metrics":default.to_dict(),"threshold":threshold,"default_policy":config.default_policy,\n        "versions":versions,"split_fingerprints":split_fingerprints,"rows_used":len(frame),"source_url":metadata["source_url"],\n        "data_context":data_context,"synthetic":metadata["synthetic"]})\n    checks=[]\n    def check(label,condition):\n        if not bool(condition):raise AssertionError(label)\n        checks.append({"check":label,"status":"PASS"})\n    sets=[set(v.index) for v in frames.values()]\n    check("Three disjoint splits",all(sets[i].isdisjoint(sets[j]) for i in range(3) for j in range(i+1,3)))\n    check("Splits cover every source row",set.union(*sets)==set(features.index))\n    check("Identifiers and target excluded",not ({"CustomerId","Surname","RowNumber",TARGET}&set(FEATURES)))\n    check("Every train row has out-of-fold scores",oof.index.equals(train.index) and np.isfinite(oof.to_numpy()).all())\n    check("Nested selection uses train scores",name==str(leaderboard.iloc[0].model))\n    check("Default threshold is fixed or selected from validation",\n        threshold==.5 if config.default_policy=="threshold_0_5" else np.isclose(pv,threshold).any())\n    check("Test probabilities are finite",np.isfinite(pt).all() and ((pt>=0)&(pt<=1)).all())\n    check("Confusion counts equal test rows",int(default.tp+default.fp+default.fn+default.tn)==len(yt))\n    check("Reload preserves predictions",np.allclose(predict_customers(example,bundle).estimated_churn_probability,\n        predict_customers(example,reloaded).estimated_churn_probability,rtol=1e-12,atol=1e-12))\n    check("Standalone helper preserves predictions",np.allclose(predict_customers(example,bundle).estimated_churn_probability,\n        standalone["predict_customers"](example,reloaded).estimated_churn_probability,rtol=1e-12,atol=1e-12))\n    check("Importance uses raw feature names",set(importance.feature)==set(FEATURES))\n    check("Synthetic/reference context is explicit",metadata["synthetic"]==(data_context=="synthetic_smoke"))\n    for key,table in tables.items():table.to_csv(directory/(key+".csv"),index=False,lineterminator="\\n")\n    for key,value in [("model_metadata",metadata),("test_scores",scores),("validation_checks",checks)]:\n        _write_json(directory/(key+".json"),value)\n    (directory/"requirements-tested.txt").write_text("\\n".join(f"{k}=={v}" for k,v in versions.items() if k not in {"python","banking-churn-reduction"})+"\\n",encoding="utf-8")\n    _write_json(directory/"run_context.json",context)\n    return TrainingRun(estimator,reloaded,v6_json(metadata),directory,frames,targets,\n        {"validation":pv,"test":pt},tables,context,checks)\n\n\ndef initialize_decision_examples(run):\n    """Create explicit validation-only defaults; does not alter the trained model/policies."""\n    pv=run.probabilities["validation"];yv=run.labels["validation"];xv=run.frames["validation"]\n    count=min(400,len(yv));budget=count*10+500\n    advisor=v6_policy_advisor(yv,pv,run.metadata["policies"],count,budget,10,100,.2,500,index=xv.index)\n    v6_save_section("advisor",advisor["inputs"],{"status":advisor["status"],"recommended":advisor["recommended"],\n        "data_context":"validation","business_inputs_are_assumptions":True},run.artifact_dir,run.context,\n        {"candidates":advisor["candidates"],"top_k_curve":advisor["top_k_curve"]})\n    uncertainty=v6_uncertain_campaign(pv,count,[.1,.3],[8,12],[80,120],500,n_draws=500 if run.metadata["config"]["smoke"] else 2000,index=xv.index)\n    v6_save_section("uncertainty",uncertainty["assumptions"],{"results":uncertainty["summary"],"assumptions":uncertainty["assumptions"]},\n        run.artifact_dir,run.context,{"draws":uncertainty["draws"]})\n    batch=v6_batch_quality(run.frames["train"],xv.head(min(200,len(xv))),run.bundle,predict_customers,pv,data_context="illustrative_validation")\n    v6_save_section("batch",{"source":"illustrative_validation"},batch["summary"],run.artifact_dir,run.context,\n        {"features":batch["features"],"scores":batch["scores"]})\n    return advisor,uncertainty,batch\n\n\ndef finish_run(run,initialize_examples=True):\n    """Publish a complete manifest only after reports, archive and export checks succeed."""\n    from .reporting import generate_executive_reports,plot_model_audit\n    plot_model_audit(run)\n    if initialize_examples:\n        initialize_decision_examples(run)\n    reports=generate_executive_reports(run.artifact_dir)\n    for path in reports.values():\n        if not Path(path).is_file() or Path(path).stat().st_size==0:raise AssertionError("Missing report export")\n    if not any(item["check"]=="HTML, PDF and aggregate summary exported" for item in run.checks):\n        run.checks.append({"check":"HTML, PDF and aggregate summary exported","status":"PASS"})\n    _write_json(run.artifact_dir/"validation_checks.json",run.checks)\n    archive=Path(shutil.make_archive(str(run.artifact_dir.parent.parent/run.run_id),"zip",root_dir=run.artifact_dir))\n    import zipfile\n    with zipfile.ZipFile(archive) as z:\n        if z.testzip() is not None:raise AssertionError("Invalid ZIP export")\n        if not {"executive_summary.html","executive_summary.pdf","public_summary.json","predict_customers.py"}<=set(z.namelist()):\n            raise AssertionError("Missing ZIP content")\n    manifest={"run_id":run.run_id,"revision":7,"status":"complete","data_context":run.metadata["data_context"],\n        "synthetic":run.metadata["synthetic"],"run_dir":str(run.artifact_dir.resolve()),"archive":str(archive.resolve()),\n        "source_sha256":run.metadata["data_sha256"],"model":run.metadata["model_name"],"checks_passed":len(run.checks),\n        "test_scores":run.metadata["test_scores"],"package_version":__version__,"config":run.metadata["config"]}\n    base=run.artifact_dir.parent.parent\n    temporary=base/(".latest-"+run.run_id+".json")\n    _write_json(temporary,manifest);temporary.replace(base/"latest_run.json")\n    return manifest\n\n\ndef run_pipeline(data_file,output_base="bank_churn_artifacts_v7",config=None,data_context="reference_source",strict_reference=True):\n    run=run_training(data_file,output_base,config,data_context,strict_reference)\n    finish_run(run)\n    return run\n', 'preprocessing.py': '"""Preprocessing fitted only inside training/CV pipelines."""\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder,StandardScaler\nfrom sklearn.impute import SimpleImputer\nfrom .schema import NUMERIC_COLS,BINARY_COLS,CATEGORY_COLS\n\n\ndef make_preprocessor():\n    return ColumnTransformer([\n        ("numeric", Pipeline([\n            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),\n            ("scaler", StandardScaler()),\n        ]), NUMERIC_COLS),\n        ("binary", SimpleImputer(strategy="most_frequent", keep_empty_features=True), BINARY_COLS),\n        ("category", Pipeline([\n            ("imputer", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),\n            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),\n        ]), CATEGORY_COLS),\n    ], remainder="drop", verbose_feature_names_out=True)\n\n\ndef make_pipeline(estimator):\n    return Pipeline([("preprocess", make_preprocessor()), ("model", estimator)])\n', 'reporting.py': '"""Self-contained aggregate HTML/PDF reports; no customer rows or fitted model publication."""\nimport base64\nfrom datetime import datetime,timezone\nimport hashlib\nimport html\nimport json\nfrom pathlib import Path\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import PrecisionRecallDisplay,RocCurveDisplay\nfrom reportlab.lib import colors\nfrom reportlab.lib.enums import TA_LEFT\nfrom reportlab.lib.pagesizes import A4\nfrom reportlab.lib.styles import ParagraphStyle,getSampleStyleSheet\nfrom reportlab.lib.units import mm\nfrom reportlab.platypus import Image,PageBreak,Paragraph,SimpleDocTemplate,Spacer,Table,TableStyle\n\nNAVY="#18324D"\nTEAL="#138B8D"\nAMBER="#A75E13"\nRED="#AD3C37"\n\n\ndef plot_model_audit(run):\n    y=run.labels["test"];p=run.probabilities["test"]\n    fig,axes=plt.subplots(1,2,figsize=(10.4,3.45))\n    RocCurveDisplay.from_predictions(y,p,ax=axes[0],curve_kwargs={"color":TEAL})\n    axes[0].plot([0,1],[0,1],"--",color="#87939D",linewidth=1)\n    PrecisionRecallDisplay.from_predictions(y,p,ax=axes[1],curve_kwargs={"color":TEAL})\n    axes[1].axhline(float(y.mean()),linestyle="--",color="#87939D",label="Prevalensi test")\n    for axis in axes:\n        axis.grid(alpha=.17);axis.spines[["top","right"]].set_visible(False)\n        axis.tick_params(labelsize=8);axis.legend(fontsize=7)\n    axes[0].set_title("Ranking risiko - test",fontsize=11,color=NAVY)\n    axes[1].set_title("Precision-recall - test",fontsize=11,color=NAVY)\n    badge="DATA SINTETIS / UJI ALUR" if run.metadata["synthetic"] else "DATA REFERENSI / AUDIT INTERNAL"\n    fig.suptitle(badge+" | "+run.metadata["model_name"],fontsize=11,fontweight="bold",color=NAVY)\n    fig.tight_layout(rect=(0,0,1,.93))\n    target=run.artifact_dir/"model_audit.png"\n    fig.savefig(target,dpi=160,bbox_inches="tight",facecolor="white")\n    plt.close(fig)\n    return target\n\n\ndef _load(path,default=None):\n    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else default\n\n\ndef aggregate_summary(directory):\n    directory=Path(directory)\n    meta=_load(directory/"model_metadata.json")\n    if not meta:raise ValueError("Model metadata is required before report export")\n    policies=pd.read_csv(directory/"test_policies.csv")\n    current=policies.loc[policies.policy==meta["default_policy"]].iloc[0]\n    sections={}\n    for kind in ["advisor","uncertainty","comparison","batch"]:\n        item=_load(directory/f"v6_{kind}_latest.json")\n        if item:\n            if item["run_id"]!=meta["run_id"] or item["source_sha256"]!=meta["data_sha256"]:\n                raise ValueError("Cross-run snapshot cannot enter a report")\n            # Include only aggregate results and business assumptions, never row-level scores.\n            sections[kind]={"inputs":item["inputs"] if kind in {"advisor","uncertainty"} else {},"summary":item["summary"]}\n    summary={"report_version":7,"run_id":meta["run_id"],"created_at_utc":datetime.now(timezone.utc).isoformat(),\n        "data_context":meta["data_context"],"synthetic":meta["synthetic"],\n        "dataset_rows":meta["rows_used"],"source_sha256":meta["data_sha256"],\n        "source_reference_matches":meta["source_reference_matches"],"model":meta["model_name"],\n        "test_scores":meta["test_scores"],"default_policy":meta["default_policy"],"threshold":meta["threshold"],\n        "default_metrics":{key:int(current[key]) if key in {"contacts","tp","fp","fn","tn"} else float(current[key])\n            for key in ["contacts","tp","fp","fn","tn","precision","recall","f1","contact_rate"]},\n        "test_policies":policies[["policy","contacts","tp","fp","fn","precision","recall"]].to_dict(orient="records"),\n        "config":meta["config"],"package_version":meta["package_version"],"versions":meta["versions"],\n        "split_fingerprints":meta["split_fingerprints"],"latest_sections":sections,\n        "limits":meta["limits"],"customer_records_included":False,"causal_campaign_effect_estimated":False}\n    return summary\n\n\ndef _fmt(value,digits=2):\n    return "-" if value is None else f"{value:,.{digits}f}"\n\n\ndef _assumption_rows(summary):\n    section=summary["latest_sections"].get("advisor")\n    if not section:return []\n    inp=section["inputs"]\n    return [["Batas kontak",str(inp["max_contacts"])],["Anggaran",_fmt(inp["budget_um"])+" UM"],\n        ["Biaya per kontak C",_fmt(inp["contact_cost_um"])+" UM"],\n        ["Manfaat per retained B",_fmt(inp["value_per_saved_um"])+" UM"],\n        ["Keberhasilan u",f"{inp[\'success_rate_assumption\']:.1%} (asumsi)"],\n        ["Biaya tetap F",_fmt(inp["fixed_cost_um"])+" UM"],\n        ["Minimum precision validation",f"{inp[\'min_validation_precision\']:.1%}"]]\n\n\ndef _decision_text(summary):\n    section=summary["latest_sections"].get("advisor")\n    if not section:return "Belum ada skenario kebijakan."\n    a=section["summary"];best=a["recommended"]\n    if a["status"]!="conditional_candidate_found":\n        return "Tidak ada kampanye bernilai positif yang memenuhi batas input. Pilihan skenario: tidak menjalankan kampanye."\n    return (f"Kandidat bersyarat {best[\'policy\']}: {best[\'contacts\']} kontak, biaya {_fmt(best[\'cost_um\'])} UM, "\n            f"expected net {_fmt(best[\'expected_net_um\'])} UM. Ini hasil perhitungan asumsi, bukan laba terukur.")\n\n\ndef _uncertainty_text(summary):\n    section=summary["latest_sections"].get("uncertainty")\n    if not section:return "Belum ada skenario rentang parameter."\n    s=section["summary"]["results"];a=section["summary"]["assumptions"]\n    return (f"{s[\'contacts\']} kontak tetap. Rentang uniform u {a[\'success_uniform\'][0]:.0%}-{a[\'success_uniform\'][1]:.0%}, "\n        f"C {_fmt(a[\'cost_uniform_um\'][0])}-{_fmt(a[\'cost_uniform_um\'][1])} UM, "\n        f"B {_fmt(a[\'benefit_uniform_um\'][0])}-{_fmt(a[\'benefit_uniform_um\'][1])} UM; "\n        f"F {_fmt(a[\'fixed_cost_um\'])} UM hanya bila ada kontak. "\n        f"Kuantil outcome net 2,5%-97,5%: {_fmt(s[\'outcome_net_q025_um\'])} sampai {_fmt(s[\'outcome_net_q975_um\'])} UM; "\n        f"proporsi draw positif {s[\'profit_fraction_under_assumptions\']:.1%}. "\n        "Distribusi parameter adalah asumsi eksplorasi; ketidakpastian kalibrasi dan perubahan populasi belum tercakup.")\n\n\ndef _ascii(text):\n    for before,after in {"\\u2013":"-","\\u2014":"-","\\u2011":"-","\\u00d7":"x","\\u2264":"<=","\\u2265":">=","\\u03a3":"Sum"}.items():\n        text=str(text).replace(before,after)\n    return str(text)\n\n\ndef _pdf(summary,directory):\n    target=directory/"executive_summary.pdf"\n    styles=getSampleStyleSheet()\n    styles.add(ParagraphStyle(name="Brand",fontName="Helvetica-Bold",fontSize=10,textColor=colors.HexColor(TEAL),spaceAfter=8))\n    styles.add(ParagraphStyle(name="Lead",fontName="Helvetica-Bold",fontSize=23,leading=27,textColor=colors.HexColor(NAVY),spaceAfter=8))\n    styles.add(ParagraphStyle(name="SmallBody",fontName="Helvetica",fontSize=9,leading=13,spaceAfter=7,textColor=colors.HexColor(NAVY)))\n    styles.add(ParagraphStyle(name="TinyNote",fontName="Helvetica",fontSize=7,leading=9,textColor=colors.HexColor("#5D6B76")))\n    styles.add(ParagraphStyle(name="TableCell",fontName="Helvetica",fontSize=8,leading=11,textColor=colors.HexColor(NAVY)))\n    styles.add(ParagraphStyle(name="SectionHead",fontName="Helvetica-Bold",fontSize=13,leading=17,textColor=colors.HexColor(NAVY),spaceBefore=9,spaceAfter=7))\n    P=lambda text,style="SmallBody":Paragraph(html.escape(_ascii(text)),styles[style])\n    doc=SimpleDocTemplate(str(target),pagesize=A4,rightMargin=18*mm,leftMargin=18*mm,topMargin=16*mm,bottomMargin=17*mm,\n        title="Banking Churn V7 - Executive Summary",author="Banking Churn Reduction project")\n    badge="DATA SINTETIS - UJI ALUR, BUKAN HASIL BANK NYATA" if summary["synthetic"] else "DATA REFERENSI - AUDIT INTERNAL"\n    story=[P("BANKING CHURN REDUCTION / V7","Brand"),P("Ringkasan hasil dan keputusan","Lead"),P(badge,"Brand")]\n    d=summary["default_metrics"];m=summary["test_scores"]\n    story.append(P(f"Model {summary[\'model\']} | Test {m[\'test_rows\']:,} catatan | {m[\'test_churn\']:,} label churn"))\n    cards=[]\n    for label,value in [("ROC-AUC",f"{m[\'roc_auc\']:.3f}"),("Average precision",f"{m[\'average_precision\']:.3f}"),\n                        ("Precision default",f"{d[\'precision\']:.1%}"),("Recall default",f"{d[\'recall\']:.1%}")]:\n        cards.append([Paragraph(label,styles["TableCell"]),Spacer(1,3*mm),\n            Paragraph(f\'<font size="20"><b>{value}</b></font>\',\n                ParagraphStyle("CardValue",fontName="Helvetica",fontSize=20,leading=24,textColor=colors.HexColor(NAVY)))])\n    cardtable=Table([cards],colWidths=[43.5*mm]*4)\n    cardtable.setStyle(TableStyle([("BACKGROUND",(0,0),(-1,-1),colors.HexColor("#EDF5F6")),("BOX",(0,0),(-1,-1),.4,colors.HexColor("#D9E5E8")),\n        ("TOPPADDING",(0,0),(-1,-1),10),("BOTTOMPADDING",(0,0),(-1,-1),10),("VALIGN",(0,0),(-1,-1),"TOP")]))\n    story.extend([cardtable,Spacer(1,9*mm)])\n    story.append(P(f"Default {summary[\'default_policy\']} memakai threshold {summary[\'threshold\']:.6f} dari validation. "\n        f"Dari {d[\'contacts\']} prioritas, terdapat {d[\'tp\']} label churn terdeteksi, {d[\'fp\']} alarm keliru, dan {d[\'fn\']} kasus terlewat. "\n        "Deteksi churn belum mengukur pelanggan yang terselamatkan."))\n    if (directory/"model_audit.png").is_file():\n        story.append(Image(str(directory/"model_audit.png"),width=174*mm,height=61*mm,kind="proportional"))\n    story.append(P("Tradeoff kebijakan yang sudah dikunci","SectionHead"))\n    header=["Kebijakan","Kontak","TP","FP","FN","Precision","Recall"]\n    rows=[[P(c,"TableCell") for c in header]]\n    for r in summary["test_policies"]:\n        rows.append([P(str(r["policy"]),"TableCell"),str(int(r["contacts"])),str(int(r["tp"])),str(int(r["fp"])),str(int(r["fn"])),f"{r[\'precision\']:.1%}",f"{r[\'recall\']:.1%}"])\n    table=Table(rows,colWidths=[45*mm,20*mm,16*mm,16*mm,16*mm,30*mm,31*mm],repeatRows=1)\n    table.setStyle(TableStyle([("BACKGROUND",(0,0),(-1,0),colors.HexColor("#E5EEF1")),("LINEBELOW",(0,0),(-1,0),.6,colors.HexColor(TEAL)),\n        ("FONTNAME",(0,1),(-1,-1),"Helvetica"),("FONTSIZE",(0,1),(-1,-1),8),("ALIGN",(1,1),(-1,-1),"RIGHT"),\n        ("VALIGN",(0,0),(-1,-1),"TOP"),("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)]))\n    story.extend([table,Spacer(1,5*mm),P("Model dipilih pada train; kebijakan dipilih pada validation; tabel test merupakan audit setelah kebijakan dikunci. Test referensi ini telah dilihat selama pengembangan dan memerlukan validasi baru.","TinyNote"),PageBreak()])\n    story.extend([P("KEPUTUSAN DAN BATAS PENGGUNAAN","Brand"),P("Baca nilai bersama asumsinya","Lead"),P(badge,"Brand"),P(_decision_text(summary))])\n    assumptions=_assumption_rows(summary)\n    if assumptions:\n        t=Table([[P(a,"TableCell"),P(b,"TableCell")] for a,b in assumptions],colWidths=[93*mm,81*mm])\n        t.setStyle(TableStyle([("ROWBACKGROUNDS",(0,0),(-1,-1),[colors.HexColor("#F1F6F7"),colors.white]),\n            ("VALIGN",(0,0),(-1,-1),"TOP"),("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)]))\n        story.append(t)\n    story.extend([P("Rentang skenario dan risiko","SectionHead"),P(_uncertainty_text(summary)),\n        P("Tindakan berikutnya","SectionHead"),P("1. Konfirmasi definisi serta horizon churn, biaya, manfaat kontribusi, dan kapasitas nyata. "\n        "2. Validasi pada periode atau populasi yang belum dipakai pengembangan. "\n        "3. Uji penawaran terhadap kelompok kontrol sebelum memperluas program."),\n        P("Sumber dan keterlacakan","SectionHead"),P(f"Run: {summary[\'run_id\']} | Versi paket proyek: {summary[\'package_version\']}","TinyNote"),\n        P(f"Konteks: {summary[\'data_context\']} | Baris sumber: {summary[\'dataset_rows\']:,}","TinyNote"),\n        P("SHA-256 sumber: "+summary["source_sha256"][:32]+" "+summary["source_sha256"][32:],"TinyNote"),\n        P("Rujukan metode: scikit-learn.org/stable/modules/classification_threshold.html ; "\n          "scikit-learn.org/stable/modules/calibration.html . Versi paket, konfigurasi, dan hash split tercatat dalam public_summary.json.","TinyNote")])\n    def footer(canvas,document):\n        canvas.saveState();canvas.setStrokeColor(colors.HexColor("#D9E5E8"));canvas.line(18*mm,13*mm,192*mm,13*mm)\n        canvas.setFont("Helvetica",7);canvas.setFillColor(colors.HexColor("#5D6B76"))\n        canvas.drawString(18*mm,9*mm,"Banking Churn V7 | "+("SYNTHETIC" if summary["synthetic"] else "INTERNAL AUDIT"))\n        canvas.drawRightString(192*mm,9*mm,str(document.page));canvas.restoreState()\n    doc.build(story,onFirstPage=footer,onLaterPages=footer)\n    return target\n\n\ndef _html(summary,directory):\n    e=lambda x:html.escape(str(x))\n    d=summary["default_metrics"];m=summary["test_scores"]\n    badge="DATA SINTETIS - UJI ALUR, BUKAN HASIL BANK NYATA" if summary["synthetic"] else "DATA REFERENSI - AUDIT INTERNAL"\n    cards="".join(f\'<div class="card"><span>{e(k)}</span><strong>{e(v)}</strong></div>\' for k,v in [\n        ("ROC-AUC",f"{m[\'roc_auc\']:.3f}"),("Average precision",f"{m[\'average_precision\']:.3f}"),\n        ("Precision default",f"{d[\'precision\']:.1%}"),("Recall default",f"{d[\'recall\']:.1%}")])\n    policy_rows="".join("<tr>"+"".join(f"<td>{e(v)}</td>" for v in [r["policy"],int(r["contacts"]),int(r["tp"]),int(r["fp"]),int(r["fn"]),f"{r[\'precision\']:.1%}",f"{r[\'recall\']:.1%}"])+"</tr>" for r in summary["test_policies"])\n    params="".join(f"<tr><td>{e(a)}</td><td>{e(b)}</td></tr>" for a,b in _assumption_rows(summary))\n    figure=""\n    if (directory/"model_audit.png").is_file():\n        figure=\'<img class="audit" alt="ROC dan precision-recall pada test" src="data:image/png;base64,\'+base64.b64encode((directory/"model_audit.png").read_bytes()).decode()+\'">\'\n    document=f\'\'\'<!doctype html><html lang="id"><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">\n<title>Banking Churn V7 - Ringkasan</title><style>\n:root{{--navy:{NAVY};--teal:{TEAL}}}*{{box-sizing:border-box}}body{{margin:0;background:#edf3f5;color:var(--navy);font:16px/1.65 system-ui,sans-serif}}\nmain{{max-width:1040px;margin:36px auto;background:#fff;padding:38px 46px;border-radius:16px}}.brand{{color:var(--teal);font-weight:750;letter-spacing:.12em;font-size:13px}}\nh1{{font-size:38px;line-height:1.2;margin:12px 0}}h2{{font-size:23px;margin-top:32px}}.badge{{display:inline-block;padding:8px 12px;background:#fff0dc;border:1px solid #e7c38f;border-radius:7px;font-size:13px;font-weight:700}}\n.cards{{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin:25px 0}}.card{{background:#eaf5f5;padding:17px;border-radius:9px}}.card span{{display:block;font-size:12px}}.card strong{{font-size:29px}}\n.audit{{width:100%;height:auto}}table{{border-collapse:collapse;width:100%;font-size:14px}}th,td{{text-align:left;padding:10px;border-bottom:1px solid #dfe8ec}}th{{background:#eaf1f4}}.table-wrap{{overflow:auto}}.note{{font-size:13px;color:#526571}}code{{overflow-wrap:anywhere}}footer{{margin-top:30px;padding-top:14px;border-top:1px solid #dfe8ec;font-size:12px;color:#526571}}\n@media(max-width:650px){{main{{padding:22px;margin:10px}}h1{{font-size:29px}}.cards{{grid-template-columns:repeat(2,1fr)}}}}@media print{{body{{background:white}}main{{margin:0;max-width:none;padding:0}}}}\n</style><main><div class="brand">BANKING CHURN REDUCTION / V7</div><h1>Ringkasan hasil dan keputusan</h1><p class="badge">{e(badge)}</p>\n<p>Model <b>{e(summary[\'model\'])}</b> | Test {m[\'test_rows\']:,} catatan | {m[\'test_churn\']:,} label churn</p><section class="cards">{cards}</section>\n<p>Default <b>{e(summary[\'default_policy\'])}</b>, threshold <b>{summary[\'threshold\']:.6f}</b> dari validation: {d[\'contacts\']} prioritas, {d[\'tp\']} churn terdeteksi, {d[\'fp\']} alarm keliru, dan {d[\'fn\']} kasus terlewat. Deteksi label historis belum mengukur pelanggan yang berhasil dipertahankan.</p>\n{figure}<h2>Tradeoff kebijakan yang sudah dikunci</h2><div class="table-wrap"><table><thead><tr><th>Kebijakan</th><th>Kontak</th><th>TP</th><th>FP</th><th>FN</th><th>Precision</th><th>Recall</th></tr></thead><tbody>{policy_rows}</tbody></table></div>\n<p class="note">Seleksi model pada train; pemilihan threshold pada validation. Test referensi telah dilihat selama pengembangan. Dibutuhkan validasi baru pada periode/populasi lain.</p>\n<h2>Keputusan bersyarat dan asumsi</h2><p>{e(_decision_text(summary))}</p><table>{params}</table><h2>Rentang skenario dan risiko</h2><p>{e(_uncertainty_text(summary))}</p>\n<h2>Tindakan berikutnya</h2><ol><li>Konfirmasi horizon churn, biaya, manfaat kontribusi, serta kapasitas operasional.</li><li>Validasi pada data baru yang belum dipakai pengembangan.</li><li>Uji penawaran terhadap kelompok kontrol sebelum memperluas program.</li></ol>\n<footer>Run <code>{e(summary[\'run_id\'])}</code> | Versi proyek {e(summary[\'package_version\'])}<br>Konteks {e(summary[\'data_context\'])} | {summary[\'dataset_rows\']:,} baris sumber<br>SHA-256 <code>{e(summary[\'source_sha256\'])}</code><br>Parameter distribusi adalah asumsi pengguna. Laporan hanya berisi agregat.<br>Rujukan: <a href="https://scikit-learn.org/stable/modules/classification_threshold.html">threshold</a> dan <a href="https://scikit-learn.org/stable/modules/calibration.html">kalibrasi</a>.</footer></main></html>\'\'\'\n    path=directory/"executive_summary.html";path.write_text(document,encoding="utf-8")\n    return path\n\n\ndef generate_executive_reports(directory):\n    directory=Path(directory)\n    summary=aggregate_summary(directory)\n    public=directory/"public_summary.json"\n    public.write_text(json.dumps(summary,ensure_ascii=False,indent=2,allow_nan=False)+"\\n",encoding="utf-8")\n    if summary["synthetic"] and (directory/"current_report.md").is_file():\n        md=(directory/"current_report.md").read_text(encoding="utf-8")\n        if not md.startswith("# DATA SINTETIS"):\n            (directory/"current_report.md").write_text("# DATA SINTETIS - UJI ALUR, BUKAN HASIL BANK NYATA\\n\\n"+md,encoding="utf-8")\n    outputs={"html":_html(summary,directory),"pdf":_pdf(summary,directory),"summary":public}\n    manifest={"run_id":summary["run_id"],"source_sha256":summary["source_sha256"],"data_context":summary["data_context"],\n        "files":{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in outputs.values()},"customer_records_included":False}\n    (directory/"report_manifest.json").write_text(json.dumps(manifest,indent=2)+"\\n",encoding="utf-8")\n    return outputs\n', 'schema.py': '"""Reference schema and file provenance; no customer records."""\nDATA_SOURCE_URL = "https://drive.google.com/file/d/1_mpH_uzTGHaYIF33pz9Is6KW_wj8cv8-/view?usp=sharing"\nDATA_DRIVE_ID = DATA_SOURCE_URL.split("/d/", 1)[1].split("/", 1)[0]\nREFERENCE_SHA256 = "3996cd1fa372e0db0cd9c0ebac35bbd4e8e3c65fb942bb010c826e7b1eeef0a0"\nSOURCE_NOTEBOOK_ID = "1j2HGhm_mdRzkCASodkoyy102L7qOZ_m9"\nTARGET = "Exited"\nNUMERIC_COLS = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]\nBINARY_COLS = ["HasCrCard", "IsActiveMember"]\nCATEGORY_COLS = ["Geography", "Gender"]\nFEATURES = NUMERIC_COLS + BINARY_COLS + CATEGORY_COLS\nNUMERIC_RULES = {\n    "CreditScore": {"minimum": 0, "integer": True},\n    "Age": {"minimum": 1, "maximum": 120, "integer": True},\n    "Tenure": {"minimum": 0, "integer": True},\n    "Balance": {"minimum": 0}, "NumOfProducts": {"minimum": 1, "integer": True},\n    "EstimatedSalary": {"minimum": 0},\n}\n', 'simulation.py': '"""Shared scenario calculations from the validated V6 lab."""\nimport html\nimport math\nimport warnings\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom IPython.display import HTML, Markdown, clear_output, display\nfrom matplotlib.ticker import FuncFormatter\nSIM_COLORS = {\'navy\': \'#18324D\', \'teal\': \'#138B8D\', \'amber\': \'#C97A28\', \'red\': \'#BD4852\'}\n\ndef _sim_scalar(value, name, minimum=None, maximum=None):\n    if isinstance(value, (bool, np.bool_)):\n        raise ValueError(f\'{name} harus angka, bukan boolean.\')\n    try:\n        number = float(value)\n    except (TypeError, ValueError) as exc:\n        raise ValueError(f\'{name} harus angka yang valid.\') from exc\n    if not np.isfinite(number) or (minimum is not None and number < minimum) or (maximum is not None and number > maximum):\n        raise ValueError(f\'{name} di luar rentang yang diizinkan.\')\n    return number\n\ndef _sim_probabilities(probabilities):\n    result = np.asarray(probabilities, dtype=float)\n    if result.ndim != 1 or result.size == 0 or (not np.isfinite(result).all()) or ((result < 0) | (result > 1)).any():\n        raise ValueError(\'Skor simulasi harus vektor probabilitas finite antara 0 dan 1.\')\n    return result\n\ndef _sim_portfolio(labels, probabilities):\n    score = _sim_probabilities(probabilities)\n    target = np.asarray(labels)\n    if target.ndim != 1 or len(target) != len(score) or (not np.isin(target, [0, 1]).all()):\n        raise ValueError(\'Label simulasi harus vektor 0/1 dengan panjang yang sama seperti skor.\')\n    return (target.astype(int), score)\n\ndef _sim_order(score, index=None):\n    keys = np.arange(len(score)) if index is None else np.asarray(index)\n    if keys.ndim != 1 or len(keys) != len(score) or pd.isna(keys).any():\n        raise ValueError(\'Index simulasi harus lengkap dan sesuai jumlah skor.\')\n    text_keys = np.asarray([str(key) for key in keys])\n    if not pd.Index(text_keys).is_unique:\n        raise ValueError(\'Index simulasi harus unik setelah diubah menjadi teks.\')\n    return (np.lexsort((text_keys, -score)), keys)\n\ndef _sim_counts(target, decision):\n    tp = int(((target == 1) & (decision == 1)).sum())\n    fp = int(((target == 0) & (decision == 1)).sum())\n    fn = int(((target == 1) & (decision == 0)).sum())\n    tn = int(((target == 0) & (decision == 0)).sum())\n    contacts, positives = (tp + fp, tp + fn)\n    denominator = 2 * tp + fp + fn\n    return {\'rows\': len(target), \'contacts\': contacts, \'contact_rate\': contacts / len(target), \'tp\': tp, \'fp\': fp, \'fn\': fn, \'tn\': tn, \'precision\': tp / contacts if contacts else np.nan, \'recall\': tp / positives if positives else np.nan, \'f1\': 2 * tp / denominator if denominator else np.nan, \'accuracy\': (tp + tn) / len(target), \'prevalence\': positives / len(target)}\n\ndef profile_what_if(baseline, changes, model_bundle, threshold, predictor):\n    cutoff = _sim_scalar(threshold, \'threshold\', 0, 1)\n    if not isinstance(baseline, pd.DataFrame) or len(baseline) != 1 or (not isinstance(changes, dict)):\n        raise ValueError(\'Baseline harus DataFrame satu baris dan changes harus dictionary.\')\n    unknown = set(changes) - set(baseline.columns)\n    if unknown:\n        raise ValueError(f\'Nama fitur skenario tidak tersedia: {sorted(unknown)}\')\n    scenario = baseline.copy(deep=True)\n    for name, value in changes.items():\n        scenario[name] = value\n    combined = pd.concat([baseline, scenario], ignore_index=True)\n    combined.index = pd.Index([\'Baseline\', \'Skenario\'], name=\'profile\')\n    with warnings.catch_warnings(record=True) as recorded:\n        warnings.simplefilter(\'always\')\n        prediction = predictor(combined, model_bundle)\n    score = _sim_probabilities(prediction[\'estimated_churn_probability\'].to_numpy())\n    if len(score) != 2:\n        raise ValueError(\'Predictor harus mengembalikan satu skor untuk setiap profil.\')\n    equal = baseline.eq(scenario).iloc[0] | (baseline.isna() & scenario.isna()).iloc[0]\n    changed = pd.DataFrame({\'baseline\': baseline.iloc[0], \'scenario\': scenario.iloc[0]}).loc[~equal]\n    comparison = pd.DataFrame({\'estimated_risk\': score, \'threshold\': cutoff, \'flag_for_review\': (score >= cutoff).astype(int)}, index=combined.index)\n    return {\'comparison\': comparison, \'feature_changes\': changed, \'risk_change_pp\': float((score[1] - score[0]) * 100), \'warnings\': [str(item.message) for item in recorded], \'scenario_input\': scenario}\n\ndef threshold_scenario(labels, probabilities, threshold):\n    target, score = _sim_portfolio(labels, probabilities)\n    cutoff = _sim_scalar(threshold, \'threshold\', 0, 1)\n    decision = (score >= cutoff).astype(int)\n    return {\'metrics\': dict(_sim_counts(target, decision), threshold=cutoff), \'decision\': decision}\n\ndef capacity_scenario(labels, probabilities, fraction, index=None):\n    target, score = _sim_portfolio(labels, probabilities)\n    share = _sim_scalar(fraction, \'fraction\', 0, 1)\n    order, keys = _sim_order(score, index)\n    k = min(len(score), math.ceil(share * len(score)))\n    chosen = order[:k]\n    decision = np.zeros(len(score), dtype=int)\n    decision[chosen] = 1\n    metrics = _sim_counts(target, decision)\n    metrics.update({\'capacity_fraction\': share, \'predicted_churn_sum\': float(score[chosen].sum()), \'lift\': metrics[\'precision\'] / metrics[\'prevalence\'] if k and metrics[\'prevalence\'] else np.nan})\n    ranked = pd.DataFrame({\'rank\': np.arange(1, len(score) + 1), \'sample_row\': keys[order], \'estimated_risk\': score[order], \'historical_label\': target[order], \'selected\': np.arange(len(score)) < k})\n    return {\'metrics\': metrics, \'decision\': decision, \'ranking\': ranked, \'selected_positions\': chosen}\n\ndef _campaign_components(probabilities, fraction, contact_cost, value_per_saved, success_rate, fixed_cost=0.0, index=None):\n    """Shared analytic calculation; comparison tables do not need Monte Carlo draws."""\n    score = _sim_probabilities(probabilities)\n    share = _sim_scalar(fraction, \'fraction\', 0, 1)\n    cost = _sim_scalar(contact_cost, \'contact_cost\', 0)\n    benefit = _sim_scalar(value_per_saved, \'value_per_saved\', 0)\n    success = _sim_scalar(success_rate, \'success_rate\', 0, 1)\n    fixed = _sim_scalar(fixed_cost, \'fixed_cost\', 0)\n    order, _ = _sim_order(score, index)\n    k = min(len(score), math.ceil(share * len(score)))\n    selected = order[:k]\n    risk_sum = float(score[selected].sum())\n    launched_fixed = fixed if k else 0.0\n    total_cost = k * cost + launched_fixed\n    expected_saved = risk_sum * success\n    expected_gross = expected_saved * benefit\n    expected_net = expected_gross - total_cost\n    success_probability = score[selected] * success\n    break_even = total_cost / (benefit * risk_sum) if benefit > 0 and risk_sum > 0 else np.nan\n    summary = {\'contacts\': k, \'risk_sum\': risk_sum, \'success_rate_assumption\': success, \'expected_saved\': expected_saved, \'expected_gross_um\': expected_gross, \'total_cost_um\': total_cost, \'expected_net_um\': expected_net, \'expected_roi\': expected_net / total_cost if total_cost > 0 else np.nan, \'break_even_success_rate\': break_even, \'break_even_feasible\': bool(np.isfinite(break_even) and break_even <= 1)}\n    return {\'summary\': summary, \'selected_positions\': selected, \'success_probability\': success_probability, \'assumptions\': {\'fraction\': share, \'contact_cost_um\': cost, \'value_per_saved_um\': benefit, \'success_rate\': success, \'fixed_cost_um\': fixed, \'fixed_cost_charged_only_if_contacts\': True, \'baseline_risk_approximates_no_campaign\': True, \'independent_outcomes\': True}}\n\ndef retention_expectation(probabilities, fraction, contact_cost, value_per_saved, success_rate, fixed_cost=0.0, index=None):\n    """Deterministic expected outcome under explicitly supplied business assumptions."""\n    result = _campaign_components(probabilities, fraction, contact_cost, value_per_saved, success_rate, fixed_cost, index)\n    return {\'summary\': result[\'summary\'], \'assumptions\': result[\'assumptions\'], \'selected_positions\': result[\'selected_positions\']}\n\ndef retention_scenario(probabilities, fraction, contact_cost, value_per_saved, success_rate, fixed_cost=0.0, n_draws=2000, seed=42, index=None):\n    result = _campaign_components(probabilities, fraction, contact_cost, value_per_saved, success_rate, fixed_cost, index)\n    draws_number = _sim_scalar(n_draws, \'n_draws\', 1, 20000)\n    random_seed = _sim_scalar(seed, \'seed\', 0, 2 ** 32 - 1)\n    if not draws_number.is_integer() or not random_seed.is_integer():\n        raise ValueError(\'n_draws dan seed harus bilangan bulat.\')\n    summary = result[\'summary\']\n    k, total_cost = (summary[\'contacts\'], summary[\'total_cost_um\'])\n    benefit = result[\'assumptions\'][\'value_per_saved_um\']\n    success_probability = result[\'success_probability\']\n    random = np.random.default_rng(int(random_seed))\n    saved = np.zeros(int(draws_number), dtype=int)\n    if k:\n        for start in range(0, len(saved), 256):\n            end = min(start + 256, len(saved))\n            saved[start:end] = (random.random((end - start, k)) < success_probability).sum(axis=1)\n    net = saved * benefit - total_cost\n    probability_profit = float(np.mean(net > 0))\n    expected_variance = float(np.sum(success_probability * (1 - success_probability)))\n    summary.update({\'profit_probability_mc\': probability_profit, \'profit_probability_mc_se\': float(np.sqrt(probability_profit * (1 - probability_profit) / len(net))), \'net_q025_um\': float(np.quantile(net, 0.025)), \'net_q50_um\': float(np.quantile(net, 0.5)), \'net_q975_um\': float(np.quantile(net, 0.975)), \'saved_mean_mc\': float(saved.mean()), \'saved_mean_mc_se_theoretical\': float(np.sqrt(expected_variance / len(saved))), \'n_draws\': len(saved), \'seed\': int(random_seed)})\n    return {\'summary\': summary, \'draws\': pd.DataFrame({\'saved_customers\': saved, \'net_benefit_um\': net}), \'selected_positions\': result[\'selected_positions\'], \'assumptions\': result[\'assumptions\']}\n', 'synthetic.py': '"""Deterministic synthetic data for integration tests, never evidence about a real bank."""\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\n\ndef make_synthetic_data(rows=1000,seed=42):\n    if isinstance(rows,bool) or not isinstance(rows,int) or rows<300:\n        raise ValueError("At least 300 synthetic rows are needed for split and calibration tests")\n    rng=np.random.default_rng(seed)\n    age=rng.integers(18,81,rows)\n    active=rng.integers(0,2,rows)\n    products=rng.integers(1,4,rows)\n    geography=rng.choice(["France","Germany","Spain"],rows)\n    balance=np.round(rng.uniform(0,200000,rows),2)\n    risk=1/(1+np.exp(-(-4.7+.065*age-.8*active+.5*(products==1)+.5*(geography=="Germany"))))\n    labels=rng.binomial(1,risk)\n    frame=pd.DataFrame({"RowNumber":np.arange(1,rows+1),"CustomerId":[f"SYNTH-{i:05d}" for i in range(rows)],\n        "Surname":["SYNTHETIC"]*rows,"CreditScore":rng.integers(350,851,rows),"Geography":geography,\n        "Gender":rng.choice(["Female","Male"],rows),"Age":age,"Tenure":rng.integers(0,11,rows),\n        "Balance":balance,"NumOfProducts":products,"HasCrCard":rng.integers(0,2,rows),\n        "IsActiveMember":active,"EstimatedSalary":np.round(rng.uniform(100,199000,rows),2),"Exited":labels})\n    if min(frame.Exited.value_counts())<30:\n        raise ValueError("Synthetic draw has too few examples of one class; change seed or row count")\n    return frame\n\n\ndef write_synthetic_csv(path,rows=1000,seed=42):\n    target=Path(path)\n    target.parent.mkdir(parents=True,exist_ok=True)\n    if target.exists():\n        raise FileExistsError("Synthetic generator does not overwrite an existing file")\n    make_synthetic_data(rows,seed).to_csv(target,index=False,lineterminator="\\n")\n    return target\n', 'training_core.py': '"""Train-only nested selection and validation-only policy functions."""\nimport json\nimport math\nimport time\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import clone\nfrom sklearn.metrics import (accuracy_score,average_precision_score,balanced_accuracy_score,brier_score_loss,\n    confusion_matrix,f1_score,fbeta_score,log_loss,precision_recall_curve,precision_score,recall_score,roc_auc_score)\nfrom sklearn.model_selection import RandomizedSearchCV,StratifiedKFold\nfrom .preprocessing import make_pipeline\nfrom .inference import apply_policy\n\n\ndef validate_label_scores(labels, probabilities):\n    label_array = np.asarray(labels)\n    probability_array = np.asarray(probabilities, dtype=float)\n    if label_array.ndim != 1 or probability_array.ndim != 1 or len(label_array) != len(probability_array):\n        raise ValueError("Label dan probabilitas harus vektor 1D dengan panjang yang sama.")\n    if len(label_array) == 0 or not np.isin(label_array, [0, 1]).all() or np.unique(label_array).size != 2:\n        raise ValueError("Label wajib lengkap, bernilai 0/1, dan memiliki kedua kelas.")\n    if not np.isfinite(probability_array).all() or ((probability_array < 0) | (probability_array > 1)).any():\n        raise ValueError("Probabilitas wajib finite dan berada di antara 0 dan 1.")\n    return label_array.astype(int), probability_array\n\n\ndef choose_threshold(labels, probabilities, beta=2.0, min_precision=None, max_contact_rate=None):\n    labels, probabilities = validate_label_scores(labels, probabilities)\n    beta = float(beta)\n    if not np.isfinite(beta) or beta <= 0:\n        raise ValueError("beta harus finite dan positif.")\n    for name, value in [("min_precision", min_precision), ("max_contact_rate", max_contact_rate)]:\n        if value is not None and (not np.isfinite(value) or not 0 <= value <= 1):\n            raise ValueError(f"{name} harus None atau angka finite antara 0 dan 1.")\n    precision, recall, thresholds = precision_recall_curve(labels, probabilities)\n    p, r = precision[:-1], recall[:-1]\n    # Bentuk ekuivalen ini menghindari overflow ketika beta besar.\n    weight = (1.0 / beta) ** 2 if beta >= 1 else beta ** 2\n    denominator = p + weight * r if beta >= 1 else weight * p + r\n    fbeta = (1 + weight) * p * r / np.maximum(denominator, np.finfo(float).eps)\n    contacts = len(probabilities) - np.searchsorted(np.sort(probabilities), thresholds, side="left")\n    table = pd.DataFrame({\n        "threshold": thresholds, "precision": p, "recall": r, "fbeta": fbeta,\n        "contacts": contacts, "contact_rate": contacts / len(probabilities),\n    })\n    table["eligible"] = True\n    if min_precision is not None:\n        table["eligible"] &= table["precision"] >= min_precision\n    if max_contact_rate is not None:\n        table["eligible"] &= table["contact_rate"] <= max_contact_rate\n    feasible = table.loc[table["eligible"]]\n    if feasible.empty:\n        raise ValueError("Tidak ada threshold validation yang memenuhi batas precision/kapasitas. Tinjau konfigurasi.")\n    best = feasible.sort_values(["fbeta", "precision", "threshold"], ascending=[False, False, False]).iloc[0]\n    return float(best["threshold"]), table\n\n\ndef metric_row(labels, probabilities, threshold, label):\n    labels, probabilities = validate_label_scores(labels, probabilities)\n    if not np.isfinite(threshold) or not 0 <= threshold <= 1:\n        raise ValueError("Threshold harus finite antara 0 dan 1.")\n    prediction = (np.asarray(probabilities) >= threshold).astype(int)\n    tn, fp, fn, tp = confusion_matrix(labels, prediction, labels=[0, 1]).ravel()\n    return {\n        "policy": label, "threshold": float(threshold),\n        "accuracy": accuracy_score(labels, prediction),\n        "balanced_accuracy": balanced_accuracy_score(labels, prediction),\n        "precision_churn": precision_score(labels, prediction, zero_division=0),\n        "recall_churn": recall_score(labels, prediction, zero_division=0),\n        "f1_churn": f1_score(labels, prediction, zero_division=0),\n        "f2_churn": fbeta_score(labels, prediction, beta=2, zero_division=0),\n        "average_precision": average_precision_score(labels, probabilities),\n        "roc_auc": roc_auc_score(labels, probabilities),\n        "brier_score": brier_score_loss(labels, probabilities),\n        "log_loss": log_loss(labels, probabilities, labels=[0, 1]),\n        "contact_rate": prediction.mean(), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),\n    }\n\n\ndef bootstrap_intervals(labels, probabilities, threshold, n_resamples=500, seed=42):\n    labels, probabilities = np.asarray(labels), np.asarray(probabilities)\n    rng = np.random.default_rng(seed)\n    scores = {"roc_auc": [], "average_precision": [], "precision_churn": [], "recall_churn": []}\n    for _ in range(n_resamples):\n        sample = rng.integers(0, len(labels), size=len(labels))\n        sample_y, sample_p = labels[sample], probabilities[sample]\n        if np.unique(sample_y).size < 2:\n            continue\n        pred = (sample_p >= threshold).astype(int)\n        scores["roc_auc"].append(roc_auc_score(sample_y, sample_p))\n        scores["average_precision"].append(average_precision_score(sample_y, sample_p))\n        scores["precision_churn"].append(precision_score(sample_y, pred, zero_division=0))\n        scores["recall_churn"].append(recall_score(sample_y, pred, zero_division=0))\n    return pd.DataFrame([\n        {"metric": name, "low_95": np.quantile(values, 0.025), "high_95": np.quantile(values, 0.975),\n         "valid_resamples": len(values)} for name, values in scores.items()\n    ])\n\n\ndef make_search(specification, seed, folds, trials):\n    parameters = specification.get("parameters", {})\n    combinations = math.prod(len(values) for values in parameters.values()) if parameters else 1\n    return RandomizedSearchCV(\n        make_pipeline(clone(specification["estimator"])), parameters,\n        n_iter=min(trials, combinations),\n        scoring={"ap": "average_precision", "roc_auc": "roc_auc", "brier": "neg_brier_score"},\n        refit="ap", cv=StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed),\n        random_state=seed, n_jobs=1, error_score="raise", return_train_score=False,\n    )\n\n\ndef nested_compare(features, labels, specifications, outer_folds=3, inner_folds=3, trials=6, seed=42):\n    if not isinstance(features, pd.DataFrame) or not features.index.is_unique:\n        raise ValueError("Nested CV memerlukan DataFrame dengan index unik.")\n    labels = np.asarray(labels)\n    if labels.ndim != 1 or len(labels) != len(features) or not np.isin(labels, [0, 1]).all():\n        raise ValueError("Label nested CV harus vektor 0/1 yang sesuai baris fitur.")\n    if min(outer_folds, inner_folds) < 2 or trials < 1 or not specifications:\n        raise ValueError("Konfigurasi fold, search trials, atau kandidat tidak valid.")\n    outer = StratifiedKFold(n_splits=outer_folds, shuffle=True, random_state=seed)\n    fold_rows, oof = [], pd.DataFrame(index=features.index)\n    fold_ids = np.full(len(features), -1, dtype=int)\n    partitions = list(outer.split(features, labels))\n    for fold_number, (_, validation_positions) in enumerate(partitions, start=1):\n        fold_ids[validation_positions] = fold_number\n    for name, specification in specifications.items():\n        scores = np.full(len(features), np.nan)\n        counts = np.zeros(len(features), dtype=int)\n        for fold_number, (training_positions, validation_positions) in enumerate(partitions, start=1):\n            started = time.perf_counter()\n            search = make_search(specification, seed + 10, inner_folds, trials)\n            search.fit(features.iloc[training_positions], labels[training_positions])\n            positive_index = int(np.flatnonzero(search.classes_ == 1)[0])\n            p = search.predict_proba(features.iloc[validation_positions])[:, positive_index]\n            scores[validation_positions] = p\n            counts[validation_positions] += 1\n            fold_rows.append({\n                "model": name, "outer_fold": fold_number, "n_fit": len(training_positions),\n                "n_evaluate": len(validation_positions),\n                "average_precision": average_precision_score(labels[validation_positions], p),\n                "roc_auc": roc_auc_score(labels[validation_positions], p),\n                "brier_score": brier_score_loss(labels[validation_positions], p),\n                "seconds": time.perf_counter() - started,\n                "best_parameters": json.dumps(search.best_params_, sort_keys=True),\n            })\n            print(f"{name} · outer {fold_number}/{outer_folds} · AP={fold_rows[-1][\'average_precision\']:.3f}", flush=True)\n        if not np.isfinite(scores).all() or not (counts == 1).all():\n            raise AssertionError("Setiap baris train harus mendapat tepat satu prediksi outer-fold.")\n        oof[name] = scores\n    folds = pd.DataFrame(fold_rows)\n    leaderboard = folds.groupby("model").agg(\n        outer_ap_mean=("average_precision", "mean"), outer_ap_sd=("average_precision", "std"),\n        outer_roc_auc_mean=("roc_auc", "mean"), outer_brier_mean=("brier_score", "mean"),\n        fit_seconds=("seconds", "sum"),\n    ).reset_index().sort_values(\n        ["outer_ap_mean", "outer_roc_auc_mean", "model"], ascending=[False, False, True],\n    ).reset_index(drop=True)\n    return leaderboard, folds, oof, pd.Series(fold_ids, index=features.index, name="outer_fold")\n\n\ndef policy_metrics(labels, probabilities, policy, name, index=None):\n    labels, p = validate_label_scores(labels, probabilities)\n    decision = apply_policy(p, policy, index)\n    tn, fp, fn, tp = confusion_matrix(labels, decision, labels=[0, 1]).ravel()\n    return {\n        "policy": name, "kind": policy["kind"], "threshold": policy.get("threshold", np.nan),\n        "contacts": int(decision.sum()), "contact_rate": decision.mean(),\n        "precision": precision_score(labels, decision, zero_division=0),\n        "recall": recall_score(labels, decision, zero_division=0),\n        "f1": f1_score(labels, decision, zero_division=0),\n        "f2": fbeta_score(labels, decision, beta=2, zero_division=0),\n        "accuracy": accuracy_score(labels, decision),\n        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),\n    }\n'}
PACKAGE_SHA256 = {'__init__.py': '2e823fd96954a59c58e52002bc49f363ce5988be90c60f04f1109bb9e44659fb', 'cli.py': '779dd5dfb98d6d64cf22ee302a3c95bc39f0a36f49951c071c330237539a3435', 'config.py': '349bee8a9b3d51c65dbe2d60553778b3f943901c849b454ba39fb7dc5af16f97', 'data.py': 'cbe97fc42a7b68033d5b32ed356356fb38ee83c2d648705d0e1e53d990f32a55', 'decision_tools.py': '42556d6f0ddb540184b8e0e95772bf824d61ca642ecd9957168320887558ed45', 'inference.py': '265a474268f1708ccedae6b6b59768f302666581dad1fea2e96ceeb07f9ed678', 'live.py': 'f1826d676d07ab0aa72ea27471e68ad4c57e7a0a9f39a99b6f8d27573be2d3f4', 'pipeline.py': 'ffbddeed1fd583319c2d32d369ee4fdebea8c6c2747eb35c3b8ebeedc846a4d2', 'preprocessing.py': '6617bd1967ed9183b02d7416139c7d505d41ee32cde89db83fdc06793d3d4727', 'reporting.py': '7cd4f49a991a25d7227151fb9c086ebb6344339f561fbfac7038a149e5cbde3b', 'schema.py': '12f1c26b341397b647fe9ae9727fb6adadc94c2b695197ef3fd565c579fd9086', 'simulation.py': '2a23747506c320a2378cdc055d1af9fa6bf5565326cb1625cf5ab5aa082ce387', 'synthetic.py': 'b0ab1f4d585a50005ff7172b6b870b85d3d1f98111d339eade1d012fcf058300', 'training_core.py': 'a482a9a516e8987c295e63f8ad782e5d9b673161fb94a6f4dcc2086445f722fd'}
import hashlib,importlib,sys,tempfile
from pathlib import Path
package_root=Path(tempfile.mkdtemp(prefix="bank-churn-v7-"))
package_dir=package_root/"banking_churn";package_dir.mkdir()
for name,source in PACKAGE_SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest()==PACKAGE_SHA256[name]
    (package_dir/name).write_text(source,encoding="utf-8")
for name in list(sys.modules):
    if name=="banking_churn" or name.startswith("banking_churn."):del sys.modules[name]
sys.path.insert(0,str(package_root));importlib.invalidate_caches()
from banking_churn import __version__
from banking_churn.config import TrainingConfig
from banking_churn.pipeline import run_training,finish_run,initialize_decision_examples
from banking_churn.inference import predict_customers,read_source_csv
from banking_churn.decision_tools import v6_policy_advisor,v6_uncertain_campaign,v6_batch_quality,v6_json,v6_number
from banking_churn.live import save_live_section,refresh_reports
from banking_churn.simulation import profile_what_if,threshold_scenario,capacity_scenario,retention_scenario
from banking_churn.synthetic import write_synthetic_csv
import numpy as np,pandas as pd,json
from IPython.display import display,Markdown,Image,FileLink
print("Modul bersama V"+__version__,"|",len(PACKAGE_SOURCES),"berkas terverifikasi.")

## 2. Sumber, cakupan, dan batas data

CSV diberikan pengguna melalui [Drive sumber](https://drive.google.com/file/d/1_mpH_uzTGHaYIF33pz9Is6KW_wj8cv8-/view). Snapshot referensi memiliki **10.000 baris, 14 kolom**, dan **2.037 label churn**. SHA-256 referensi: `3996cd1fa372e0db0cd9c0ebac35bbd4e8e3c65fb942bb010c826e7b1eeef0a0`. Angka tersebut mendeskripsikan file ini; penerbit asal, waktu ekstraksi, horizon churn, mata uang, dan lisensi redistribusi belum terverifikasi.

`Exited` adalah target 0/1. Enam fitur numerik, dua fitur biner, dan dua fitur kategori dipakai. `RowNumber`, `CustomerId`, serta `Surname` tidak menjadi prediktor. Pelanggan berulang dengan ID sama ditolak karena memerlukan split per pelanggan/waktu. Missing fitur ditangani dalam pipeline, tanpa mengisi target yang hilang. Nilai di luar domain dan kategori asing ditandai saat inferensi.

Mode **synthetic_smoke** membangkitkan data fiktif berlabel untuk menguji semua tahap dengan waktu singkat. Ukuran data, distribusi, model terpilih, dan metriknya berbeda dari referensi. Semua laporan mode ini diberi label DATA SINTETIS. Keberhasilan CI sintetis membuktikan alur berfungsi, bukan akurasi pada bank nyata.

In [ ]:
#@title Konfigurasi V7
Mode_data = "reference_source" #@param ["reference_source", "synthetic_smoke"]
Path_CSV = "churn.csv" #@param {type:"string"}
Seed = 42 #@param {type:"integer"}
Jumlah_baris_sintetis = 1000 #@param {type:"integer"}
OUTPUT_BASE = Path("bank_churn_artifacts_v7")
DATA_FILE = Path(Path_CSV)
STRICT_REFERENCE = True
if Mode_data=="synthetic_smoke":
    import uuid
    DATA_FILE=OUTPUT_BASE/"inputs"/("synthetic-"+uuid.uuid4().hex[:8]+".csv")
    write_synthetic_csv(DATA_FILE,Jumlah_baris_sintetis,Seed)
    CONFIG=TrainingConfig.quick(Seed);STRICT_REFERENCE=False
elif Mode_data=="reference_source":
    CONFIG=TrainingConfig(seed=Seed)
    if not DATA_FILE.is_file():
        import gdown
        DATA_FILE.parent.mkdir(parents=True,exist_ok=True)
        downloaded=gdown.download(id="1_mpH_uzTGHaYIF33pz9Is6KW_wj8cv8-",output=str(DATA_FILE),quiet=False)
        if not downloaded:raise FileNotFoundError("Unduhan gagal. Unggah CSV sumber ke Files dan isi Path_CSV.")
else:raise ValueError("Mode data tidak dikenal")
CONFIG.validate()
print("Konteks:",Mode_data,"| sumber tersedia:",DATA_FILE.is_file(),"| seed:",Seed)
print("Data sintetis: uji alur saja." if CONFIG.smoke else "Hash CSV referensi akan diverifikasi sebelum training.")

## 3. Training, kalibrasi, dan pemilihan kebijakan

Alur: **verifikasi CSV → split 60% train / 20% validation / 20% test → nested CV pada train → kalibrasi pada train → threshold pada validation → audit test → simulasi → ekspor**.

Preprocessing terdiri dari imputasi numerik median + standardisasi, serta imputasi kategori + one-hot encoding. Semua transformer di-fit hanya di dalam fold training sehingga statistik validation/test tidak bocor. Model referensi mencakup Dummy, Logistic Regression, Random Forest, HistGradientBoosting, dan XGBoost. Outer CV 3 fold mengaudit pencarian parameter 3 fold; enam percobaan dibatasi untuk runtime. Pemilihan keluarga model memakai average precision dari train, dengan Dummy sebagai pembanding. Mode sintetis memakai tiga keluarga, 2×2 fold, dan satu percobaan.

Kalibrasi sigmoid membungkus pencarian parameter sehingga setiap fold kalibrasi melatih ulang pencarian pada bagiannya sendiri. Skor validation dipakai untuk threshold F0,5, F1, dan F2. F0,5 memberi bobot precision lebih besar; F2 memberi bobot recall lebih besar; F1 menyeimbangkan keduanya. Default adalah F1. Ada pula threshold 0,5 serta kebijakan top-k berdasarkan kapasitas. Semua pilihan dikunci sebelum test dinilai. Jika nilai ranking sama, pemilihan top-k memakai urutan ID indeks yang stabil.

Test ini sudah dilihat selama iterasi V5/V6/V7, sehingga tetap **audit internal**. Mengganti seed pada data yang sama tidak menghasilkan validasi eksternal. Dataset tidak memiliki timestamp yang memadai; temporal split yang sah memerlukan tanggal observasi dan horizon label baru.

In [ ]:
run=run_training(DATA_FILE,OUTPUT_BASE,CONFIG,Mode_data,STRICT_REFERENCE)
ARTIFACT_DIR=run.artifact_dir
print("Run:",run.run_id,"| model:",run.metadata["model_name"])
print(run.tables["split_summary"].to_string(index=False))
print("\nNested CV pada train:")
print(run.tables["nested_model_comparison"].to_string(index=False))
print("\nKalibrasi pada validation (diagnostik, bukan pemilihan test):")
print(run.tables["calibration_diagnostic"].to_string(index=False))

## 4. Membaca audit model

**ROC-AUC** mengukur ranking label positif terhadap negatif. **Average precision (AP)** merangkum precision-recall sepanjang threshold; bandingkan dengan prevalensi test, bukan dengan accuracy. **Brier/log loss** mengukur kesalahan probabilitas, lebih rendah lebih baik. **Precision** adalah bagian prioritas yang berlabel churn; **recall** adalah bagian seluruh churn yang tertangkap. TP/FP/FN/TN menjelaskan jumlah konkret, dan contact rate harus sesuai kapasitas tim.

Interval bootstrap menggambarkan variasi resampling pada test tetap, bukan seluruh ketidakpastian proses training, perubahan populasi, atau dampak intervensi. Permutation importance dihitung pada validation, memperlihatkan ketergantungan prediksi terhadap fitur; fitur berkorelasi dapat saling berbagi informasi. Audit segmen diberi penanda sampel terbatas. Perbedaan segmen tidak langsung menjadi bukti diskriminasi atau penyebab churn.

In [ ]:
print("Skor test:",json.dumps(v6_json(run.metadata["test_scores"]),indent=2))
print("\nKebijakan dikunci pada validation; tabel berikut audit test:")
print(run.tables["test_policies"].to_string(index=False))
print("\nInterval bootstrap test:",run.tables["test_bootstrap_intervals"].to_string(index=False),sep="\n")
print("\nPermutation importance validation:",run.tables["validation_importance"].to_string(index=False),sep="\n")
print("\nAudit segmen test:",run.tables["test_segments"].to_string(index=False),sep="\n")
manifest=finish_run(run)
display(Image(filename=str(ARTIFACT_DIR/"model_audit.png")))
display(Markdown((ARTIFACT_DIR/"current_report.md").read_text(encoding="utf-8")))

## 5. Laboratorium simulasi - tujuh form

Semua form memakai estimator tetap; perubahan form tidak melatih model ulang. Form threshold, kapasitas, advisor, dan simulasi ekonomi memakai **validation**, agar eksplorasi tidak memilih kebijakan lewat test. Angka pada empat form dasar adalah snapshot eksplorasi, terpisah dari default audit model. Advisor, ketidakpastian, dan kualitas batch memperbarui ringkasan eksekutif run yang sama.

**UM** adalah unit moneter asumsi. Rumus expected net adalah `B × u × jumlah risiko pelanggan terpilih - C × jumlah kontak - F`, dengan F hanya dibebankan bila ada kontak. Ini memerlukan asumsi bahwa skor mendekati risiko tanpa kampanye, u peluang menyelamatkan pelanggan yang akan churn, serta B manfaat kontribusi setelah biaya relevan. Dataset tidak mengestimasi u atau membuktikan asumsi tersebut. Zero kontak menghasilkan biaya dan net nol.

In [ ]:
X_train=run.frames["train"];X_valid=run.frames["validation"]
y_valid=run.labels["validation"];validation_probability=run.probabilities["validation"]
policies=run.metadata["policies"]
def save_exploration(kind,inputs,summary,tables=None):
    payload=v6_json({"kind":kind,"run_id":run.run_id,"source_sha256":run.metadata["data_sha256"],
        "data_context":Mode_data,"scope":"validation_exploration","inputs":inputs,"summary":summary})
    (ARTIFACT_DIR/("form_"+kind+"_latest.json")).write_text(json.dumps(payload,ensure_ascii=False,indent=2,allow_nan=False),encoding="utf-8")
    for key,table in (tables or {}).items():table.to_csv(ARTIFACT_DIR/("form_"+kind+"_"+key+"_latest.csv"),index=False)
    refresh_reports(ARTIFACT_DIR)
    print("Snapshot eksplorasi tersimpan. Default model tetap:",run.metadata["default_policy"])
print("Form siap | konteks:",Mode_data,"|",len(X_valid),"baris validation")

### 5.1. What-if profil
Bandingkan profil fiktif yang sama dengan status keaktifan atau jumlah produk berbeda. Selisih skor menunjukkan respons estimator, **bukan pengaruh kausal** perubahan produk/keaktifan. Periksa juga apakah kombinasi fitur masuk akal dan masih dalam dukungan training.

In [ ]:
#@title Form 1 - What-if profil fiktif
Usia_profil = 45 #@param {type:"integer"}
Aktif_skenario = 1 #@param [0,1] {type:"raw"}
Produk_skenario = 2 #@param {type:"integer"}
baseline=pd.DataFrame([dict(CreditScore=650,Age=Usia_profil,Tenure=5,Balance=50000.,NumOfProducts=1,
    EstimatedSalary=60000.,HasCrCard=1,IsActiveMember=0,Geography="France",Gender="Male")])
profile_result=profile_what_if(baseline,{"IsActiveMember":Aktif_skenario,"NumOfProducts":Produk_skenario},run.bundle,run.metadata["threshold"],predict_customers)
print(profile_result["comparison"].to_string());print("Perubahan skor (poin persen):",profile_result["risk_change_pp"])
print("Bukan estimasi efek intervensi.")
save_exploration("profile",{"age":Usia_profil,"active":Aktif_skenario,"products":Produk_skenario},
    {"risk_change_pp":profile_result["risk_change_pp"]},{"comparison":profile_result["comparison"].reset_index()})

### 5.2. Threshold
Threshold lebih rendah biasanya menambah prioritas dan recall, dengan tambahan false positive. Form ini hanya memperlihatkan tradeoff pada validation. Threshold model yang tersimpan tetap berasal dari konfigurasi training.

In [ ]:
#@title Form 2 - Eksplorasi threshold validation
Threshold_simulasi = 0.30 #@param {type:"number"}
threshold_result=threshold_scenario(y_valid,validation_probability,Threshold_simulasi)
print(json.dumps(v6_json(threshold_result["metrics"]),indent=2))
save_exploration("threshold",{"threshold":Threshold_simulasi},threshold_result["metrics"])

### 5.3. Kapasitas
Top-k memilih risiko tertinggi sesuai proporsi batch, tanpa menjamin semua yang dipilih akan churn. Bandingkan recall dan precision pada kapasitas tim nyata; hasilnya bergantung pada komposisi batch. Zero persen berarti tidak ada kontak.

In [ ]:
#@title Form 3 - Kapasitas tim pada validation
Kapasitas_persen = 20.0 #@param {type:"number"}
fraction=v6_number(Kapasitas_persen,"Kapasitas",high=100)/100
capacity_result=capacity_scenario(y_valid,validation_probability,fraction,index=X_valid.index)
print(json.dumps(v6_json(capacity_result["metrics"]),indent=2))
save_exploration("capacity",{"capacity_percent":Kapasitas_persen},capacity_result["metrics"])

### 5.4. Retensi dengan parameter tetap
Monte Carlo membangkitkan jumlah retained fiktif menggunakan `risiko × u` untuk pelanggan terpilih. Kuantil net merangkum variasi outcome di bawah asumsi tetap; belum memasukkan kesalahan kalibrasi, heterogenitas treatment, atau perubahan populasi. Distribusi ini bukan laba yang telah diamati.

In [ ]:
#@title Form 4 - Retensi dan biaya dengan asumsi tetap
Kontak_retensi_persen = 20.0 #@param {type:"number"}
Biaya_kontak_retensi_UM = 10.0 #@param {type:"number"}
Manfaat_retained_retensi_UM = 100.0 #@param {type:"number"}
Keberhasilan_retensi_persen = 20.0 #@param {type:"number"}
Biaya_tetap_retensi_UM = 500.0 #@param {type:"number"}
Jumlah_draw_retensi = 2000 #@param {type:"integer"}
retention_inputs=dict(fraction=v6_number(Kontak_retensi_persen,"Kapasitas",high=100)/100,
    contact_cost=Biaya_kontak_retensi_UM,value_per_saved=Manfaat_retained_retensi_UM,
    success_rate=v6_number(Keberhasilan_retensi_persen,"Keberhasilan",high=100)/100,
    fixed_cost=Biaya_tetap_retensi_UM,n_draws=Jumlah_draw_retensi,seed=Seed)
retention_result=retention_scenario(validation_probability,index=X_valid.index,**retention_inputs)
print(json.dumps(v6_json(retention_result["summary"]),indent=2))
print("Bersyarat pada asumsi, bukan dampak bisnis terukur.")
save_exploration("retention",retention_inputs,{"results":retention_result["summary"],"assumptions":retention_result["assumptions"]},
    {"draws":retention_result["draws"]})

### 5.5. Advisor kapasitas dan anggaran
Advisor memeriksa kebijakan yang sudah dikunci dan setiap ukuran top-k yang memenuhi batas kontak/anggaran. Minimum precision memakai label validation. Kebijakan threshold yang melampaui kapasitas ditandai tidak layak, tanpa pemotongan diam-diam. Tidak menjalankan kampanye selalu kandidat dengan net nol. Advisor hanya menyarankan kandidat bila expected net positif dan batas terpenuhi; ia tidak mengganti threshold audit model. Optimasi pada validation memerlukan konfirmasi di data baru dan pilot.

In [ ]:
#@title Form 5 - Advisor kapasitas dan anggaran validation
Batas_kontak = 200 #@param {type:"integer"}
Anggaran_UM = 2500.0 #@param {type:"number"}
Biaya_kontak_UM = 10.0 #@param {type:"number"}
Manfaat_retained_UM = 100.0 #@param {type:"number"}
Keberhasilan_asumsi_persen = 20.0 #@param {type:"number"}
Biaya_tetap_advisor_UM = 500.0 #@param {type:"number"}
Minimum_precision_validation_persen = 0.0 #@param {type:"number"}
advisor=v6_policy_advisor(y_valid,validation_probability,policies,Batas_kontak,Anggaran_UM,Biaya_kontak_UM,
    Manfaat_retained_UM,v6_number(Keberhasilan_asumsi_persen,"Keberhasilan",high=100)/100,
    fixed_cost=Biaya_tetap_advisor_UM,min_precision=v6_number(Minimum_precision_validation_persen,"Precision",high=100)/100,index=X_valid.index)
print(advisor["candidates"].to_string(index=False))
print("Keputusan bersyarat:",advisor["status"],json.dumps(v6_json(advisor["recommended"]),indent=2))
save_live_section("advisor",advisor["inputs"],{"status":advisor["status"],"recommended":advisor["recommended"],
    "data_context":"validation","business_inputs_are_assumptions":True},ARTIFACT_DIR,run.context,
    {"candidates":advisor["candidates"],"top_k_curve":advisor["top_k_curve"]})
print("HTML/PDF dan ZIP diperbarui memakai input advisor ini.")

### 5.6. Rentang parameter dan outcome
Berbeda dari form 4, rentang uniform pada u, C, dan B juga berubah di setiap draw. Kontak tetap. Laporan memisahkan rentang expected net akibat asumsi parameter dari rentang outcome net yang juga mengandung variasi retained. Proporsi draw positif bergantung pada distribusi asumsi yang dipilih; ia bukan peluang laba terverifikasi.

In [ ]:
#@title Form 6 - Ketidakpastian asumsi bisnis
Kontak_ketidakpastian = 200 #@param {type:"integer"}
Keberhasilan_min_persen = 10.0 #@param {type:"number"}
Keberhasilan_max_persen = 30.0 #@param {type:"number"}
Biaya_min_UM = 8.0 #@param {type:"number"}
Biaya_max_UM = 12.0 #@param {type:"number"}
Manfaat_min_UM = 80.0 #@param {type:"number"}
Manfaat_max_UM = 120.0 #@param {type:"number"}
Biaya_tetap_ketidakpastian_UM = 500.0 #@param {type:"number"}
Jumlah_draw_ketidakpastian = 2000 #@param {type:"integer"}
uncertainty=v6_uncertain_campaign(validation_probability,Kontak_ketidakpastian,
    [Keberhasilan_min_persen/100,Keberhasilan_max_persen/100],[Biaya_min_UM,Biaya_max_UM],
    [Manfaat_min_UM,Manfaat_max_UM],Biaya_tetap_ketidakpastian_UM,n_draws=Jumlah_draw_ketidakpastian,seed=Seed,index=X_valid.index)
print(json.dumps(v6_json(uncertainty["summary"]),indent=2))
save_live_section("uncertainty",uncertainty["assumptions"],{"results":uncertainty["summary"],"assumptions":uncertainty["assumptions"]},
    ARTIFACT_DIR,run.context,{"draws":uncertainty["draws"]})
print("Rentang skenario pada HTML/PDF dan ZIP diperbarui.")

### 5.7. Kualitas batch dan scoring
Contoh validation tersedia setelah training. Data stres sintetis sengaja menggeser usia, saldo, dan kategori untuk mendemonstrasikan peringatan. Pilihan CSV saya membaca file yang diunggah ke Files; fitur wajib harus lengkap, kolom header ganda/asing ditolak, dan ID pelanggan disimpan hanya pada hasil lokal. Jangan mengunggah data pribadi ke repo public.

Pemeriksaan membandingkan missingness, kategori/rentang, serta distribusi fitur terhadap train dan skor terhadap validation. Batas alert adalah heuristik eksplorasi, bukan uji signifikansi. Pergeseran distribusi belum membuktikan penurunan akurasi. Tanpa label baru yang sesuai horizon, akurasi batch tidak dihitung; label opsional hanya untuk audit dan tidak melatih model ulang.

In [ ]:
#@title Form 7 - Audit batch dan scoring
Sumber_batch = "Contoh validation" #@param ["Contoh validation","Stres sintetis","CSV saya"]
Path_batch_CSV = "synthetic_customers.csv" #@param {type:"string"}
if Sumber_batch=="CSV saya":
    batch=read_source_csv(Path(Path_batch_CSV));batch_context="user_supplied_unverified"
else:
    batch=X_valid.head(min(200,len(X_valid))).copy();batch_context="illustrative_validation"
    if Sumber_batch=="Stres sintetis":
        batch["Age"]=95;batch["Balance"]=900000.;batch["Geography"]="SYNTHETIC_UNKNOWN";batch_context="synthetic_stress_demo"
batch_result=v6_batch_quality(X_train,batch,run.bundle,predict_customers,validation_probability,data_context=batch_context)
print(json.dumps(v6_json(batch_result["summary"]),indent=2))
print(batch_result["features"].to_string(index=False))
save_live_section("batch",{"source":Sumber_batch,"path":Path_batch_CSV if Sumber_batch=="CSV saya" else None},batch_result["summary"],
    ARTIFACT_DIR,run.context,{"features":batch_result["features"],"scores":batch_result["scores"]})
print("Hasil scoring disimpan lokal pada run; ringkasan public hanya agregat.")

## 6. Rekomendasi dan pengembangan berikutnya

1. **Definisikan keputusan:** siapa yang boleh dihubungi, horizon churn, kapasitas harian, consent, biaya kontak, nilai kontribusi, dan batas false positive. Pilih kebijakan pada validation sesuai tujuan; skor tinggi saja belum menjamin manfaat intervensi.
2. **Perbaiki data:** catat tanggal snapshot, tanggal churn, riwayat transaksi/keluhan/engagement, kanal kontak, dan riwayat penawaran. Audit cakupan, missingness, duplikasi pelanggan, serta status lisensi sumber.
3. **Validasi baru:** bekukan versi pipeline dan kebijakan, lalu nilai pada periode atau populasi yang belum pernah dipakai. Gunakan time/group split bila beberapa catatan berasal dari pelanggan sama. Laporkan ranking, kalibrasi, kapasitas, interval, dan error per segmen.
4. **Pilot treatment/control:** randomisasikan pelanggan eligible ke penawaran dan kontrol, tetapkan outcome serta horizon sebelum eksperimen. Ukur selisih churn, biaya incremental, nilai kontribusi, dan efek per segmen. Ukuran sampel memerlukan baseline, minimum detectable effect, serta power yang disepakati.
5. **Operasional:** simpan versi model/sumber/fitur dan jejak keputusan, batasi akses hasil scoring, serta pantau schema, distribusi, kalibrasi dan outcome saat label tersedia. Jadwal retraining harus dipicu evaluasi yang disetujui, bukan hanya alert distribusi.

Model uplift atau optimasi penawaran dapat dipertimbangkan setelah tersedia treatment/control yang memadai. Data saat ini belum dapat memilih intervensi yang menyebabkan retensi. Dashboard dan API scoring sebaiknya memakai modul inferensi yang sama serta versi bundle yang terverifikasi.

## 7. Pemeriksaan dan ekspor

Run dianggap complete setelah verifikasi split, prediksi reload, helper inferensi mandiri, metrik, laporan, dan arsip berhasil. `latest_run.json` menunjuk run terakhir yang lengkap. HTML bersifat mandiri, PDF merangkum hasil dan asumsi, `public_summary.json` berisi agregat, sedangkan ZIP juga memuat model dan hasil batch lokal sehingga harus ditinjau sebelum dibagikan.

Ubah form lalu unduh ZIP terbaru tanpa training ulang. Tautan di bawah bisa dipakai di Jupyter; di Colab buka panel **Files > bank_churn_artifacts_v7**, pilih PDF/HTML dalam folder run atau ZIP pada root folder tersebut, lalu Download.

In [ ]:
manifest=finish_run(run,initialize_examples=False)
print(json.dumps(v6_json(manifest),indent=2))
print("Pemeriksaan:",len(run.checks),"PASS")
for item in run.checks:print("PASS -",item["check"])
for filename in ["executive_summary.pdf","executive_summary.html","public_summary.json","current_report.md"]:
    display(FileLink(str(ARTIFACT_DIR/filename)))
display(FileLink(manifest["archive"]))
print("ZIP memuat bundle dan scoring lokal. Public repo hanya memakai ringkasan agregat yang ditinjau.")

## Referensi metode dan catatan versi

- [scikit-learn: pemilihan threshold](https://scikit-learn.org/stable/modules/classification_threshold.html)
- [scikit-learn: kalibrasi probabilitas](https://scikit-learn.org/stable/modules/calibration.html)
- [scikit-learn: nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)
- [scikit-learn: permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html)
- [uv: lockfile dan lingkungan](https://docs.astral.sh/uv/concepts/projects/layout/#the-lockfile)

V7 mengunci dependensi, menyatukan training/inference, menambah uji alur lengkap sintetis dan laporan HTML/PDF. Perubahan arsitektur tidak otomatis meningkatkan performa prediktif. Angka run ini selalu dihitung kembali dari sumber dan konfigurasi yang dicatat. V6 dan notebook asli tetap tersedia terpisah.